# METHODOLOGY

**Single notebook that unifies:**
1. **3-Stage Feature Selection** (Spearman UFS → RFE-per-dimension → RFECV cross-group) per model
2. **Per-fold Hyperparameter Tuning** with QWK-aligned scoring
3. **Dimensional level classification**
4. **Telagana Cross Corpus Validation**
5. **ONE STOP ENGLISH RANK CORRELATION**
6. **BASELINES COMPARISON**



# SECTION 0 — Install & Imports


In [4]:
# Memory + GPU diagnostic — run BEFORE anything else
import os, subprocess, sys
print(f'Python: {sys.version.split()[0]}')


# 4. Sentencepiece installed? (DeBERTa needs it; e5 doesn't but checking anyway)
try:
    import sentencepiece
    print(f'\nsentencepiece: {sentencepiece.__version__}')
except ImportError:
    print('\nsentencepiece: NOT INSTALLED')

Python: 3.12.13

sentencepiece: 0.2.1


In [5]:
!pip install -q textstat lightgbm xgboost gensim

In [6]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL tranformer — Install  (run once, then MANUALLY restart kernel, then skip)
# ═══════════════════════════════════════════════════════════════════════════
# DO NOT put os._exit() here — restart manually via Runtime > Restart kernel.
#
# Kaggle 2026 base image ships torch 2.3+ / transformers 4.40+; these pins
# keep everything consistent without downgrading.
#
!pip install -q --upgrade \
    "transformers>=4.46.0,<5.0" \
    "tokenizers>=0.20" \
    "accelerate>=0.26.0" \
    "sentencepiece>=0.1.99" \
    "protobuf>=3.20,<5.0"
# Do NOT touch torch/torchvision — Kaggle pre-installs CUDA-compiled binaries.
# Reinstalling them loses the CUDA build and breaks GPU training entirely.
print("Done. Now do Runtime > Restart kernel, then skip this cell.")

Done. Now do Runtime > Restart kernel, then skip this cell.


In [7]:

import warnings
warnings.filterwarnings('ignore')

import os, sys, json, math, glob, pickle, time, random, copy
import numpy as np
import pandas as pd
from collections import defaultdict, Counter

# scipy / sklearn
from scipy.stats import spearmanr, pearsonr, rankdata
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import (StratifiedKFold, KFold, cross_val_score,
                                     RandomizedSearchCV, GridSearchCV, ParameterGrid)
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.feature_selection import RFE, RFECV
from sklearn.metrics import (make_scorer, cohen_kappa_score, f1_score,
                              accuracy_score, precision_score, recall_score,
                              r2_score, mean_absolute_error, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import Ridge, ElasticNet, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import get_scorer

# torch / transformers
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

# xgboost
import xgboost as xgb
from xgboost import XGBClassifier

# matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print('All imports successful')

# New base models
try:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier
    LGBM_AVAILABLE = True
except ImportError:
    LGBM_AVAILABLE = False
    print('WARNING: lightgbm not installed — pip install lightgbm')
from sklearn.svm import SVR


Device: cuda
All imports successful


In [8]:
import numpy, pandas, torch
print(f"numpy   : {numpy.__version__}   (should be 2.x)")
print(f"pandas  : {pandas.__version__}")
print(f"torch   : {torch.__version__}")
print(f"GPU     : {torch.cuda.is_available()}")

# Confirm DeBERTa loads (this is the expensive one)
# from transformers import AutoTokenizer
# tok = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')
# print("DeBERTa tokenizer loaded OK")

numpy   : 2.0.2   (should be 2.x)
pandas  : 2.3.3
torch   : 2.10.0+cu128
GPU     : True


# SECTION 1 — Configuration

Edit the paths to point at your CSV/XLSX. The notebook is currently set up for the original `english_only_feature_full.xlsx` workbook. The five outer folds are kept identical to NB1's setup for reproducibility.


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION  ← edit these for your environment
# ══════════════════════════════════════════════════════════════════════════════
import os
# Detect Colab vs local
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    from google.colab import drive
    print('Detected Colab.')
except Exception:
    IN_COLAB = False
    print('Local environment.')

PROJECT_DIR ='/kaggle/working/'
# Edit this path to point at your XLSX. If running in Colab, mount Drive and update.

CSV_PATH    ='/kaggle/input/datasets/maureenmiranda11/18-05-26/english_supp_14.05.26_cleant.xlsx'

LABEL_COL     = 'grade'
TEXT_ID       = 'index'
TEXT_COL      = 'content'
TARGET_GRADES = [2, 3, 4, 5, 6,7,8]
NUM_CLASSES   = len(TARGET_GRADES)

# ── CV settings ────────────────────────────────────────────────────────────────
N_OUTER_FOLDS = 5
N_INNER_FOLDS = 4 
N_TUNE_FOLDS  = 4

# ── Feature-selection thresholds (from NB1) ───────────────────────────────────
SPEARMAN_THR = 0.15
P_THR        = 0.005
PEARSON_THR  = 0.90
SCORE_DELTA  = 0.001

# ── Random Forest tuning controls ─────────────────────────────────────────────
N_ITER_RF        = 160
N_RF_RESTARTS    = 1
RF_MIN_INNER_GAIN = 0.02

# ── Output directory ──────────────────────────────────────────────────────────
CKPT_DIR = '/kaggle/working/ckpt'
os.makedirs(CKPT_DIR, exist_ok=True)

# ── Transformer choice ────────────────────────────────────────────────────────
# DeBERTa-v3-base outperforms RoBERTa-base on small/medium classification (He et al. 2021).
# Set USE_DEBERTA=False to fall back to RoBERTa for ablation.
USE_DEBERTA = False

if USE_DEBERTA:
    TRANSFORMER_MODEL_NAME = 'microsoft/deberta-v3-base'
else:
    TRANSFORMER_MODEL_NAME = 'roberta-base'

TRANSFORMER_CFG = {
    'model_name'    : TRANSFORMER_MODEL_NAME,
    'max_length'    : 512,
    'freeze_layers' : 11,
    'dropout'       : 0.5,
    'epochs'        : 4,
    'batch_size'    : 4,
    'learning_rate' : 2e-5,
    'warmup_ratio'  : 0.2,
    'weight_decay'  : 0.01,
    'grad_clip'     : 1.0,
}

# XGBoost config (used as the *baseline* hybrid secondary; later we replace it with the best ensemble)
XGBOOST_CFG = {
    'n_estimators'     : 400,
    'max_depth'        : 3,
    'learning_rate'    : 0.1,
    'gamma'            : 0,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'eval_metric'      : 'mlogloss',
    'random_state'     : RANDOM_STATE,
    'n_jobs'           : 1,
}

print(f'Project dir   : {PROJECT_DIR}')
print(f'CSV path      : {CSV_PATH}')
print(f'Output dir    : {CKPT_DIR}')
print(f'Target grades : {TARGET_GRADES}  (NUM_CLASSES = {NUM_CLASSES})')
print(f'Outer folds   : {N_OUTER_FOLDS} | Inner folds: {N_INNER_FOLDS} | Tune folds: {N_TUNE_FOLDS}')
print(f'Transformer   : {TRANSFORMER_CFG["model_name"]} freeze={TRANSFORMER_CFG["freeze_layers"]}/12')

Detected Colab.
Project dir   : /kaggle/working/
CSV path      : /kaggle/input/datasets/maureenmiranda11/18-05-26/english_supp_14.05.26_cleant.xlsx
Output dir    : /kaggle/working/ckpt
Target grades : [2, 3, 4, 5, 6, 7, 8]  (NUM_CLASSES = 7)
Outer folds   : 5 | Inner folds: 4 | Tune folds: 4
Transformer   : roberta-base freeze=11/12


# SECTION 2 — Data Loading + textstat Augmentation


In [10]:
# 2A — Load workbook
df_raw = pd.read_excel(CSV_PATH)
df_raw.columns = df_raw.columns.str.strip()
print(f'Loaded: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns')

# Preserve raw text BEFORE any column drops (used by transformer + TF-IDF MLP)
_raw_texts_by_index = None
if TEXT_COL in df_raw.columns:
    _raw_texts_by_index = df_raw[TEXT_COL].fillna('').astype(str).copy()


Loaded: 172 rows x 707 columns


In [11]:
# 2B — textstat readability features (matches NB1 D10.2 codes)
try:
    import textstat
except Exception:
    %pip install -q textstat
    import textstat

# Source frame for text retrieval (in case TEXT_COL was dropped earlier)
_source_df = df_raw.copy()
_TEXT_CANDS = [TEXT_COL, 'content', 'story', 'text', 'essay', 'passage', 'prompt', 'response', 'article']

def _pick_text_col(frame):
    for c in _TEXT_CANDS:
        if c and c in frame.columns:
            return c
    excluded = {TEXT_ID, LABEL_COL, 'aggregates.lexicon_count'}
    for c in frame.columns:
        if c not in excluded and frame[c].dtype == object:
            return c
    return None

_text_col = _pick_text_col(_source_df)
if _text_col is None:
    raise ValueError('No text column found for textstat augmentation.')

texts_for_textstat = _source_df[_text_col].fillna('').astype(str)

def _safe_textstat(fn, t, default=np.nan):
    try:
        return fn(t)
    except Exception:
        return default

if 'textstat.linsear_write_formula' not in df_raw.columns:
    df_raw['textstat.linsear_write_formula'] = texts_for_textstat.apply(
        lambda t: _safe_textstat(textstat.linsear_write_formula, t))
if 'textstat.reading_time' not in df_raw.columns:
    df_raw['textstat.reading_time'] = texts_for_textstat.apply(
        lambda t: _safe_textstat(textstat.reading_time, t))
if 'textstat.spache_readability' not in df_raw.columns:
    df_raw['textstat.spache_readability'] = texts_for_textstat.apply(
        lambda t: _safe_textstat(textstat.spache_readability, t))
if 'textstat.text_standard' not in df_raw.columns:
    df_raw['textstat.text_standard'] = texts_for_textstat.apply(
        lambda t: _safe_textstat(lambda x: textstat.text_standard(x, float_output=True), t))

print('textstat columns added (or already present).')


textstat columns added (or already present).


In [12]:
# 2C — Drop ID/text columns (text already saved separately) + label filter
cols_to_drop = [c for c in [TEXT_ID, TEXT_COL] if c and c in df_raw.columns]
df_no_text = df_raw.drop(columns=cols_to_drop, errors='ignore')

# Drop rows with missing target labels
n_missing = int(df_no_text[LABEL_COL].isna().sum())
if n_missing:
    drop_mask_text = df_no_text[LABEL_COL].notna().to_numpy()
    df_no_text = df_no_text.dropna(subset=[LABEL_COL]).reset_index(drop=True)
    if _raw_texts_by_index is not None:
        _raw_texts_by_index = _raw_texts_by_index[drop_mask_text].reset_index(drop=True)
    print(f'Dropped {n_missing} rows with missing {LABEL_COL!r}')

# Restrict to TARGET_GRADES
keep_mask = df_no_text[LABEL_COL].isin(TARGET_GRADES).to_numpy()
df_no_text = df_no_text[keep_mask].reset_index(drop=True)
if _raw_texts_by_index is not None:
    _raw_texts_by_index = _raw_texts_by_index[keep_mask].reset_index(drop=True)

print(f'After grade filter: {df_no_text.shape}')
print('Class distribution:')
print(df_no_text[LABEL_COL].value_counts().sort_index())

# Stash texts for downstream usage (transformer / TF-IDF)
all_texts = _raw_texts_by_index.tolist() if _raw_texts_by_index is not None else None
print(f'Texts available for transformer/TF-IDF: {len(all_texts) if all_texts else 0}')


After grade filter: (172, 709)
Class distribution:
grade
2    26
3    19
4    21
5    27
6    27
7    26
8    26
Name: count, dtype: int64
Texts available for transformer/TF-IDF: 172


# SECTION 3 — Feature Group Taxonomy

Two sources of feature grouping are defined:

1. **`feature_groups`** — original NB1 grouping (D1_SURFACE, D2_LEXICAL, …) that maps friendly subgroup names to feature lists.
2. **`DIMENSION_FEATURE_MAP`** — comprehensive 10-dimension taxonomy (D1–D10) with toolkit precedence (LFTK > textstat > lingfeat > cohmetrix). When present, this map overrides `feature_groups` via the adapter.

If you have an `english_only_feature` list from NB1 (filtered set of features that exist in the workbook), it will take precedence in Section 4.


In [13]:
"""
COMPREHENSIVE TEXT COMPLEXITY DIMENSION → FEATURE MAPPING
==========================================================

Updated Taxonomy: 10 Dimensions across 4 Macro-Tiers.

Precedence for overlapping features: LFTK > textstat > lingfeat > cohmetrix

Toolkit Abbreviations:
  - LFTK:      LFTK library 
  - textstat:   textstat Python library
  - lingfeat:   LingFeat (brucewlee/lingfeat)
  - cohmetrix:  Coh-Metrix Core 
Each feature entry: {
    "toolkit": str,
    "code": str,          # toolkit's feature code
    "name": str,          # human-readable name
    "description": str    # what it measures
}

DISCARDED (due to precedence — feature exists in a preferred toolkit):

  LingFeat subgroups DISCARDED (102 features, all replaced by LFTK equivalents):
  - lingfeat ShaF_  (7 features: TokSenM/S/L, as_Token/Sylla/Chara counts) → LFTK t_word, t_sent, t_char, a_syll_pw etc.
  - lingfeat TraF_  (6 features: SmogInd/ColeLia/Gunning/AutoRea/FleschG/LinseaW) → LFTK fkre, fkgl, fogi, smog, cole, auto
  - lingfeat POSF_  (33 features: POS counts, ratios, content/function word ratios) → LFTK n_noun/n_verb/etc. (17 POS + unique variants = more granular)
  - lingfeat VarF_  (12 features: Simp/Squa/Corr × No/Ve/Aj/Av Variation) → LFTK simp/root/corr_*_var (15 POS types = more granular)
  - lingfeat TTRF_  (5 features: SimpTTR/CorrTTR/BiLoTTR/UberTTR/MTLDTTR) → LFTK simp_ttr, root_ttr, corr_ttr, bilog_ttr, uber_ttr
  - lingfeat PsyF_  (15 features: AoA from Kuperman/Bird/Bristol/CorteseKhanna) → LFTK a_kup_pw, a_bry_pw
  - lingfeat WorF_  (24 features: SubtlexUS frequency variants) → LFTK a_subtlex_us_zipf_pw (same corpus)

  LingFeat subgroups RETAINED (130 features, unique to lingfeat):
  - lingfeat TrSF_  (6 features): parse tree height/flattened tree → NOT in LFTK
  - lingfeat PhrF_  (48 features): phrasal counts + inter-phrase ratios → NOT in LFTK
  - lingfeat EnDF_  (6 features): entity mention density → partial overlap, retained for per-token granularity
  - lingfeat EnGF_  (22 features): entity grid transitions + local coherence → NOT in LFTK
  - lingfeat WoKF_  (16 features): Wikipedia LDA topic model → NOT in LFTK
  - lingfeat WBKF_  (16 features): WeeBit LDA topic model → NOT in LFTK
  - lingfeat OSKF_  (16 features): OneStopEng LDA topic model → NOT in LFTK

  CohMetrix indices DISCARDED (11 indices, all replaced by LFTK equivalents):
  - cohmetrix DESSC (sentence count) → LFTK t_sent
  - cohmetrix DESWC (word count) → LFTK t_word
  - cohmetrix DESSL (mean sentence length in words) → LFTK a_word_ps
  - cohmetrix DESWLsy (mean syllables per word) → LFTK a_syll_pw
  - cohmetrix DESWLlt (mean letters per word) → LFTK a_char_pw
  - cohmetrix READFRE / RDFRE (Flesch Reading Ease) → LFTK fkre
  - cohmetrix READFK / RDFKGL (Flesch-Kincaid Grade Level) → LFTK fkgl
  - cohmetrix LDTTRc (corrected TTR) → LFTK corr_ttr
  - cohmetrix LDTTRa (all-words TTR) → LFTK simp_ttr
  - cohmetrix WRDAOAc (MRC Age of Acquisition) → LFTK a_kup_pw / a_bry_pw (broader norms)
  NOTE: CohMetrix WRDAOAc is ALSO retained in D3.2 because it uses MRC norms (different source from Kuperman/Brysbaert).
"""

DIMENSION_FEATURE_MAP = {

    # =========================================================================
    # MACRO-TIER A: SUB-WORD & WORD LEVEL
    # =========================================================================

    "D1_Surface_Word_Level": {
        "description": "Surface-level word and phonological properties: word length, syllable counts, phonological regularity, orthographic patterns. These are the most basic decoding-level features.",
        "subdimensions": {

            "D1.1_Word_and_Sentence_Counts": {
                "description": "Raw counts of words, sentences, characters, paragraphs — foundational surface features, plus manual length-distribution statistics computed directly on the tokenised text.",
                "features": [

                    {"toolkit": "LFTK", "code": "t_sent", "name": "total_number_of_sentences", "description": "Total sentence count"},
                    {"toolkit": "LFTK", "code": "t_char", "name": "total_number_of_characters", "description": "Total character count"},
                    {"toolkit": "LFTK", "code": "t_uword", "name": "total_number_of_unique_words", "description": "Total unique word count (types)"},
                    {"toolkit": "LFTK", "code": "t_stopword", "name": "total_number_of_stop_words", "description": "Total stop words"},
                    {"toolkit": "LFTK", "code": "t_punct", "name": "total_number_of_punctuations", "description": "Total punctuation marks"},
                    {"toolkit": "LFTK", "code": "a_char_pw", "name": "avg_chars_per_word", "description": "Average character length per word"},
                    {"toolkit": "LFTK", "code": "a_stopword_pw", "name": "avg_stopwords_per_word", "description": "Proportion of stop words"},
                    {"toolkit": "cohmetrix", "code": "DESWLltd", "name": "word_length_letters_sd", "description": "SD of word length (letters)"},
                    {"toolkit": "manual", "code": "avg_word_length_chars", "name": "avg_word_length_chars", "description": "Mean orthographic word length in characters over all tokens matching [A-Za-z]+(?:'[A-Za-z]+)?"},
                    {"toolkit": "manual", "code": "cv_word_length_chars", "name": "cv_word_length_chars", "description": "Coefficient of variation (SD/mean) of word length in characters; length-normalised dispersion is comparable across stories of different lengths"},
                ]
            },

            "D1.2_Syllable_Features": {
                "description": "Syllable counts and polysyllabic word ratios — key phonological surface features. Manual entries use phonologically grounded IPA vowel-phone counts (espeak-ng en-gb), which corrects pyphen's well-known undercounts on words like ``rhythmic`` or ``forest``.",
                "features": [
                    {"toolkit": "LFTK", "code": "t_syll", "name": "total_syllables", "description": "Total syllable count in text"},
                    {"toolkit": "LFTK", "code": "t_syll2", "name": "total_words_more_than_2_syllables", "description": "Count of polysyllabic words (>2 syllables)"},
                    {"toolkit": "LFTK", "code": "t_syll3", "name": "total_words_more_than_3_syllables", "description": "Count of words with >3 syllables"},
                    {"toolkit": "LFTK", "code": "a_syll_ps", "name": "avg_syllables_per_sentence", "description": "Average syllables per sentence"},
                    {"toolkit": "LFTK", "code": "a_syll_pw", "name": "avg_syllables_per_word", "description": "Average syllables per word"},
                    {"toolkit": "cohmetrix", "code": "DESWLsyd", "name": "word_length_syllables_sd", "description": "SD of word length in syllables"},
                    {"toolkit": "manual", "code": "avg_syllables_per_word", "name": "avg_syllables_per_word", "description": "Mean syllables per word using IPA vowel-phone counts (espeak-ng en-gb); falls back to pyphen only when phonemiser fails"},
                    {"toolkit": "manual", "code": "cv_syllables_per_word", "name": "cv_syllables_per_word", "description": "Coefficient of variation of syllables-per-word; captures syllabic-length dispersion independently of mean length"},
                    {"toolkit": "manual", "code": "type_polysyllabic_ratio", "name": "type_polysyllabic_ratio", "description": "Proportion of unique types whose IPA-derived syllable count is ≥3; type-level rather than token-level avoids skew from repeated function-word tokens"},
                ]
            },

            "D1.3_Reading_Time": {
                "description": "Estimated reading time at different reading speeds.",
                "features": [
                    {"toolkit": "LFTK", "code": "rt_fast", "name": "reading_time_fast", "description": "Estimated reading time for fast readers (seconds)"},
                    {"toolkit": "LFTK", "code": "rt_average", "name": "reading_time_average", "description": "Estimated reading time for average readers"},
                    {"toolkit": "LFTK", "code": "rt_slow", "name": "reading_time_slow", "description": "Estimated reading time for slow readers"},
                    {"toolkit": "taales", "code": "LD_Mean_RT", "name": "lexical_decision_mean_rt", "description": "Mean lexical decision reaction time from the English Lexicon Project"},
                    {"toolkit": "taales", "code": "LD_Mean_RT_Zscore", "name": "lexical_decision_mean_rt_zscore", "description": "Z-scored mean lexical decision reaction time"},
                ]
            },

            "D1.4_Phonological_and_Orthographic_Depth": {
                "description": "Phonological complexity (clusters, onsets, GPC regularity) and orthographic-depth signals derived from espeak-ng IPA transcription (en-gb), plus a letter-level n-gram surprisal model trained on the NLTK Gutenberg corpus (~2.6M tokens of British literary English: Austen, Carroll, Chesterton, KJV, etc.). All features are stateless and text-only — no NCERT-corpus conditioning.",
                "features": [
                    {"toolkit": "manual", "code": "mean_chars_per_syllable", "name": "mean_chars_per_syllable", "description": "Mean orthographic characters per IPA-derived syllable; orthographic-depth proxy — words where many letters map to few syllables (e.g. ``through``, ``rhythm``) are flagged as deeper"},
                    {"toolkit": "manual", "code": "gpc_regularity", "name": "gpc_regularity", "description": "Mean |1 − n_alpha_chars / n_phonemes| across tokens; deviation from a 1-to-1 grapheme-to-phoneme correspondence (espeak-ng IPA). Higher = more irregular spelling/pronunciation"},
                    {"toolkit": "manual", "code": "consonant_cluster_density", "name": "consonant_cluster_density", "description": "Number of IPA consonant clusters of length ≥2 per word; captures syllable-margin complexity"},
                    {"toolkit": "manual", "code": "hard_consonant_cluster_ratio", "name": "hard_consonant_cluster_ratio", "description": "Proportion of words containing at least one IPA consonant cluster of length ≥3 (e.g. /str/ in ``strength``)"},
                    {"toolkit": "manual", "code": "onset_complexity_mean", "name": "onset_complexity_mean", "description": "Mean syllable-onset size in phonemes (sonority-aware Maximum Onset Principle split over IPA)"},
                    {"toolkit": "manual", "code": "syllable_complexity", "name": "syllable_complexity", "description": "Open-syllable ratio: proportion of syllables ending in a vowel phone after MOP segmentation; high values indicate CV-heavy (simpler) syllable structure"},
                    {"toolkit": "manual", "code": "letter_surprisal", "name": "letter_surprisal", "description": "Mean per-character interpolated bigram+trigram letter surprisal (bits), trained on NLTK Gutenberg with Laplace smoothing; captures orthographic predictability"},
                ]
            },
        }
    },


    "D2_Morphological_Complexity": {
        "description": "Morphological structure of words: morpheme counts, derivational/inflectional complexity, compound words, root-morpheme profile. Sourced primarily from TAMMI 2.0 (Crossley & MacArthur, linguisticanalysistools.org), which is built directly on the MorphoLex-en database (Sanchez-Gutierrez et al. 2017). The Morphological Complexity Index follows Brezina & Pallotti (2019). Three additional inflectional features TAMMI does not cover are computed via spaCy. Total D2 feature count: 45 (42 TAMMI + 3 spaCy supplements).",
        "subdimensions": {

            "D2.1_Basic_Morpheme_Counts": {
                "description": "Token-level counts and per-content-word averages of morpheme segments (prefixes, roots, suffixes, affixes). Form the raw substrate on which the MorphoLex per-cw means in later subgroups are computed.",
                "features": [
                    {"toolkit": "TAMMI", "code": "Inflected_Tokens", "name": "Inflected_Tokens", "description": "Count of content-word tokens carrying inflectional morphology (spaCy POS-driven match in TAMMI pipeline)"},
                    {"toolkit": "TAMMI", "code": "Derivational_Tokens", "name": "Derivational_Tokens", "description": "Count of content-word tokens carrying at least one derivational affix per MorphoLex"},
                    {"toolkit": "TAMMI", "code": "Tokens_w_Prefixes", "name": "Tokens_w_Prefixes", "description": "Count of content-word tokens with one or more prefixes"},
                    {"toolkit": "TAMMI", "code": "Tokens_w_Affixes", "name": "Tokens_w_Affixes", "description": "Count of content-word tokens with one or more affixes (prefix + suffix slots combined)"},
                    {"toolkit": "TAMMI", "code": "number_prefixes_per_cw", "name": "number_prefixes_per_cw", "description": "Mean number of prefixes per content word"},
                    {"toolkit": "TAMMI", "code": "number_roots_per_cw", "name": "number_roots_per_cw", "description": "Mean number of root morphemes per content word"},
                    {"toolkit": "TAMMI", "code": "number_suffixes_per_cw", "name": "number_suffixes_per_cw", "description": "Mean number of suffixes per content word"},
                    {"toolkit": "TAMMI", "code": "number_affixes_per_cw", "name": "number_affixes_per_cw", "description": "Mean number of affixes per content word"},
                    {"toolkit": "TAMMI", "code": "num_roots_affixes_per_cw", "name": "num_roots_affixes_per_cw", "description": "Mean number of root+affix morphemes per content word"},
                    {"toolkit": "TAMMI", "code": "num_morphemes_per_cw", "name": "num_morphemes_per_cw", "description": "Mean total morphemes (prefixes + roots + suffixes) per content word"},
                ]
            },

            "D2.2_Derivational_Complexity": {
                "description": "Per-cw MorphoLex statistics over derivational morphemes (prefix slots, suffix slots, and the combined affix pool), plus derivational morphological complexity over 10-content-word windows. Each MorphoLex statistic — family size, frequency, log-frequency, length, hapax counts, and PFMF (Percent of Morpho-family with More Frequent words) — is averaged across all affix slots of each content word and then over all content words in the text.",
                "features": [
                    {"toolkit": "TAMMI", "code": "perc_more_freq_words_morpho_family_prefix_per_cw", "name": "perc_more_freq_words_morpho_family_prefix_per_cw", "description": "PFMF — percent of more frequent words in the prefix's morphological family, averaged per cw"},
                    {"toolkit": "TAMMI", "code": "prefix_family_size_per_cw", "name": "prefix_family_size_per_cw", "description": "Mean morphological-family size of the word's prefix(es), per cw"},
                    {"toolkit": "TAMMI", "code": "prefix_freq_per_cw", "name": "prefix_freq_per_cw", "description": "Mean MorphoLex prefix frequency per cw"},
                    {"toolkit": "TAMMI", "code": "prefix_log_freq_per_cw", "name": "prefix_log_freq_per_cw", "description": "Mean log-frequency of the prefix(es) per cw"},
                    {"toolkit": "TAMMI", "code": "prefix_len_per_cw", "name": "prefix_len_per_cw", "description": "Mean orthographic length of the prefix(es) per cw"},
                    {"toolkit": "TAMMI", "code": "prefix_in_hapax_per_cw", "name": "prefix_in_hapax_per_cw", "description": "Mean count of hapax legomena (MorphoLex frequency 1) the prefix appears in, per cw"},
                    {"toolkit": "TAMMI", "code": "hapax_in_prefix_per_cw", "name": "hapax_in_prefix_per_cw", "description": "Mean count of hapax legomena within the prefix's morphological family, per cw"},
                    {"toolkit": "TAMMI", "code": "perc_more_freq_words_morpho_family_suffix_per_cw", "name": "perc_more_freq_words_morpho_family_suffix_per_cw", "description": "Suffix PFMF, per cw"},
                    {"toolkit": "TAMMI", "code": "suffix_family_size_per_cw", "name": "suffix_family_size_per_cw", "description": "Mean morphological-family size of the suffix(es), per cw"},
                    {"toolkit": "TAMMI", "code": "suffix_freq_per_cw", "name": "suffix_freq_per_cw", "description": "Mean MorphoLex suffix frequency per cw"},
                    {"toolkit": "TAMMI", "code": "suffix_log_freq_per_cw", "name": "suffix_log_freq_per_cw", "description": "Mean log-frequency of the suffix(es) per cw"},
                    {"toolkit": "TAMMI", "code": "suffix_len_per_cw", "name": "suffix_len_per_cw", "description": "Mean orthographic length of the suffix(es) per cw"},
                    {"toolkit": "TAMMI", "code": "suffix_in_hapax_per_cw", "name": "suffix_in_hapax_per_cw", "description": "Mean count of hapax legomena the suffix appears in, per cw"},
                    {"toolkit": "TAMMI", "code": "hapax_in_suffix_per_cw", "name": "hapax_in_suffix_per_cw", "description": "Mean count of hapax legomena within the suffix's morphological family, per cw"},
                    {"toolkit": "TAMMI", "code": "perc_more_freq_words_morpho_family_affix_per_cw", "name": "perc_more_freq_words_morpho_family_affix_per_cw", "description": "Combined-affix PFMF (prefix + suffix slots pooled), per cw"},
                    {"toolkit": "TAMMI", "code": "affix_family_size_per_cw", "name": "affix_family_size_per_cw", "description": "Mean morphological-family size of any affix (prefix or suffix), per cw"},
                    {"toolkit": "TAMMI", "code": "affix_freq_per_cw", "name": "affix_freq_per_cw", "description": "Mean MorphoLex affix frequency per cw"},
                    {"toolkit": "TAMMI", "code": "affix_log_freq_per_cw", "name": "affix_log_freq_per_cw", "description": "Mean log-frequency of affixes per cw"},
                    {"toolkit": "TAMMI", "code": "affix_len_per_cw", "name": "affix_len_per_cw", "description": "Mean orthographic length of affixes per cw"},
                    {"toolkit": "TAMMI", "code": "affix_in_hapax_per_cw", "name": "affix_in_hapax_per_cw", "description": "Mean count of hapax legomena any affix appears in, per cw"},
                    {"toolkit": "TAMMI", "code": "hapax_in_affix_per_cw", "name": "hapax_in_affix_per_cw", "description": "Mean count of hapax legomena within affixes' morphological families, per cw"},
                    {"toolkit": "TAMMI", "code": "mean_subset_derivational_variety_10", "name": "mean_subset_derivational_variety_10", "description": "Mean count of unique derivational affixes across non-overlapping 10-cw windows"},
                    {"toolkit": "TAMMI", "code": "derivational_TTR_10", "name": "derivational_TTR_10", "description": "Type-token ratio of derivational affixes computed within each 10-cw window and averaged"},
                    {"toolkit": "TAMMI", "code": "derivational_MCI_10", "name": "derivational_MCI_10", "description": "Morphological Complexity Index (Brezina & Pallotti 2019) for derivational affixes over 10-cw windows"},
                ]
            },

            "D2.3_Inflectional_Complexity": {
                "description": "Inflectional variety / TTR / MCI over 10-cw windows from TAMMI, plus three spaCy-derived inflectional features that TAMMI does not cover (irregular-verb density, comparative/superlative density, and inflection-class entropy over morphological feature tuples).",
                "features": [
                    {"toolkit": "TAMMI", "code": "mean_subset_inflectional_variety_10", "name": "mean_subset_inflectional_variety_10", "description": "Mean count of unique inflectional forms across non-overlapping 10-cw windows"},
                    {"toolkit": "TAMMI", "code": "inflectional_TTR_10", "name": "inflectional_TTR_10", "description": "Type-token ratio of inflectional forms within each 10-cw window, averaged"},
                    {"toolkit": "TAMMI", "code": "inflectional_MCI_10", "name": "inflectional_MCI_10", "description": "Morphological Complexity Index (Brezina & Pallotti 2019) for inflectional forms over 10-cw windows"},
                ]
            },

            "D2.4_Compound_Words": {
                "description": "Compound-word detection via MorphoLex (a token counts as a compound when its morpheme tree has ≥2 root morphemes).",
                "features": [
                    {"toolkit": "TAMMI", "code": "Compounds", "name": "Compounds", "description": "Count of content-word tokens classified as compounds (≥2 root morphemes in their MorphoLex parse)"},
                ]
            },

            "D2.5_Root_Morpheme_Profile": {
                "description": "Per-cw MorphoLex statistics for the root morpheme(s) of each content word. Roots carry most of the lexical-semantic load and their family size / frequency profile is a distinct dimension from derivational affix profile.",
                "features": [
                    {"toolkit": "TAMMI", "code": "perc_more_freq_words_morpho_family_root", "name": "perc_more_freq_words_morpho_family_root", "description": "Root PFMF — percent of more frequent words in the root's morphological family, averaged per cw"},
                    {"toolkit": "TAMMI", "code": "root_family_size_per_cw", "name": "root_family_size_per_cw", "description": "Mean morphological-family size of the root(s) per cw"},
                    {"toolkit": "TAMMI", "code": "root_freq_per_cw", "name": "root_freq_per_cw", "description": "Mean MorphoLex root frequency per cw"},
                    {"toolkit": "TAMMI", "code": "root_log_freq_per_cw", "name": "root_log_freq_per_cw", "description": "Mean log-frequency of the root(s) per cw"},
                ]
            },
        }
    },


    "D3_Lexical_Sophistication": {
        "description": "Word-level difficulty: frequency, familiarity, age of acquisition, concreteness, imageability, vocabulary band, polysemy, hypernymy, psycholinguistic properties.",
        "subdimensions": {

            "D3.1_Word_Frequency": {
                "description": "How frequent/common each word is in reference corpora. Toolkit entries use SubtlexUS (LFTK), CELEX (cohmetrix), and Kucera-Francis (taales); the manual entries are computed via ``wordfreq`` on the Zipf scale (log10 occurrences per billion) over lemmatised, NER-filtered content tokens.",
                "features": [
                    {"toolkit": "LFTK", "code": "t_subtlex_us_zipf", "name": "total_subtlex_us_zipf", "description": "Total Subtlex US Zipf frequency of all words"},
                    {"toolkit": "LFTK", "code": "a_subtlex_us_zipf_pw", "name": "avg_subtlex_us_zipf_per_word", "description": "Average Subtlex US Zipf frequency per word"},
                    {"toolkit": "LFTK", "code": "a_subtlex_us_zipf_ps", "name": "avg_subtlex_us_zipf_per_sentence", "description": "Average Subtlex US Zipf frequency per sentence"},
                    # cohmetrix frequency — kept because it uses CELEX (different corpus from SubtlexUS)
                    {"toolkit": "cohmetrix", "code": "WRDFRQc", "name": "word_frequency_content_words", "description": "Avg CELEX word frequency for content words"},
                    {"toolkit": "cohmetrix", "code": "WRDFRQa", "name": "word_frequency_all_words", "description": "Avg CELEX word frequency for all words"},
                    {"toolkit": "cohmetrix", "code": "WRDFRQmc", "name": "word_frequency_min_in_sentences", "description": "Avg minimum word frequency in sentences"},
                    {"toolkit": "taales", "code": "KF_Nsamp_AW", "name": "kf_nsamp_all_words", "description": "Kucera-Francis sample frequency count for all words"},
                    {"toolkit": "manual", "code": "log_word_frequency_zipf", "name": "log_word_frequency_zipf", "description": "Mean ``wordfreq`` Zipf frequency (log10 per billion, scale ≈0–8) over lemmatised NER-filtered content tokens"},
                    {"toolkit": "manual", "code": "low_freq_ratio_zipf_under_2", "name": "low_freq_ratio_zipf_under_2", "description": "Proportion of tokens with Zipf < 2.0 — very rare words (e.g. ``crepuscular``, ``foible``)"},
                    {"toolkit": "manual", "code": "low_freq_ratio_zipf_under_3", "name": "low_freq_ratio_zipf_under_3", "description": "Proportion of tokens with Zipf < 3.0 — rare/specialist vocabulary"},
                    {"toolkit": "manual", "code": "low_freq_ratio_zipf_under_4", "name": "low_freq_ratio_zipf_under_4", "description": "Proportion of tokens with Zipf < 4.0 — uncommon vocabulary (below typical conversational threshold)"},
                    {"toolkit": "manual", "code": "mean_zipf_below_median", "name": "mean_zipf_below_median", "description": "Mean Zipf among tokens at or below the text's own median Zipf — characterises the rare half of the text's vocabulary"},
                ]
            },

            "D3.2_Age_of_Acquisition": {
                "description": "How early in life words are typically learned.",
                "features": [
                    {"toolkit": "LFTK", "code": "t_kup", "name": "total_kuperman_aoa", "description": "Total Kuperman AoA across all words"},
                    {"toolkit": "LFTK", "code": "t_bry", "name": "total_brysbaert_aoa", "description": "Total Brysbaert AoA across all words"},
                    {"toolkit": "LFTK", "code": "a_kup_pw", "name": "avg_kuperman_aoa_per_word", "description": "Average Kuperman AoA per word"},
                    {"toolkit": "LFTK", "code": "a_bry_pw", "name": "avg_brysbaert_aoa_per_word", "description": "Average Brysbaert AoA per word"},
                    {"toolkit": "LFTK", "code": "a_kup_ps", "name": "avg_kuperman_aoa_per_sentence", "description": "Average Kuperman AoA per sentence"},
                    {"toolkit": "LFTK", "code": "a_bry_ps", "name": "avg_brysbaert_aoa_per_sentence", "description": "Average Brysbaert AoA per sentence"},
                    # cohmetrix AoA uses MRC norms (Gilhooly & Logie) — different from Kuperman/Brysbaert
                    {"toolkit": "cohmetrix", "code": "WRDAOAc", "name": "aoa_mrc_content_words", "description": "Avg MRC Age of Acquisition for content words"},
                    {"toolkit": "taales", "code": "aoe_inverse_average", "name": "aoe_inverse_average", "description": "Inverse average of age of exposure ratings"},
                    {"toolkit": "taales", "code": "aoe_index_above_threshold_40", "name": "aoe_index_above_threshold_40", "description": "Proportion of words with AoE > 40 months"},
                ]
            },

            "D3.3_Psycholinguistic_Word_Properties": {
                "description": "Familiarity, concreteness, imageability, meaningfulness, valence, arousal.",
                "features": [
                    {"toolkit": "cohmetrix", "code": "WRDFAMc", "name": "word_familiarity_content", "description": "Avg MRC familiarity rating for content words"},
                    {"toolkit": "manual", "code": "mean_concreteness", "name": "mean_concreteness", "description": "Mean concreteness of content-word lemmas on 1–5 scale"},
                    {"toolkit": "cohmetrix", "code": "WRDIMGc", "name": "word_imageability_content", "description": "Avg MRC imageability for content words"},
                    {"toolkit": "cohmetrix", "code": "WRDMEAc", "name": "word_meaningfulness_content", "description": "Avg Toglia-Battig meaningfulness for content words"},
                    {"toolkit": "cohmetrix", "code": "WRDVALm", "name": "word_valence_mean", "description": "Avg word valence (emotional positivity)"},
                    {"toolkit": "cohmetrix", "code": "WRDVALmc", "name": "word_valence_content", "description": "Avg word valence for content words"},
                    {"toolkit": "cohmetrix", "code": "WRDTRNm", "name": "word_arousal_mean", "description": "Avg word arousal"},
                    {"toolkit": "cohmetrix", "code": "WRDTRNmc", "name": "word_arousal_content", "description": "Avg word arousal for content words"},
                ]
            },

            "D3.4_Word_Specificity_and_Polysemy": {
                "description": "How specific or ambiguous words are (WordNet-based).",
                "features": [
                    {"toolkit": "cohmetrix", "code": "WRDPOLc", "name": "polysemy_content_words", "description": "Avg WordNet polysemy (number of senses) for content words"},
                    {"toolkit": "cohmetrix", "code": "WRDHYPn", "name": "hypernymy_nouns", "description": "Avg WordNet hypernymy depth for nouns (word specificity)"},
                    {"toolkit": "cohmetrix", "code": "WRDHYPv", "name": "hypernymy_verbs", "description": "Avg WordNet hypernymy depth for verbs"},
                    {"toolkit": "cohmetrix", "code": "WRDHYPnv", "name": "hypernymy_nouns_verbs", "description": "Avg WordNet hypernymy depth for nouns+verbs"},
                ]
            },

            "D3.5_Vocabulary_Band": {
                "description": "Coverage against graded vocabulary lists. The Oxford 5000 (combined 3000 base + 2000 additional) is consulted by lemma+POS first, then by lemma alone; Dale-Chall coverage uses the textstat-bundled easy-word list. UWL/AWL features were removed in favour of these two robust lists.",
                "features": [
                    {"toolkit": "cohmetrix", "code": "READNDC", "name": "dale_chall_readability", "description": "New Dale-Chall readability (uses Dale-Chall 3000 word list)"},
                    {"toolkit": "manual", "code": "oxford_5000_coverage", "name": "oxford_5000_coverage", "description": "Proportion of non-entity, NER-filtered content tokens whose (lemma, Oxford-POS) pair is in the Oxford 5000 (3000 base + 2000 additional); lemma-only fallback when POS lookup misses"},
                    {"toolkit": "manual", "code": "dale_chall_3000_coverage", "name": "dale_chall_3000_coverage", "description": "Proportion of non-entity content tokens whose lemma appears in the Dale-Chall ~3000 easy-word list bundled with textstat"},
                ]
            },

            "D3.6_Ngram_Collocational_Strength": {
                "description": "Surface n-gram frequency normalised against the NLTK Gutenberg corpus (~2.6M tokens of British literary English). PMI and T-score features were dropped: PMI is noise-dominated at the per-story scale (Brezina 2018 §3.3) and t-score correlates ~.99 with raw bigram log-frequency on NCERT, so a single per-million signal is more interpretable.",
                "features": [
                    {"toolkit": "manual", "code": "avg_bigram_freq_per_million", "name": "avg_bigram_freq_per_million", "description": "Mean Gutenberg bigram frequency of consecutive content-word pairs, normalised per million Gutenberg tokens"},
                    {"toolkit": "manual", "code": "avg_trigram_freq_per_million", "name": "avg_trigram_freq_per_million", "description": "Mean Gutenberg trigram frequency of consecutive content-word triples, normalised per million Gutenberg tokens"},
                    {"toolkit": "taales", "code": "BNC_Written_Bigram_Proportion", "name": "bnc_written_bigram_proportion", "description": "Proportion of bigrams from BNC written corpus"},
                    {"toolkit": "taales", "code": "BNC_Spoken_Freq_FW_Log", "name": "bnc_spoken_freq_log", "description": "Log-transformed BNC spoken frequency for function words"},
                ]
            },

            "D3.7_Lexical_Surprisal": {
                "description": "Information-theoretic predictability (Hale 2001; Levy 2008): how many bits of information each lexical item contributes given (a) no prior context and (b) the preceding word. Both models are estimated on NLTK Gutenberg with add-1 (Laplace) smoothing over the lemma vocabulary; surprisal of each token is ``-log2 P(w)`` or ``-log2 P(w_t | w_{t-1})``, then averaged across the text. Less predictable text ⇒ higher mean surprisal ⇒ higher integration cost for the reader.",
                "features": [
                    {"toolkit": "manual", "code": "lexical_surprisal_mean", "name": "lexical_surprisal_mean", "description": "Mean unigram surprisal in bits, ``-log2 P(w)`` with Laplace smoothing on Gutenberg unigram counts; high values flag rare-word-heavy texts"},
                    {"toolkit": "manual", "code": "forward_conditional_surprisal_mean", "name": "forward_conditional_surprisal_mean", "description": "Mean forward conditional surprisal in bits, ``-log2 P(w_t | w_{t-1})`` with Laplace smoothing on Gutenberg bigram counts; high values flag unexpected word-to-word transitions and unusual collocations"},
                ]
            },

            "D3.8_Orthographic_Neighbourhood": {
                "description": "Orthographic neighbourhood density (Coltheart, Davelaar, Jonasson & Besner 1977): for each token, count how many real English words differ by exactly one letter substitution (same length). Dense neighbourhoods (e.g. ``cat`` → bat/hat/mat/...) facilitate visual word recognition; sparse neighbourhoods (e.g. ``yacht``, ``chrysanthemum``) increase decoding effort. Computed against the Gutenberg lexicon restricted to types occurring ≥5 times; cached via ``lru_cache``.",
                "features": [
                    {"toolkit": "manual", "code": "coltheart_n_mean", "name": "coltheart_n_mean", "description": "Mean Coltheart N — number of one-letter-substitution neighbours of each surface token among Gutenberg word types occurring ≥5 times; computed only on alphabetic tokens of length ≥2"},
                ]
            },
        }
    },



    # =========================================================================
    # MACRO-TIER B: SENTENCE LEVEL
    # =========================================================================

    "D4_Syntactic_Complexity": {
        "description": "Sentence-level grammatical complexity: parse tree depth, clause density, phrasal complexity, sentence structure similarity, syntactic pattern density.",
        "subdimensions": {

            "D4.1_Parse_Tree_Structure": {
                "description": "Depth and shape of constituency/dependency parse trees.",
                "features": [
                    {"toolkit": "lingfeat", "code": "to_TreeH_C", "name": "total_tree_height", "description": "Total parse tree height across all sentences"},
                    {"toolkit": "lingfeat", "code": "as_TreeH_C", "name": "avg_tree_height_per_sentence", "description": "Average parse tree height per sentence"},
                    {"toolkit": "lingfeat", "code": "at_TreeH_C", "name": "avg_tree_height_per_token", "description": "Average parse tree height per token"},
                    {"toolkit": "lingfeat", "code": "to_FTree_C", "name": "total_flattened_tree_length", "description": "Total flattened tree length"},
                    {"toolkit": "lingfeat", "code": "as_FTree_C", "name": "avg_flattened_tree_per_sentence", "description": "Average flattened tree length per sentence"},
                    {"toolkit": "lingfeat", "code": "at_FTree_C", "name": "avg_flattened_tree_per_token", "description": "Average flattened tree length per token"},
                ]
            },

            "D4.2_Sentence_Structure_and_Embedding": {
                "description": "Left embeddedness, sentence structure similarity, minimum edit distance between sentences.",
                "features": [
                    {"toolkit": "cohmetrix", "code": "SYNLE", "name": "left_embeddedness", "description": "Mean words before main verb of main clause (working memory load)"},
                    {"toolkit": "cohmetrix", "code": "SYNSTRUTa", "name": "syntactic_similarity_adjacent", "description": "Proportion of intersection tree nodes between adjacent sentences"},
                    {"toolkit": "cohmetrix", "code": "SYNSTRUTt", "name": "syntactic_similarity_all", "description": "Proportion of intersection tree nodes between all sentences"},
                    {"toolkit": "cohmetrix", "code": "SYNMEDpos", "name": "min_edit_distance_pos", "description": "Mean min edit distance between adjacent sentences (POS tags)"},
                    {"toolkit": "cohmetrix", "code": "SYNMEDwrd", "name": "min_edit_distance_words", "description": "Mean min edit distance between adjacent sentences (words)"},
                    {"toolkit": "cohmetrix", "code": "SYNMEDlem", "name": "min_edit_distance_lemmas", "description": "Mean min edit distance between adjacent sentences (lemmas)"},
                ]
            },

            "D4.3_Phrasal_Complexity": {
                "description": "Density and ratios of noun phrases, verb phrases, adverbial phrases, prepositional phrases, subordinate clauses.",
                "features": [
                    {"toolkit": "cohmetrix", "code": "SYNNP", "name": "modifiers_per_noun_phrase", "description": "Mean number of modifiers per noun phrase"},
                    {"toolkit": "cohmetrix", "code": "DRNP", "name": "noun_phrase_incidence", "description": "Incidence score of noun phrases (per 1000 words)"},
                    {"toolkit": "cohmetrix", "code": "DRVP", "name": "verb_phrase_incidence", "description": "Incidence score of verb phrases"},
                    {"toolkit": "cohmetrix", "code": "DRAP", "name": "adverbial_phrase_incidence", "description": "Incidence score of adverbial phrases"},
                    {"toolkit": "cohmetrix", "code": "DRPP", "name": "prepositional_phrase_incidence", "description": "Incidence score of prepositional phrases"},
                    # lingfeat PhrF_ — retained because it provides inter-phrase RATIOS not in cohmetrix
                    {"toolkit": "lingfeat", "code": "to_NoPhr_C", "name": "total_noun_phrases", "description": "Total noun phrase count"},
                    {"toolkit": "lingfeat", "code": "PhrF_:as_NoPhr_C", "name": "avg_noun_phrases_per_sentence", "description": "Avg noun phrases per sentence"},
                    {"toolkit": "lingfeat", "code": "PhrF_:at_NoPhr_C", "name": "avg_noun_phrases_per_token", "description": "Avg noun phrases per token"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_NoVeP_C", "name": "ratio_NP_to_VP", "description": "Ratio of noun phrases to verb phrases"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_NoSuP_C", "name": "ratio_NP_to_SubClauses", "description": "Ratio of noun phrases to subordinate clauses"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_NoPrP_C", "name": "ratio_NP_to_PP", "description": "Ratio of noun phrases to prepositional phrases"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_NoAjP_C", "name": "ratio_NP_to_AdjP", "description": "Ratio of noun phrases to adjective phrases"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_NoAvP_C", "name": "ratio_NP_to_AdvP", "description": "Ratio of noun phrases to adverb phrases"},
                    {"toolkit": "lingfeat", "code": "PhrF_:to_VePhr_C", "name": "total_verb_phrases", "description": "Total verb phrase count"},
                    {"toolkit": "lingfeat", "code": "PhrF_:as_VePhr_C", "name": "avg_verb_phrases_per_sentence", "description": "Avg verb phrases per sentence"},
                    {"toolkit": "lingfeat", "code": "PhrF_:at_VePhr_C", "name": "avg_verb_phrases_per_token", "description": "Avg verb phrases per token"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_VeNoP_C", "name": "ratio_VP_to_NP", "description": "Ratio of verb phrases to noun phrases"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_VeSuP_C", "name": "ratio_VP_to_SubClauses", "description": "Ratio of verb phrases to subordinate clauses"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_VePrP_C", "name": "ratio_VP_to_PP", "description": "Ratio of verb phrases to prepositional phrases"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_VeAjP_C", "name": "ratio_VP_to_AdjP", "description": "Ratio of verb phrases to adjective phrases"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_VeAvP_C", "name": "ratio_VP_to_AdvP", "description": "Ratio of verb phrases to adverb phrases"},
                    {"toolkit": "lingfeat", "code": "PhrF_:to_SuPhr_C", "name": "total_subordinate_clauses", "description": "Total subordinate clause count"},
                    {"toolkit": "lingfeat", "code": "PhrF_:as_SuPhr_C", "name": "avg_subordinate_clauses_per_sentence", "description": "Avg subordinate clauses per sentence"},
                    {"toolkit": "lingfeat", "code": "PhrF_:at_SuPhr_C", "name": "avg_subordinate_clauses_per_token", "description": "Avg subordinate clauses per token"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_SuNoP_C", "name": "ratio_SubCl_to_NP", "description": "Ratio of subordinate clauses to noun phrases"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_SuVeP_C", "name": "ratio_SubCl_to_VP", "description": "Ratio of subordinate clauses to verb phrases"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_SuPrP_C", "name": "ratio_SubCl_to_PP", "description": "Ratio of subordinate clauses to prepositional phrases"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_SuAjP_C", "name": "ratio_SubCl_to_AdjP", "description": "Ratio of subordinate clauses to adjective phrases"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_SuAvP_C", "name": "ratio_SubCl_to_AdvP", "description": "Ratio of subordinate clauses to adverb phrases"},
                    {"toolkit": "lingfeat", "code": "PhrF_:to_PrPhr_C", "name": "total_prepositional_phrases", "description": "Total prepositional phrase count"},
                    {"toolkit": "lingfeat", "code": "PhrF_:as_PrPhr_C", "name": "avg_prepositional_phrases_per_sentence", "description": "Avg prepositional phrases per sentence"},
                    {"toolkit": "lingfeat", "code": "PhrF_:at_PrPhr_C", "name": "avg_prepositional_phrases_per_token", "description": "Avg prepositional phrases per token"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_PrNoP_C", "name": "ratio_PP_to_NP", "description": "Ratio of PP to NP"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_PrVeP_C", "name": "ratio_PP_to_VP", "description": "Ratio of PP to VP"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_PrSuP_C", "name": "ratio_PP_to_SubCl", "description": "Ratio of PP to subordinate clauses"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_PrAjP_C", "name": "ratio_PP_to_AdjP", "description": "Ratio of PP to adjective phrases"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_PrAvP_C", "name": "ratio_PP_to_AdvP", "description": "Ratio of PP to adverb phrases"},
                    {"toolkit": "lingfeat", "code": "PhrF_:to_AjPhr_C", "name": "total_adjective_phrases", "description": "Total adjective phrase count"},
                    {"toolkit": "lingfeat", "code": "PhrF_:as_AjPhr_C", "name": "avg_adjective_phrases_per_sentence", "description": "Avg adjective phrases per sentence"},
                    {"toolkit": "lingfeat", "code": "PhrF_:at_AjPhr_C", "name": "avg_adjective_phrases_per_token", "description": "Avg adjective phrases per token"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_AjNoP_C", "name": "ratio_AdjP_to_NP", "description": "Ratio of adjective phrases to noun phrases"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_AjVeP_C", "name": "ratio_AdjP_to_VP", "description": "Ratio of adjective phrases to verb phrases"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_AjSuP_C", "name": "ratio_AdjP_to_SubCl", "description": "Ratio of adjective phrases to subordinate clauses"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_AjPrP_C", "name": "ratio_AdjP_to_PP", "description": "Ratio of adjective phrases to PP"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_AjAvP_C", "name": "ratio_AdjP_to_AdvP", "description": "Ratio of adjective phrases to adverb phrases"},
                    {"toolkit": "lingfeat", "code": "PhrF_:to_AvPhr_C", "name": "total_adverb_phrases", "description": "Total adverb phrase count"},
                    {"toolkit": "lingfeat", "code": "PhrF_:as_AvPhr_C", "name": "avg_adverb_phrases_per_sentence", "description": "Avg adverb phrases per sentence"},
                    {"toolkit": "lingfeat", "code": "PhrF_:at_AvPhr_C", "name": "avg_adverb_phrases_per_token", "description": "Avg adverb phrases per token"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_AvNoP_C", "name": "ratio_AdvP_to_NP", "description": "Ratio of adverb phrases to noun phrases"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_AvVeP_C", "name": "ratio_AdvP_to_VP", "description": "Ratio of adverb phrases to verb phrases"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_AvSuP_C", "name": "ratio_AdvP_to_SubCl", "description": "Ratio of adverb phrases to subordinate clauses"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_AvPrP_C", "name": "ratio_AdvP_to_PP", "description": "Ratio of adverb phrases to PP"},
                    {"toolkit": "lingfeat", "code": "PhrF_:ra_AvAjP_C", "name": "ratio_AdvP_to_AdjP", "description": "Ratio of adverb phrases to adjective phrases"},
                ]
            },

            "D4.4_Syntactic_Pattern_Density": {
                "description": "Incidence of special syntactic patterns: passive voice, negation, gerunds, infinitives.",
                "features": [
                    {"toolkit": "cohmetrix", "code": "DRPVAL", "name": "passive_voice_incidence", "description": "Incidence score of agentless passive voice forms"},
                    {"toolkit": "cohmetrix", "code": "DRNEG", "name": "negation_incidence", "description": "Incidence score of negation expressions"},
                    {"toolkit": "cohmetrix", "code": "DRGERUND", "name": "gerund_incidence", "description": "Incidence score of gerunds"},
                    {"toolkit": "cohmetrix", "code": "DRINF", "name": "infinitive_incidence", "description": "Incidence score of infinitives"},
                   
                ]
            },

            "D4.5_Average_Word_and_Sentence_Length": {
                "description": "Averaged surface measures: words per sentence, characters per word, syllables per word.",
                "features": [
                    {"toolkit": "LFTK", "code": "a_word_ps", "name": "avg_words_per_sentence", "description": "Average number of words per sentence"},
                    {"toolkit": "LFTK", "code": "a_char_ps", "name": "avg_chars_per_sentence", "description": "Average characters per sentence"},
                    {"toolkit": "LFTK", "code": "a_stopword_ps", "name": "avg_stopwords_per_sentence", "description": "Average stop words per sentence"},
                    {"toolkit": "cohmetrix", "code": "DESSLd", "name": "sentence_length_sd", "description": "SD of sentence length (words)"},
                    
                ]
            },
        }
    },


    "D5_Grammatical_Density_and_Diversity": {
        "description": "Distribution and diversity of parts of speech. POS density reveals informational load (noun-heavy = dense, verb-heavy = action-rich). POS variation reveals lexical diversity within each category.",
        "subdimensions": {

          
            "D5.1_POS_Density_Per_Word": {
                "description": "Proportion of each POS per word (= POS density).",
                "features": [
                    {"toolkit": "LFTK", "code": "a_adj_pw", "name": "adjective_density", "description": "Adjectives / total words"},
                    {"toolkit": "LFTK", "code": "a_adp_pw", "name": "adposition_density", "description": "Adpositions / total words"},
                    {"toolkit": "LFTK", "code": "a_adv_pw", "name": "adverb_density", "description": "Adverbs / total words"},
                    {"toolkit": "LFTK", "code": "a_aux_pw", "name": "auxiliary_density", "description": "Auxiliaries / total words"},
                    {"toolkit": "LFTK", "code": "a_cconj_pw", "name": "coordinating_conjunction_density", "description": "Coord. conjunctions / total words"},
                    {"toolkit": "LFTK", "code": "a_det_pw", "name": "determiner_density", "description": "Determiners / total words"},
                    {"toolkit": "LFTK", "code": "a_intj_pw", "name": "interjection_density", "description": "Interjections / total words"},
                    {"toolkit": "LFTK", "code": "a_noun_pw", "name": "noun_density", "description": "Nouns / total words"},
                    {"toolkit": "LFTK", "code": "a_num_pw", "name": "numeral_density", "description": "Numerals / total words"},
                    {"toolkit": "LFTK", "code": "a_part_pw", "name": "particle_density", "description": "Particles / total words"},
                    {"toolkit": "LFTK", "code": "a_pron_pw", "name": "pronoun_density", "description": "Pronouns / total words"},
                    {"toolkit": "LFTK", "code": "a_propn_pw", "name": "proper_noun_density", "description": "Proper nouns / total words"},
                    {"toolkit": "LFTK", "code": "a_sconj_pw", "name": "subordinating_conjunction_density", "description": "Subord. conjunctions / total words"},
                    {"toolkit": "LFTK", "code": "a_sym_pw", "name": "symbol_density", "description": "Symbols / total words"},
                    {"toolkit": "LFTK", "code": "a_verb_pw", "name": "verb_density", "description": "Verbs / total words"},
                    {"toolkit": "LFTK", "code": "a_punct_pw", "name": "punctuation_density", "description": "Punctuation POS / total words"},
                    {"toolkit": "LFTK", "code": "a_space_pw", "name": "whitespace_density", "description": "Whitespace POS / total words"},
                    # cohmetrix incidence scores (per 1000 words) — kept since they use Charniak parser POS (PTB tagset), different granularity
                    {"toolkit": "cohmetrix", "code": "WRDNOUN", "name": "noun_incidence_1000", "description": "Noun incidence score (per 1000 words, PTB tags)"},
                    {"toolkit": "cohmetrix", "code": "WRDVERB", "name": "verb_incidence_1000", "description": "Verb incidence score (per 1000 words)"},
                    {"toolkit": "cohmetrix", "code": "WRDADJ", "name": "adjective_incidence_1000", "description": "Adjective incidence score (per 1000 words)"},
                    {"toolkit": "cohmetrix", "code": "WRDADV", "name": "adverb_incidence_1000", "description": "Adverb incidence score (per 1000 words)"},
                    {"toolkit": "cohmetrix", "code": "WRDPRO", "name": "personal_pronoun_incidence", "description": "Personal pronoun incidence (per 1000 words)"},
                ]
            },

            "D5.2_POS_Density_Per_Sentence": {
                "description": "Average POS counts per sentence.",
                "features": [
                    {"toolkit": "LFTK", "code": "a_adj_ps", "name": "avg_adjectives_per_sentence", "description": "Avg adjectives per sentence"},
                    {"toolkit": "LFTK", "code": "a_adp_ps", "name": "avg_adpositions_per_sentence", "description": "Avg adpositions per sentence"},
                    {"toolkit": "LFTK", "code": "a_adv_ps", "name": "avg_adverbs_per_sentence", "description": "Avg adverbs per sentence"},
                    {"toolkit": "LFTK", "code": "a_aux_ps", "name": "avg_auxiliaries_per_sentence", "description": "Avg auxiliaries per sentence"},
                    {"toolkit": "LFTK", "code": "a_cconj_ps", "name": "avg_coordinating_conjunctions_per_sentence", "description": "Avg coord. conjunctions per sentence"},
                    {"toolkit": "LFTK", "code": "a_det_ps", "name": "avg_determiners_per_sentence", "description": "Avg determiners per sentence"},
                    {"toolkit": "LFTK", "code": "a_intj_ps", "name": "avg_interjections_per_sentence", "description": "Avg interjections per sentence"},
                    {"toolkit": "LFTK", "code": "a_noun_ps", "name": "avg_nouns_per_sentence", "description": "Avg nouns per sentence"},
                    {"toolkit": "LFTK", "code": "a_num_ps", "name": "avg_numerals_per_sentence", "description": "Avg numerals per sentence"},
                    {"toolkit": "LFTK", "code": "a_part_ps", "name": "avg_particles_per_sentence", "description": "Avg particles per sentence"},
                    {"toolkit": "LFTK", "code": "a_pron_ps", "name": "avg_pronouns_per_sentence", "description": "Avg pronouns per sentence"},
                    {"toolkit": "LFTK", "code": "a_propn_ps", "name": "avg_proper_nouns_per_sentence", "description": "Avg proper nouns per sentence"},
                    {"toolkit": "LFTK", "code": "a_sconj_ps", "name": "avg_subordinating_conjunctions_per_sentence", "description": "Avg subord. conjunctions per sentence"},
                    {"toolkit": "LFTK", "code": "a_sym_ps", "name": "avg_symbols_per_sentence", "description": "Avg symbols per sentence"},
                    {"toolkit": "LFTK", "code": "a_verb_ps", "name": "avg_verbs_per_sentence", "description": "Avg verbs per sentence"},
                    {"toolkit": "LFTK", "code": "a_punct_ps", "name": "avg_punctuation_per_sentence", "description": "Avg punctuation POS per sentence"},
                    {"toolkit": "LFTK", "code": "a_space_ps", "name": "avg_whitespace_per_sentence", "description": "Avg whitespace POS per sentence"},
                ]
            },

            "D5.3_POS_Variation_Lexical_Diversity_per_POS": {
                "description": "Variation ratios (simple, root, corrected) for each POS category — captures lexical richness within each grammatical class.",
                "features": [
                    # Simple variation (unique/total)
                    {"toolkit": "LFTK", "code": "simp_noun_var", "name": "simple_noun_variation", "description": "Unique nouns / total nouns"},
                    {"toolkit": "LFTK", "code": "simp_verb_var", "name": "simple_verb_variation", "description": "Unique verbs / total verbs"},
                    {"toolkit": "LFTK", "code": "simp_adj_var", "name": "simple_adjective_variation", "description": "Unique adjectives / total adjectives"},
                    {"toolkit": "LFTK", "code": "simp_adv_var", "name": "simple_adverb_variation", "description": "Unique adverbs / total adverbs"},
                    {"toolkit": "LFTK", "code": "simp_adp_var", "name": "simple_adposition_variation", "description": "Unique adpositions / total adpositions"},
                    {"toolkit": "LFTK", "code": "simp_aux_var", "name": "simple_auxiliary_variation", "description": "Unique auxiliaries / total auxiliaries"},
                    {"toolkit": "LFTK", "code": "simp_cconj_var", "name": "simple_cconj_variation", "description": "Unique coord conjunctions / total"},
                    {"toolkit": "LFTK", "code": "simp_det_var", "name": "simple_determiner_variation", "description": "Unique determiners / total"},
                    {"toolkit": "LFTK", "code": "simp_intj_var", "name": "simple_interjection_variation", "description": "Unique interjections / total"},
                    {"toolkit": "LFTK", "code": "simp_num_var", "name": "simple_numeral_variation", "description": "Unique numerals / total"},
                    {"toolkit": "LFTK", "code": "simp_part_var", "name": "simple_particle_variation", "description": "Unique particles / total"},
                    {"toolkit": "LFTK", "code": "simp_pron_var", "name": "simple_pronoun_variation", "description": "Unique pronouns / total"},
                    {"toolkit": "LFTK", "code": "simp_propn_var", "name": "simple_proper_noun_variation", "description": "Unique proper nouns / total"},
                    {"toolkit": "LFTK", "code": "simp_sconj_var", "name": "simple_sconj_variation", "description": "Unique subord conjunctions / total"},
                    {"toolkit": "LFTK", "code": "simp_sym_var", "name": "simple_symbol_variation", "description": "Unique symbols / total"},
                    {"toolkit": "LFTK", "code": "simp_punct_var", "name": "simple_punctuation_variation", "description": "Unique punctuation / total"},
                    {"toolkit": "LFTK", "code": "simp_space_var", "name": "simple_whitespace_variation", "description": "Unique whitespace / total"},
                    # Root variation (unique/sqrt(total))
                    {"toolkit": "LFTK", "code": "root_noun_var", "name": "root_noun_variation", "description": "Unique nouns / sqrt(total nouns)"},
                    {"toolkit": "LFTK", "code": "root_verb_var", "name": "root_verb_variation", "description": "Unique verbs / sqrt(total verbs)"},
                    {"toolkit": "LFTK", "code": "root_adj_var", "name": "root_adjective_variation", "description": "Unique adj / sqrt(total adj)"},
                    {"toolkit": "LFTK", "code": "root_adv_var", "name": "root_adverb_variation", "description": "Unique adv / sqrt(total adv)"},
                    {"toolkit": "LFTK", "code": "root_adp_var", "name": "root_adposition_variation", "description": "Unique adp / sqrt(total adp)"},
                    {"toolkit": "LFTK", "code": "root_aux_var", "name": "root_auxiliary_variation", "description": "Unique aux / sqrt(total aux)"},
                    {"toolkit": "LFTK", "code": "root_cconj_var", "name": "root_cconj_variation", "description": "Unique cconj / sqrt(total)"},
                    {"toolkit": "LFTK", "code": "root_det_var", "name": "root_determiner_variation", "description": "Unique det / sqrt(total)"},
                    {"toolkit": "LFTK", "code": "root_intj_var", "name": "root_interjection_variation", "description": "Unique intj / sqrt(total)"},
                    {"toolkit": "LFTK", "code": "root_num_var", "name": "root_numeral_variation", "description": "Unique num / sqrt(total)"},
                    {"toolkit": "LFTK", "code": "root_part_var", "name": "root_particle_variation", "description": "Unique part / sqrt(total)"},
                    {"toolkit": "LFTK", "code": "root_pron_var", "name": "root_pronoun_variation", "description": "Unique pron / sqrt(total)"},
                    {"toolkit": "LFTK", "code": "root_propn_var", "name": "root_proper_noun_variation", "description": "Unique propn / sqrt(total)"},
                    {"toolkit": "LFTK", "code": "root_sconj_var", "name": "root_sconj_variation", "description": "Unique sconj / sqrt(total)"},
                    {"toolkit": "LFTK", "code": "root_sym_var", "name": "root_symbol_variation", "description": "Unique sym / sqrt(total)"},
                    {"toolkit": "LFTK", "code": "root_punct_var", "name": "root_punctuation_variation", "description": "Unique punct / sqrt(total)"},
                    {"toolkit": "LFTK", "code": "root_space_var", "name": "root_whitespace_variation", "description": "Unique space / sqrt(total)"},
                    # Corrected variation (unique/sqrt(2*total))
                    {"toolkit": "LFTK", "code": "corr_noun_var", "name": "corrected_noun_variation", "description": "Unique nouns / sqrt(2*total nouns)"},
                    {"toolkit": "LFTK", "code": "corr_verb_var", "name": "corrected_verb_variation", "description": "Unique verbs / sqrt(2*total verbs)"},
                    {"toolkit": "LFTK", "code": "corr_adj_var", "name": "corrected_adjective_variation", "description": "Unique adj / sqrt(2*total adj)"},
                    {"toolkit": "LFTK", "code": "corr_adv_var", "name": "corrected_adverb_variation", "description": "Unique adv / sqrt(2*total adv)"},
                    {"toolkit": "LFTK", "code": "corr_adp_var", "name": "corrected_adposition_variation", "description": "Unique adp / sqrt(2*total adp)"},
                    {"toolkit": "LFTK", "code": "corr_aux_var", "name": "corrected_auxiliary_variation", "description": "Unique aux / sqrt(2*total aux)"},
                    {"toolkit": "LFTK", "code": "corr_cconj_var", "name": "corrected_cconj_variation", "description": "Unique cconj / sqrt(2*total)"},
                    {"toolkit": "LFTK", "code": "corr_det_var", "name": "corrected_determiner_variation", "description": "Unique det / sqrt(2*total)"},
                    {"toolkit": "LFTK", "code": "corr_intj_var", "name": "corrected_interjection_variation", "description": "Unique intj / sqrt(2*total)"},
                    {"toolkit": "LFTK", "code": "corr_num_var", "name": "corrected_numeral_variation", "description": "Unique num / sqrt(2*total)"},
                    {"toolkit": "LFTK", "code": "corr_part_var", "name": "corrected_particle_variation", "description": "Unique part / sqrt(2*total)"},
                    {"toolkit": "LFTK", "code": "corr_pron_var", "name": "corrected_pronoun_variation", "description": "Unique pron / sqrt(2*total)"},
                    {"toolkit": "LFTK", "code": "corr_propn_var", "name": "corrected_proper_noun_variation", "description": "Unique propn / sqrt(2*total)"},
                    {"toolkit": "LFTK", "code": "corr_sconj_var", "name": "corrected_sconj_variation", "description": "Unique sconj / sqrt(2*total)"},
                    {"toolkit": "LFTK", "code": "corr_sym_var", "name": "corrected_symbol_variation", "description": "Unique sym / sqrt(2*total)"},
                    {"toolkit": "LFTK", "code": "corr_punct_var", "name": "corrected_punctuation_variation", "description": "Unique punct / sqrt(2*total)"},
                    {"toolkit": "LFTK", "code": "corr_space_var", "name": "corrected_whitespace_variation", "description": "Unique space / sqrt(2*total)"},
                ]
            },

           

            "D5.4_Pronoun_Types": {
                "description": "Density of pronouns by person and number — important for narrative POV and referential complexity.",
                "features": [
                    {"toolkit": "cohmetrix", "code": "WRDPRP1s", "name": "pronoun_1st_person_singular", "description": "First person singular pronoun incidence (I, me, my)"},
                    {"toolkit": "cohmetrix", "code": "WRDPRP1p", "name": "pronoun_1st_person_plural", "description": "First person plural pronoun incidence (we, us, our)"},
                    {"toolkit": "cohmetrix", "code": "WRDPRP2", "name": "pronoun_2nd_person", "description": "Second person pronoun incidence (you, your)"},
                    {"toolkit": "cohmetrix", "code": "WRDPRP3s", "name": "pronoun_3rd_person_singular", "description": "Third person singular pronoun incidence (he, she, it)"},
                    {"toolkit": "cohmetrix", "code": "WRDPRP3p", "name": "pronoun_3rd_person_plural", "description": "Third person plural pronoun incidence (they, them)"},
                ]
            },
        }
    },


    # =========================================================================
    # MACRO-TIER C: INTER-SENTENCE & DISCOURSE LEVEL
    #
    # D6 = COHESION (explicit textual overlap/linking mechanisms)
    # D7 = COHERENCE (semantic/conceptual continuity & integration)
    # D8 = DISCOURSE STRUCTURE (global text organisation & topic flow)
    # =========================================================================

    "D6_Textual_Cohesion": {
        "description": "EXPLICIT textual linking mechanisms: lexical overlap between sentences, connective usage, anaphoric devices. These are surface-level cues that help the reader link adjacent and distant text segments. Focus: LOCAL linking devices.",
        "subdimensions": {

            "D6.1_Referential_Overlap_Binary": {
                "description": "Binary overlap measures: do adjacent/all sentence pairs share nouns, arguments, stems, content words?",
                "features": [
                    {"toolkit": "cohmetrix", "code": "CRFNO1", "name": "noun_overlap_adjacent_local", "description": "Adjacent sentence noun overlap (exact match, binary)"},
                    {"toolkit": "cohmetrix", "code": "CRFNOa", "name": "noun_overlap_all_global", "description": "Global noun overlap across all sentence pairs"},
                    {"toolkit": "cohmetrix", "code": "CRFAO1", "name": "argument_overlap_adjacent", "description": "Adjacent sentence argument overlap (nouns+pronouns)"},
                    {"toolkit": "cohmetrix", "code": "CRFAOa", "name": "argument_overlap_all", "description": "Global argument overlap"},
                    {"toolkit": "cohmetrix", "code": "CRFSO1", "name": "stem_overlap_adjacent", "description": "Adjacent sentence stem overlap (lemma-based)"},
                    {"toolkit": "cohmetrix", "code": "CRFSOa", "name": "stem_overlap_all", "description": "Global stem overlap"},
                    {"toolkit": "cohmetrix", "code": "CRFANP1", "name": "anaphor_overlap_adjacent", "description": "Adjacent sentence anaphoric overlap (pronouns referring back)"},
                    {"toolkit": "cohmetrix", "code": "CRFANPa", "name": "anaphor_overlap_all", "description": "Global anaphoric overlap"},
                ]
            },

            "D6.2_Referential_Overlap_Proportional": {
                "description": "Proportional content word overlap — sensitive to sentence length.",
                "features": [
                    {"toolkit": "cohmetrix", "code": "CRFCWO1", "name": "content_word_overlap_adjacent", "description": "Proportional content word overlap, adjacent sentences"},
                    {"toolkit": "cohmetrix", "code": "CRFCWO1d", "name": "content_word_overlap_adjacent_sd", "description": "SD of content word overlap, adjacent"},
                    {"toolkit": "cohmetrix", "code": "CRFCWOa", "name": "content_word_overlap_all", "description": "Proportional content word overlap, all sentences"},
                    {"toolkit": "cohmetrix", "code": "CRFCWOad", "name": "content_word_overlap_all_sd", "description": "SD of content word overlap, all"},
                ]
            },

            "D6.3_Connective_Devices": {
                "description": "Explicit connectives that signal logical, causal, temporal, additive, adversative relationships.",
                "features": [
                    {"toolkit": "cohmetrix", "code": "CNCAll", "name": "all_connectives_incidence", "description": "Incidence of all connectives (per 1000 words)"},
                    {"toolkit": "cohmetrix", "code": "CNCCaus", "name": "causal_connectives_incidence", "description": "Incidence of causal connectives (because, so)"},
                    {"toolkit": "cohmetrix", "code": "CNCLogic", "name": "logical_connectives_incidence", "description": "Incidence of logical connectives (and, or)"},
                    {"toolkit": "cohmetrix", "code": "CNCADC", "name": "adversative_connectives_incidence", "description": "Incidence of adversative/contrastive connectives (although, whereas)"},
                    {"toolkit": "cohmetrix", "code": "CNCTemp", "name": "temporal_connectives_incidence", "description": "Incidence of temporal connectives (first, until)"},
                    {"toolkit": "cohmetrix", "code": "CNCTempx", "name": "temporal_connectives_extended", "description": "Incidence of extended temporal connectives"},
                    {"toolkit": "cohmetrix", "code": "CNCAdd", "name": "additive_connectives_incidence", "description": "Incidence of additive connectives (and, moreover)"},
                    {"toolkit": "cohmetrix", "code": "CNCPos", "name": "positive_connectives_incidence", "description": "Incidence of positive connectives"},
                    {"toolkit": "cohmetrix", "code": "CNCNeg", "name": "negative_connectives_incidence", "description": "Incidence of negative connectives (however, but)"},
                ]
            },

            "D6.4_Entity_Density": {
                "description": "Named entity density — how many referents are introduced and maintained.",
                "features": [
                    {"toolkit": "LFTK", "code": "t_n_ent", "name": "total_named_entities", "description": "Total named entity count"},
                    {"toolkit": "LFTK", "code": "a_n_ent_pw", "name": "avg_entities_per_word", "description": "Average named entities per word"},
                    {"toolkit": "LFTK", "code": "a_n_ent_ps", "name": "avg_entities_per_sentence", "description": "Average named entities per sentence"},
                    {"toolkit": "LFTK", "code": "t_n_ent_person", "name": "total_person_entities", "description": "Total PERSON entities"},
                    {"toolkit": "LFTK", "code": "a_n_ent_person_pw", "name": "avg_person_entities_per_word", "description": "PERSON entities per word"},
                    {"toolkit": "LFTK", "code": "a_n_ent_person_ps", "name": "avg_person_entities_per_sentence", "description": "PERSON entities per sentence"},
                    {"toolkit": "LFTK", "code": "t_n_ent_gpe", "name": "total_gpe_entities", "description": "Total GPE (geopolitical) entities"},
                    {"toolkit": "LFTK", "code": "a_n_ent_gpe_pw", "name": "avg_gpe_entities_per_word", "description": "GPE entities per word"},
                    {"toolkit": "LFTK", "code": "a_n_ent_gpe_ps", "name": "avg_gpe_entities_per_sentence", "description": "GPE entities per sentence"},
                    {"toolkit": "LFTK", "code": "t_n_ent_org", "name": "total_org_entities", "description": "Total ORG entities"},
                    {"toolkit": "LFTK", "code": "a_n_ent_org_pw", "name": "avg_org_entities_per_word", "description": "ORG entities per word"},
                    {"toolkit": "LFTK", "code": "a_n_ent_org_ps", "name": "avg_org_entities_per_sentence", "description": "ORG entities per sentence"},
                    {"toolkit": "LFTK", "code": "t_n_ent_loc", "name": "total_loc_entities", "description": "Total LOC entities"},
                    {"toolkit": "LFTK", "code": "a_n_ent_loc_pw", "name": "avg_loc_entities_per_word", "description": "LOC entities per word"},
                    {"toolkit": "LFTK", "code": "a_n_ent_loc_ps", "name": "avg_loc_entities_per_sentence", "description": "LOC entities per sentence"},
                    {"toolkit": "LFTK", "code": "t_n_ent_date", "name": "total_date_entities", "description": "Total DATE entities"},
                    {"toolkit": "LFTK", "code": "a_n_ent_date_pw", "name": "avg_date_entities_per_word", "description": "DATE entities per word"},
                    {"toolkit": "LFTK", "code": "a_n_ent_date_ps", "name": "avg_date_entities_per_sentence", "description": "DATE entities per sentence"},
                    {"toolkit": "LFTK", "code": "t_n_ent_time", "name": "total_time_entities", "description": "Total TIME entities"},
                    {"toolkit": "LFTK", "code": "a_n_ent_time_pw", "name": "avg_time_entities_per_word", "description": "TIME entities per word"},
                    {"toolkit": "LFTK", "code": "a_n_ent_time_ps", "name": "avg_time_entities_per_sentence", "description": "TIME entities per sentence"},
                    {"toolkit": "LFTK", "code": "t_n_ent_event", "name": "total_event_entities", "description": "Total EVENT entities"},
                    {"toolkit": "LFTK", "code": "a_n_ent_event_pw", "name": "avg_event_entities_per_word", "description": "EVENT entities per word"},
                    {"toolkit": "LFTK", "code": "a_n_ent_event_ps", "name": "avg_event_entities_per_sentence", "description": "EVENT entities per sentence"},
                    {"toolkit": "LFTK", "code": "t_n_ent_norp", "name": "total_norp_entities", "description": "Total NORP (nationality/religious/political) entities"},
                    {"toolkit": "LFTK", "code": "a_n_ent_norp_pw", "name": "avg_norp_entities_per_word", "description": "NORP entities per word"},
                    {"toolkit": "LFTK", "code": "a_n_ent_norp_ps", "name": "avg_norp_entities_per_sentence", "description": "NORP entities per sentence"},
                    {"toolkit": "LFTK", "code": "t_n_ent_fac", "name": "total_fac_entities", "description": "Total FAC (facility) entities"},
                    {"toolkit": "LFTK", "code": "a_n_ent_fac_pw", "name": "avg_fac_entities_per_word", "description": "FAC entities per word"},
                    {"toolkit": "LFTK", "code": "a_n_ent_fac_ps", "name": "avg_fac_entities_per_sentence", "description": "FAC entities per sentence"},
                    {"toolkit": "LFTK", "code": "t_n_ent_product", "name": "total_product_entities", "description": "Total PRODUCT entities"},
                    {"toolkit": "LFTK", "code": "a_n_ent_product_pw", "name": "avg_product_entities_per_word", "description": "PRODUCT entities per word"},
                    {"toolkit": "LFTK", "code": "a_n_ent_product_ps", "name": "avg_product_entities_per_sentence", "description": "PRODUCT entities per sentence"},
                    {"toolkit": "LFTK", "code": "t_n_ent_art", "name": "total_art_entities", "description": "Total WORK_OF_ART entities"},
                    {"toolkit": "LFTK", "code": "a_n_ent_art_pw", "name": "avg_art_entities_per_word", "description": "WORK_OF_ART entities per word"},
                    {"toolkit": "LFTK", "code": "a_n_ent_art_ps", "name": "avg_art_entities_per_sentence", "description": "WORK_OF_ART entities per sentence"},
                    {"toolkit": "LFTK", "code": "t_n_ent_law", "name": "total_law_entities", "description": "Total LAW entities"},
                    {"toolkit": "LFTK", "code": "a_n_ent_law_pw", "name": "avg_law_entities_per_word", "description": "LAW entities per word"},
                    {"toolkit": "LFTK", "code": "a_n_ent_law_ps", "name": "avg_law_entities_per_sentence", "description": "LAW entities per sentence"},
                    {"toolkit": "LFTK", "code": "t_n_ent_language", "name": "total_language_entities", "description": "Total LANGUAGE entities"},
                    {"toolkit": "LFTK", "code": "a_n_ent_language_pw", "name": "avg_language_entities_per_word", "description": "LANGUAGE entities per word"},
                    {"toolkit": "LFTK", "code": "a_n_ent_language_ps", "name": "avg_language_entities_per_sentence", "description": "LANGUAGE entities per sentence"},
                    {"toolkit": "LFTK", "code": "t_n_ent_money", "name": "total_money_entities", "description": "Total MONEY entities"},
                    {"toolkit": "LFTK", "code": "a_n_ent_money_pw", "name": "avg_money_entities_per_word", "description": "MONEY entities per word"},
                    {"toolkit": "LFTK", "code": "a_n_ent_money_ps", "name": "avg_money_entities_per_sentence", "description": "MONEY entities per sentence"},
                    {"toolkit": "LFTK", "code": "t_n_ent_quantity", "name": "total_quantity_entities", "description": "Total QUANTITY entities"},
                    {"toolkit": "LFTK", "code": "a_n_ent_quantity_pw", "name": "avg_quantity_entities_per_word", "description": "QUANTITY entities per word"},
                    {"toolkit": "LFTK", "code": "a_n_ent_quantity_ps", "name": "avg_quantity_entities_per_sentence", "description": "QUANTITY entities per sentence"},
                    {"toolkit": "LFTK", "code": "t_n_ent_percent", "name": "total_percent_entities", "description": "Total PERCENT entities"},
                    {"toolkit": "LFTK", "code": "a_n_ent_percent_pw", "name": "avg_percent_entities_per_word", "description": "PERCENT entities per word"},
                    {"toolkit": "LFTK", "code": "a_n_ent_percent_ps", "name": "avg_percent_entities_per_sentence", "description": "PERCENT entities per sentence"},
                    {"toolkit": "LFTK", "code": "t_n_ent_ordinal", "name": "total_ordinal_entities", "description": "Total ORDINAL entities"},
                    {"toolkit": "LFTK", "code": "a_n_ent_ordinal_pw", "name": "avg_ordinal_entities_per_word", "description": "ORDINAL entities per word"},
                    {"toolkit": "LFTK", "code": "a_n_ent_ordinal_ps", "name": "avg_ordinal_entities_per_sentence", "description": "ORDINAL entities per sentence"},
                    {"toolkit": "LFTK", "code": "t_n_ent_cardinal", "name": "total_cardinal_entities", "description": "Total CARDINAL entities"},
                    {"toolkit": "LFTK", "code": "a_n_ent_cardinal_pw", "name": "avg_cardinal_entities_per_word", "description": "CARDINAL entities per word"},
                    {"toolkit": "LFTK", "code": "a_n_ent_cardinal_ps", "name": "avg_cardinal_entities_per_sentence", "description": "CARDINAL entities per sentence"},
                    # lingfeat entity density (total/per-sentence/per-token)
                    {"toolkit": "lingfeat", "code": "EnDF_:to_EntiM_C", "name": "total_entity_mentions", "description": "Total entity mentions (lingfeat)"},
                    {"toolkit": "lingfeat", "code": "EnDF_:as_EntiM_C", "name": "avg_entity_mentions_per_sentence", "description": "Avg entity mentions per sentence"},
                    {"toolkit": "lingfeat", "code": "EnDF_:at_EntiM_C", "name": "avg_entity_mentions_per_token", "description": "Avg entity mentions per token"},
                    {"toolkit": "lingfeat", "code": "EnDF_:to_UEnti_C", "name": "total_unique_entities", "description": "Total unique entities"},
                    {"toolkit": "lingfeat", "code": "EnDF_:as_UEnti_C", "name": "avg_unique_entities_per_sentence", "description": "Avg unique entities per sentence"},
                    {"toolkit": "lingfeat", "code": "EnDF_:at_UEnti_C", "name": "avg_unique_entities_per_token", "description": "Avg unique entities per token"},
                ]
            },
        }
    },


    "D7_Semantic_Coherence": {
        "description": "IMPLICIT semantic continuity: how well ideas connect conceptually across the text. Measured via latent semantic analysis (LSA), sentence embeddings, situation model dimensions (causal, intentional, temporal coherence). Focus: MEANING-LEVEL continuity.",
        "subdimensions": {

            "D7.1_LSA_Semantic_Overlap": {
                "description": "Latent Semantic Analysis cosine between sentence pairs — captures deep semantic similarity beyond surface word overlap.",
                "features": [
                    {"toolkit": "cohmetrix", "code": "LSASS1", "name": "lsa_adjacent_sentences_mean", "description": "Mean LSA cosine between adjacent sentences"},
                    {"toolkit": "cohmetrix", "code": "LSASS1d", "name": "lsa_adjacent_sentences_sd", "description": "SD of LSA cosine between adjacent sentences"},
                    {"toolkit": "cohmetrix", "code": "LSASSp", "name": "lsa_all_sentences_mean", "description": "Mean LSA cosine between all sentence pairs"},
                    {"toolkit": "cohmetrix", "code": "LSASSpd", "name": "lsa_all_sentences_sd", "description": "SD of LSA cosine, all sentence pairs within paragraphs"},
                    {"toolkit": "cohmetrix", "code": "LSAPP1", "name": "lsa_adjacent_paragraphs_mean", "description": "Mean LSA cosine between adjacent paragraphs"},
                    {"toolkit": "cohmetrix", "code": "LSAPP1d", "name": "lsa_adjacent_paragraphs_sd", "description": "SD of LSA cosine between adjacent paragraphs"},
                    {"toolkit": "cohmetrix", "code": "LSAGN", "name": "lsa_givenness_mean", "description": "Average givenness of each sentence (LSA-based)"},
                    {"toolkit": "cohmetrix", "code": "LSAGNd", "name": "lsa_givenness_sd", "description": "SD of givenness (LSA-based)"},
                    {"toolkit": "taales", "code": "Sem_D", "name": "semantic_diversity", "description": "Semantic diversity measured via LSA/embedding distances"},
                ]
            },

            "D7.2_Situation_Model_Causal_Coherence": {
                "description": "Causal coherence: causal verbs, causal verb overlap, causal particles — how well causal chains are maintained.",
                "features": [
                    {"toolkit": "cohmetrix", "code": "SMCAUSv", "name": "causal_verbs_incidence", "description": "Incidence of causal verbs"},
                    {"toolkit": "cohmetrix", "code": "SMCAUSvp", "name": "causal_verb_particles_incidence", "description": "Incidence of causal verbs + particles"},
                    {"toolkit": "cohmetrix", "code": "SMCAUSr", "name": "causal_cohesion_ratio", "description": "Ratio of causal particles to causal verbs (causal cohesion)"},
                    {"toolkit": "cohmetrix", "code": "SMCAUSlsa", "name": "causal_lsa_verb_overlap", "description": "LSA overlap between causal verbs"},
                    {"toolkit": "cohmetrix", "code": "SMCAUSwn", "name": "causal_wordnet_verb_overlap", "description": "WordNet overlap between causal verbs"},
                ]
            },

            "D7.3_Situation_Model_Intentional_Coherence": {
                "description": "Intentional coherence: intentional verbs/actions, intentional particles — goal-directed event coherence.",
                "features": [
                    {"toolkit": "cohmetrix", "code": "SMINTEp", "name": "intentional_content_incidence", "description": "Incidence of intentional actions/events/particles"},
                    {"toolkit": "cohmetrix", "code": "SMINTEr", "name": "intentional_cohesion_ratio", "description": "Ratio of intentional particles to intentional actions"},
                ]
            },

            "D7.4_Situation_Model_Temporal_Coherence": {
                "description": "Temporal coherence: tense/aspect consistency, temporal verb overlap.",
                "features": [
                    {"toolkit": "cohmetrix", "code": "SMTEMP", "name": "temporal_cohesion_tense_aspect", "description": "Repetition score for tense and aspect consistency"},
                ]
            },

           

            "D7.5_Lexical_Diversity_TTR": {
                "description": "Type-token ratio and advanced lexical diversity measures — captures how much new vocabulary the reader must process. Lower TTR = more repetition = more cohesive.",
                "features": [
                    {"toolkit": "LFTK", "code": "simp_ttr", "name": "simple_ttr", "description": "Types / tokens"},
                    {"toolkit": "LFTK", "code": "root_ttr", "name": "root_ttr", "description": "Types / sqrt(tokens)"},
                    {"toolkit": "LFTK", "code": "corr_ttr", "name": "corrected_ttr", "description": "Types / sqrt(2*tokens)"},
                    {"toolkit": "LFTK", "code": "bilog_ttr", "name": "bilogarithmic_ttr", "description": "log(types) / log(tokens)"},
                    {"toolkit": "LFTK", "code": "uber_ttr", "name": "uber_ttr", "description": "log²(types) / log(tokens/types)"},
                    {"toolkit": "LFTK", "code": "simp_ttr_no_lem", "name": "simple_ttr_no_lemma", "description": "TTR without lemmatization"},
                    {"toolkit": "LFTK", "code": "root_ttr_no_lem", "name": "root_ttr_no_lemma", "description": "Root TTR without lemmatization"},
                    {"toolkit": "LFTK", "code": "corr_ttr_no_lem", "name": "corrected_ttr_no_lemma", "description": "Corrected TTR without lemmatization"},
                    {"toolkit": "LFTK", "code": "bilog_ttr_no_lem", "name": "bilog_ttr_no_lemma", "description": "Bilog TTR without lemmatization"},
                    {"toolkit": "LFTK", "code": "uber_ttr_no_lem", "name": "uber_ttr_no_lemma", "description": "Uber TTR without lemmatization"},
                ]
            },
        }
    },


    "D8_Discourse_Structure_and_Organisation": {
        "description": "GLOBAL text organisation: entity grid (how entities transition between grammatical roles across sentences), topic structure, paragraph-level coherence, rhetorical structure. Focus: HOW the text is ORGANISED at a macro level.",
        "subdimensions": {

            "D8.1_Entity_Grid_Transitions": {
                "description": "Entity grid (Barzilay & Lapata, 2008): transitions of entities between Subject(S), Object(O), Other(X), Absent(N) roles across sentence pairs. Captures how entities are threaded through the discourse.",
                "features": [
                    {"toolkit": "lingfeat", "code": "EnGF_:ra_SSTo_C", "name": "entity_grid_SS_ratio", "description": "Ratio of S→S transitions (entity stays as subject)"},
                    {"toolkit": "lingfeat", "code": "EnGF_:ra_SOTo_C", "name": "entity_grid_SO_ratio", "description": "Ratio of S→O transitions"},
                    {"toolkit": "lingfeat", "code": "EnGF_:ra_SXTo_C", "name": "entity_grid_SX_ratio", "description": "Ratio of S→X transitions"},
                    {"toolkit": "lingfeat", "code": "EnGF_:ra_SNTo_C", "name": "entity_grid_SN_ratio", "description": "Ratio of S→N transitions (entity drops out)"},
                    {"toolkit": "lingfeat", "code": "EnGF_:ra_OSTo_C", "name": "entity_grid_OS_ratio", "description": "Ratio of O→S transitions"},
                    {"toolkit": "lingfeat", "code": "EnGF_:ra_OOTo_C", "name": "entity_grid_OO_ratio", "description": "Ratio of O→O transitions"},
                    {"toolkit": "lingfeat", "code": "EnGF_:ra_OXTo_C", "name": "entity_grid_OX_ratio", "description": "Ratio of O→X transitions"},
                    {"toolkit": "lingfeat", "code": "EnGF_:ra_ONTo_C", "name": "entity_grid_ON_ratio", "description": "Ratio of O→N transitions"},
                    {"toolkit": "lingfeat", "code": "EnGF_:ra_XSTo_C", "name": "entity_grid_XS_ratio", "description": "Ratio of X→S transitions"},
                    {"toolkit": "lingfeat", "code": "EnGF_:ra_XOTo_C", "name": "entity_grid_XO_ratio", "description": "Ratio of X→O transitions"},
                    {"toolkit": "lingfeat", "code": "EnGF_:ra_XXTo_C", "name": "entity_grid_XX_ratio", "description": "Ratio of X→X transitions"},
                    {"toolkit": "lingfeat", "code": "EnGF_:ra_XNTo_C", "name": "entity_grid_XN_ratio", "description": "Ratio of X→N transitions"},
                    {"toolkit": "lingfeat", "code": "EnGF_:ra_NSTo_C", "name": "entity_grid_NS_ratio", "description": "Ratio of N→S transitions (entity reintroduced as subject)"},
                    {"toolkit": "lingfeat", "code": "EnGF_:ra_NOTo_C", "name": "entity_grid_NO_ratio", "description": "Ratio of N→O transitions"},
                    {"toolkit": "lingfeat", "code": "EnGF_:ra_NXTo_C", "name": "entity_grid_NX_ratio", "description": "Ratio of N→X transitions"},
                    {"toolkit": "lingfeat", "code": "EnGF_:ra_NNTo_C", "name": "entity_grid_NN_ratio", "description": "Ratio of N→N transitions"},
                ]
            },

            "D8.2_Local_Coherence_Scores": {
                "description": "Aggregate local coherence derived from the entity grid — overall coherence quality score.",
                "features": [
                    {"toolkit": "lingfeat", "code": "EnGF_:LoCohPA_S", "name": "local_coherence_PA", "description": "Local coherence score (PA: probability-all)"},
                    {"toolkit": "lingfeat", "code": "EnGF_:LoCohPW_S", "name": "local_coherence_PW", "description": "Local coherence score (PW: probability-weighted)"},
                    {"toolkit": "lingfeat", "code": "EnGF_:LoCohPU_S", "name": "local_coherence_PU", "description": "Local coherence score (PU: probability-unweighted)"},
                    {"toolkit": "lingfeat", "code": "EnGF_:LoCoDPA_S", "name": "local_coherence_distance_PA", "description": "Local coherence distance (PA)"},
                    {"toolkit": "lingfeat", "code": "EnGF_:LoCoDPW_S", "name": "local_coherence_distance_PW", "description": "Local coherence distance (PW)"},
                    {"toolkit": "lingfeat", "code": "EnGF_:LoCoDPU_S", "name": "local_coherence_distance_PU", "description": "Local coherence distance (PU)"},
                ]
            },

            "D8.3_Topic_Modelling_Semantic_Structure": {
                "description": "LDA topic modelling: semantic richness, clarity, noise — how focused or diffuse the text's topics are.",
                "features": [
                    # lingfeat: trained LDA models (Wikipedia / WeeBit / OneStopEng) at 50/100/150/200 topics
                    {"toolkit": "lingfeat", "code": "WoKF_:WRich05_S", "name": "wiki_semantic_richness_50", "description": "Semantic richness (50 topics, Wikipedia LDA)"},
                    {"toolkit": "lingfeat", "code": "WoKF_:WClar05_S", "name": "wiki_semantic_clarity_50", "description": "Semantic clarity (50 topics, Wikipedia LDA)"},
                    {"toolkit": "lingfeat", "code": "WoKF_:WNois05_S", "name": "wiki_semantic_noise_50", "description": "Semantic noise (50 topics, Wikipedia LDA)"},
                    {"toolkit": "lingfeat", "code": "WoKF_:WTopc05_S", "name": "wiki_topic_count_50", "description": "Number of topics (50 topics, Wikipedia LDA)"},
                    {"toolkit": "lingfeat", "code": "WoKF_:WRich10_S", "name": "wiki_semantic_richness_100", "description": "Semantic richness (100 topics)"},
                    {"toolkit": "lingfeat", "code": "WoKF_:WClar10_S", "name": "wiki_semantic_clarity_100", "description": "Semantic clarity (100 topics)"},
                    {"toolkit": "lingfeat", "code": "WoKF_:WNois10_S", "name": "wiki_semantic_noise_100", "description": "Semantic noise (100 topics)"},
                    {"toolkit": "lingfeat", "code": "WoKF_:WTopc10_S", "name": "wiki_topic_count_100", "description": "Number of topics (100 topics)"},
                    {"toolkit": "lingfeat", "code": "WoKF_:WRich15_S", "name": "wiki_semantic_richness_150", "description": "Semantic richness (150 topics)"},
                    {"toolkit": "lingfeat", "code": "WoKF_:WClar15_S", "name": "wiki_semantic_clarity_150", "description": "Semantic clarity (150 topics)"},
                    {"toolkit": "lingfeat", "code": "WoKF_:WNois15_S", "name": "wiki_semantic_noise_150", "description": "Semantic noise (150 topics)"},
                    {"toolkit": "lingfeat", "code": "WoKF_:WTopc15_S", "name": "wiki_topic_count_150", "description": "Number of topics (150 topics)"},
                    {"toolkit": "lingfeat", "code": "WoKF_:WRich20_S", "name": "wiki_semantic_richness_200", "description": "Semantic richness (200 topics)"},
                    {"toolkit": "lingfeat", "code": "WoKF_:WClar20_S", "name": "wiki_semantic_clarity_200", "description": "Semantic clarity (200 topics)"},
                    {"toolkit": "lingfeat", "code": "WoKF_:WNois20_S", "name": "wiki_semantic_noise_200", "description": "Semantic noise (200 topics)"},
                    {"toolkit": "lingfeat", "code": "WoKF_:WTopc20_S", "name": "wiki_topic_count_200", "description": "Number of topics (200 topics)"},
                    {"toolkit": "lingfeat", "code": "WBKF_:BRich05_S", "name": "weebit_semantic_richness_50", "description": "Semantic richness (50 topics, WeeBit LDA)"},
                    {"toolkit": "lingfeat", "code": "WBKF_:BClar05_S", "name": "weebit_semantic_clarity_50", "description": "Semantic clarity (50 topics, WeeBit LDA)"},
                    {"toolkit": "lingfeat", "code": "WBKF_:BNois05_S", "name": "weebit_semantic_noise_50", "description": "Semantic noise (50 topics, WeeBit)"},
                    {"toolkit": "lingfeat", "code": "WBKF_:BTopc05_S", "name": "weebit_topic_count_50", "description": "Number of topics (50 topics, WeeBit)"},
                    {"toolkit": "lingfeat", "code": "WBKF_:BRich10_S", "name": "weebit_semantic_richness_100", "description": "Semantic richness (100 topics, WeeBit)"},
                    {"toolkit": "lingfeat", "code": "WBKF_:BClar10_S", "name": "weebit_semantic_clarity_100", "description": "Semantic clarity (100 topics, WeeBit)"},
                    {"toolkit": "lingfeat", "code": "WBKF_:BNois10_S", "name": "weebit_semantic_noise_100", "description": "Semantic noise (100 topics, WeeBit)"},
                    {"toolkit": "lingfeat", "code": "WBKF_:BTopc10_S", "name": "weebit_topic_count_100", "description": "Number of topics (100 topics, WeeBit)"},
                    {"toolkit": "lingfeat", "code": "WBKF_:BRich15_S", "name": "weebit_semantic_richness_150", "description": "Semantic richness (150 topics, WeeBit)"},
                    {"toolkit": "lingfeat", "code": "WBKF_:BClar15_S", "name": "weebit_semantic_clarity_150", "description": "Semantic clarity (150 topics, WeeBit)"},
                    {"toolkit": "lingfeat", "code": "WBKF_:BNois15_S", "name": "weebit_semantic_noise_150", "description": "Semantic noise (150 topics, WeeBit)"},
                    {"toolkit": "lingfeat", "code": "WBKF_:BTopc15_S", "name": "weebit_topic_count_150", "description": "Number of topics (150 topics, WeeBit)"},
                    {"toolkit": "lingfeat", "code": "WBKF_:BRich20_S", "name": "weebit_semantic_richness_200", "description": "Semantic richness (200 topics, WeeBit)"},
                    {"toolkit": "lingfeat", "code": "WBKF_:BClar20_S", "name": "weebit_semantic_clarity_200", "description": "Semantic clarity (200 topics, WeeBit)"},
                    {"toolkit": "lingfeat", "code": "WBKF_:BNois20_S", "name": "weebit_semantic_noise_200", "description": "Semantic noise (200 topics, WeeBit)"},
                    {"toolkit": "lingfeat", "code": "WBKF_:BTopc20_S", "name": "weebit_topic_count_200", "description": "Number of topics (200 topics, WeeBit)"},
                    {"toolkit": "lingfeat", "code": "OSKF_:ORich05_S", "name": "onestop_semantic_richness_50", "description": "Semantic richness (50 topics, OneStopEng LDA)"},
                    {"toolkit": "lingfeat", "code": "OSKF_:OClar05_S", "name": "onestop_semantic_clarity_50", "description": "Semantic clarity (50 topics, OneStopEng)"},
                    {"toolkit": "lingfeat", "code": "OSKF_:ONois05_S", "name": "onestop_semantic_noise_50", "description": "Semantic noise (50 topics, OneStopEng)"},
                    {"toolkit": "lingfeat", "code": "OSKF_:OTopc05_S", "name": "onestop_topic_count_50", "description": "Number of topics (50 topics, OneStopEng)"},
                    {"toolkit": "lingfeat", "code": "OSKF_:ORich10_S", "name": "onestop_semantic_richness_100", "description": "Semantic richness (100 topics, OneStopEng)"},
                    {"toolkit": "lingfeat", "code": "OSKF_:OClar10_S", "name": "onestop_semantic_clarity_100", "description": "Semantic clarity (100 topics, OneStopEng)"},
                    {"toolkit": "lingfeat", "code": "OSKF_:ONois10_S", "name": "onestop_semantic_noise_100", "description": "Semantic noise (100 topics, OneStopEng)"},
                    {"toolkit": "lingfeat", "code": "OSKF_:OTopc10_S", "name": "onestop_topic_count_100", "description": "Number of topics (100 topics, OneStopEng)"},
                    {"toolkit": "lingfeat", "code": "OSKF_:ORich15_S", "name": "onestop_semantic_richness_150", "description": "Semantic richness (150 topics, OneStopEng)"},
                    {"toolkit": "lingfeat", "code": "OSKF_:OClar15_S", "name": "onestop_semantic_clarity_150", "description": "Semantic clarity (150 topics, OneStopEng)"},
                    {"toolkit": "lingfeat", "code": "OSKF_:ONois15_S", "name": "onestop_semantic_noise_150", "description": "Semantic noise (150 topics, OneStopEng)"},
                    {"toolkit": "lingfeat", "code": "OSKF_:OTopc15_S", "name": "onestop_topic_count_150", "description": "Number of topics (150 topics, OneStopEng)"},
                    {"toolkit": "lingfeat", "code": "OSKF_:ORich20_S", "name": "onestop_semantic_richness_200", "description": "Semantic richness (200 topics, OneStopEng)"},
                    {"toolkit": "lingfeat", "code": "OSKF_:OClar20_S", "name": "onestop_semantic_clarity_200", "description": "Semantic clarity (200 topics, OneStopEng)"},
                    {"toolkit": "lingfeat", "code": "OSKF_:ONois20_S", "name": "onestop_semantic_noise_200", "description": "Semantic noise (200 topics, OneStopEng)"},
                    {"toolkit": "lingfeat", "code": "OSKF_:OTopc20_S", "name": "onestop_topic_count_200", "description": "Number of topics (200 topics, OneStopEng)"},
                ]
            },
            "D8.4_Indian_Corpus_Topics": {
            "description": "AdSem features (Lee, Jang & Lee, 2021, EMNLP) computed from LDA topic models trained on a random sample of 10,000 Bhasha-Wiki-Indic articles (soketlabs/bhasha-wiki-indic, Indian-context English Wikipedia). Four metrics — Richness, Clarity, Noise, Topic count — at four topic granularities (K=50/100/150/200). Background corpus chosen to match Indian educational content; LDA trained with symmetric priors, 5 passes, seed=42.",
            "features": [
                {"toolkit": "manual", "code": "BWiki05Rich_S", "name": "wiki_india_50_topics_richness", "description": "Semantic richness at K=50: Σ (rank+1)·p_i over descending topic probabilities; captures topical breadth"},
                {"toolkit": "manual", "code": "BWiki05Clar_S", "name": "wiki_india_50_topics_clarity", "description": "Semantic clarity at K=50: (1/n)·Σ(p_0−p_i) for i=1..n; measures dominance of the top topic over others"},
                {"toolkit": "manual", "code": "BWiki05Nois_S", "name": "wiki_india_50_topics_noise", "description": "Semantic noise at K=50: n·Σ(p−1/n)⁴ / (Σ(p−1/n)²)²; kurtosis of topic distribution relative to uniform baseline"},
                {"toolkit": "manual", "code": "BWiki05Topc_S", "name": "wiki_india_50_topics_count", "description": "Number of active topics at K=50: count of topics returned above Gensim's default probability threshold"},

                {"toolkit": "manual", "code": "BWiki10Rich_S", "name": "wiki_india_100_topics_richness", "description": "Semantic richness at K=100"},
                {"toolkit": "manual", "code": "BWiki10Clar_S", "name": "wiki_india_100_topics_clarity", "description": "Semantic clarity at K=100"},
                {"toolkit": "manual", "code": "BWiki10Nois_S", "name": "wiki_india_100_topics_noise", "description": "Semantic noise at K=100"},
                {"toolkit": "manual", "code": "BWiki10Topc_S", "name": "wiki_india_100_topics_count", "description": "Number of active topics at K=100"},

                {"toolkit": "manual", "code": "BWiki15Rich_S", "name": "wiki_india_150_topics_richness", "description": "Semantic richness at K=150"},
                {"toolkit": "manual", "code": "BWiki15Clar_S", "name": "wiki_india_150_topics_clarity", "description": "Semantic clarity at K=150"},
                {"toolkit": "manual", "code": "BWiki15Nois_S", "name": "wiki_india_150_topics_noise", "description": "Semantic noise at K=150"},
                {"toolkit": "manual", "code": "BWiki15Topc_S", "name": "wiki_india_150_topics_count", "description": "Number of active topics at K=150"},

                {"toolkit": "manual", "code": "BWiki20Rich_S", "name": "wiki_india_200_topics_richness", "description": "Semantic richness at K=200"},
                {"toolkit": "manual", "code": "BWiki20Clar_S", "name": "wiki_india_200_topics_clarity", "description": "Semantic clarity at K=200"},
                {"toolkit": "manual", "code": "BWiki20Nois_S", "name": "wiki_india_200_topics_noise", "description": "Semantic noise at K=200"},
                {"toolkit": "manual", "code": "BWiki20Topc_S", "name": "wiki_india_200_topics_count", "description": "Number of active topics at K=200"}
            ]
}
        }
    },


    # =========================================================================
    # MACRO-TIER D: TEXT-GLOBAL & SEMANTIC LEVEL
    # =========================================================================

    "D9_Semantics_and_Pragmatics": {
        "description": "Broad semantic/narrative/pragmatic complexity: concreteness/abstractness, figurative language, narrative structure, cultural/pragmatic load. Merged from former D9 (Semantic Abstractness), D10 (Narrative Complexity), D11 (Pragmatic & Cultural Load).",
        "subdimensions": {


            "D9.1_Figurative_Language": {
                "description": "Metaphor, simile,  personification density.",
                "features": [
                    {"toolkit": "manual", "code": "simile_density", "name": "simile_density", "description": "Simile count per sentence via spaCy Matcher over core English comparison patterns"},
                    {"toolkit": "manual", "code": "metaphor_density", "name": "metaphor_density", "description": "Proportion of words classified as metaphorical by XLM-RoBERTa (VUA Metaphor Corpus)"},
                    {"toolkit": "manual", "code": "personification_density", "name": "personification_density", "description": "Personification instances per sentence via WordNet human-entity typing and vector-backed human-action predicates"},
                ]
            },

            "D9.2_Narrative_Structure": {
                "description": "Character count, character introduction rate, story arc complexity.",
                "features": [
                    {"toolkit": "manual", "code": "information_density_content_words", "name": "information_density_content_words", "description": "Content-word ratio (NOUN+VERB+ADJ+ADV / total words); lexical density proxy"},
                    {"toolkit": "manual", "code": "propositional_density", "name": "propositional_density", "description": "Propositions per word via CPIDR algorithm (ideadensity package)"},
                    {"toolkit": "manual", "code": "character_count", "name": "character_count", "description": "Unique characters via spaCy NER PERSON + PROPN deduplication"},
                    {"toolkit": "manual", "code": "character_introduction_rate", "name": "character_introduction_rate", "description": "Average new characters introduced per sentence"},
                    {"toolkit": "manual", "code": "story_arc_complexity", "name": "story_arc_complexity", "description": "Std deviation of per-sentence VADER compound sentiment scores"},
               
                ]
            },

            "D9.3_Temporal_and_Tense_Structure": {
                "description": "Temporal sequencing and tense dynamics, including explicit temporal-expression load.",
                "features": [
                    {"toolkit": "manual", "code": "chronological_order_score", "name": "chronological_order_score", "description": "Forward-vs-backward temporal progression score using broad temporal/discourse sequencing markers (not only DATE/TIME entities)"},
                    {"toolkit": "manual", "code": "temporal_expression_density", "name": "temporal_expression_density", "description": "Temporal expressions per sentence using broad temporal lexicon (sequence adverbs/connectives + temporal cues)"},
                    {"toolkit": "manual", "code": "tense_consistency", "name": "tense_consistency", "description": "Proportion of finite verbs sharing the dominant tense (spaCy morph)"},
                    {"toolkit": "manual", "code": "tense_switch_frequency", "name": "tense_switch_frequency", "description": "Tense switches per finite verb (normalised)"},
                   
                ]
            },

            "D9.4_Register_and_Formality": {
                "description": "Formality score, contraction density, pronoun-based narrative perspective.",
                "features": [
                    {"toolkit": "manual", "code": "formality_score", "name": "formality_score", "description": "Heylighen & Dewaele (1999) formality index [0-100] from POS frequencies"},
                    {"toolkit": "manual", "code": "contraction_density", "name": "contraction_density", "description": "Contractions per word token using contraction-expansion library checks with regex fallback for robustness"},
                    {"toolkit": "manual", "code": "first_person_pronoun_density", "name": "first_person_pronoun_density", "description": "First-person pronoun tokens / total word tokens"},
                    {"toolkit": "manual", "code": "second_person_pronoun_density", "name": "second_person_pronoun_density", "description": "Second-person pronoun tokens / total word tokens"},
                    {"toolkit": "manual", "code": "third_person_pronoun_density", "name": "third_person_pronoun_density", "description": "Third-person pronoun tokens / total word tokens"},
                   
                ]
            },

            "D9.5_Pragmatic_Inference_Load": {
                "description": "Sentence type distribution, speech act diversity, implicature markers.",
                "features": [
                    {"toolkit": "manual", "code": "question_density", "name": "question_density", "description": "Proportion of sentences ending with a question mark"},
                    {"toolkit": "manual", "code": "exclamation_density", "name": "exclamation_density", "description": "Proportion of sentences ending with an exclamation mark"},
                    {"toolkit": "manual", "code": "imperative_density", "name": "imperative_density", "description": "Proportion of imperative clauses via dependency structure and imperative cue patterns"},
                    {"toolkit": "manual", "code": "implicature_marker_density", "name": "implicature_marker_density", "description": "Discourse/stance marker density per sentence using expanded multiword + lexical pragmatic marker inventory"},
                    
                ]
            },

            "D9.6_Cultural_and_World_Knowledge_Load": {
                "description": "Culture-specific references, domain-specific vocabulary, proper noun density.",
                "features": [
                    {
                        "toolkit": "manual",
                        "code": "cultural_entity_density",
                        "name": "cultural_entity_density",
                        "description": "Fraction of detected named entities (PERSON, GPE, LOC, ORG, NORP, FAC, EVENT, WORK_OF_ART, PROPN) that match an Indian-context lexicon built from Bhasha-Wiki-Indic (~146k entities, freq>=3); 0.0 if no entities detected"
                    },
                    {
                        "toolkit": "manual",
                        "code": "domain_vocab_ratio",
                        "name": "domain_vocab_ratio",
                        "description": "Fraction of content lemmas (NOUN/VERB/ADJ/ADV, len>2) with Zipf frequency below 3.0 via wordfreq  indicates domain-specific or low-frequency vocabulary load"
                    }
    ],
    },

            "D9.7_Text_Easability_Principal_Components": {
                "description": "CohMetrix's PCA-derived composite scores that span multiple linguistic levels — serve as pre-computed dimension scores from CohMetrix.",
                "features": [
                    {"toolkit": "cohmetrix", "code": "PCNARz", "name": "narrativity_z", "description": "Text Easability: Narrativity (z-score)"},
                    {"toolkit": "cohmetrix", "code": "PCNARp", "name": "narrativity_percentile", "description": "Text Easability: Narrativity (percentile)"},
                    {"toolkit": "cohmetrix", "code": "PCSYNz", "name": "syntactic_simplicity_z", "description": "Text Easability: Syntactic simplicity (z-score)"},
                    {"toolkit": "cohmetrix", "code": "PCSYNp", "name": "syntactic_simplicity_percentile", "description": "Text Easability: Syntactic simplicity (percentile)"},
                    {"toolkit": "cohmetrix", "code": "PCCNCz", "name": "word_concreteness_z", "description": "Text Easability: Word concreteness (z-score)"},
                    {"toolkit": "cohmetrix", "code": "PCCNCp", "name": "word_concreteness_percentile", "description": "Text Easability: Word concreteness (percentile)"},
                    {"toolkit": "cohmetrix", "code": "PCREFz", "name": "referential_cohesion_z", "description": "Text Easability: Referential cohesion (z-score)"},
                    {"toolkit": "cohmetrix", "code": "PCREFp", "name": "referential_cohesion_percentile", "description": "Text Easability: Referential cohesion (percentile)"},
                    {"toolkit": "cohmetrix", "code": "PCDCz", "name": "deep_cohesion_z", "description": "Text Easability: Deep cohesion (z-score)"},
                    {"toolkit": "cohmetrix", "code": "PCDCp", "name": "deep_cohesion_percentile", "description": "Text Easability: Deep cohesion (percentile)"},
                    {"toolkit": "cohmetrix", "code": "PCVERBz", "name": "verb_cohesion_z", "description": "Text Easability: Verb cohesion (z-score)"},
                    {"toolkit": "cohmetrix", "code": "PCVERBp", "name": "verb_cohesion_percentile", "description": "Text Easability: Verb cohesion (percentile)"},
                    {"toolkit": "cohmetrix", "code": "PCCONNz", "name": "connectivity_z", "description": "Text Easability: Connectivity (z-score)"},
                    {"toolkit": "cohmetrix", "code": "PCCONNp", "name": "connectivity_percentile", "description": "Text Easability: Connectivity (percentile)"},
                    {"toolkit": "cohmetrix", "code": "PCTEMPz", "name": "temporality_z", "description": "Text Easability: Temporality (z-score)"},
                    {"toolkit": "cohmetrix", "code": "PCTEMPp", "name": "temporality_percentile", "description": "Text Easability: Temporality (percentile)"},
                ]
            },
        }
    },


    "D10_Readability_Surface_Indices": {
        "description": "Traditional readability formulas — surface proxies that combine word length, sentence length, syllable counts into a single score. Useful as baselines and for teacher-facing interpretation.",
        "subdimensions": {

            "D10.1_Classic_Readability_Formulas": {
                "description": "Flesch-Kincaid, Gunning Fog, SMOG, Coleman-Liau, ARI.",
                "features": [
                    {"toolkit": "LFTK", "code": "fkre", "name": "flesch_reading_ease", "description": "Flesch-Kincaid Reading Ease (0-100, higher = easier)"},
                    {"toolkit": "LFTK", "code": "fkgl", "name": "flesch_kincaid_grade_level", "description": "Flesch-Kincaid Grade Level"},
                    {"toolkit": "LFTK", "code": "fogi", "name": "gunning_fog_index", "description": "Gunning Fog Index"},
                    {"toolkit": "LFTK", "code": "smog", "name": "smog_index", "description": "SMOG Index"},
                    {"toolkit": "LFTK", "code": "cole", "name": "coleman_liau_index", "description": "Coleman-Liau Index"},
                    {"toolkit": "LFTK", "code": "auto", "name": "automated_readability_index", "description": "Automated Readability Index"},
                ]
            },

            "D10.2_Additional_Readability": {
                "description": "Dale-Chall, L2 readability, and other composite scores.",
                "features": [
                    {"toolkit": "cohmetrix", "code": "READNDC", "name": "dale_chall_readability", "description": "New Dale-Chall Readability Score"},
                    {"toolkit": "cohmetrix", "code": "RDL2", "name": "l2_readability_score", "description": "Second language readability score (Coh-Metrix L2)"},
                    {"toolkit": "textstat", "code": "textstat.spache_readability", "name": "spache_readability", "description": "Spache readability formula (grades 1-3)"},
                    {"toolkit": "textstat", "code": "textstat.linsear_write_formula", "name": "linsear_write", "description": "Linsear Write Formula"},
                    {"toolkit": "textstat", "code": "textstat.text_standard", "name": "text_standard_consensus", "description": "Consensus grade level from multiple formulas"},
                    {"toolkit": "textstat", "code": "textstat.reading_time", "name": "reading_time", "description": "Estimated reading time"},
                ]
            },
        }
    },
}



# =============================================================================
# FEATURE COUNT SUMMARY
# =============================================================================

def count_features():
    total = 0
    toolkit_counts = {}
    for dim_key, dim_val in DIMENSION_FEATURE_MAP.items():
        for sub_key, sub_val in dim_val["subdimensions"].items():
            for feat in sub_val.get("features", []):
                total += 1
                tk = feat["toolkit"]
                toolkit_counts[tk] = toolkit_counts.get(tk, 0) + 1
    return total, toolkit_counts


if __name__ == "__main__":
    total, by_toolkit = count_features()
    print(f"TOTAL FEATURES MAPPED: {total}")
    print(f"BY TOOLKIT: {by_toolkit}")
    print()

    print("\nDIMENSION SUMMARY:")
    for dim_key, dim_val in DIMENSION_FEATURE_MAP.items():
        feat_count = sum(len(sv.get("features", [])) for sv in dim_val["subdimensions"].values())
        print(f"  {dim_key}: {feat_count} features")



TOTAL FEATURES MAPPED: 532
BY TOOLKIT: {'LFTK': 185, 'cohmetrix': 99, 'manual': 64, 'taales': 8, 'TAMMI': 42, 'lingfeat': 130, 'textstat': 4}


DIMENSION SUMMARY:
  D1_Surface_Word_Level: 31 features
  D2_Morphological_Complexity: 42 features
  D3_Lexical_Sophistication: 43 features
  D4_Syntactic_Complexity: 73 features
  D5_Grammatical_Density_and_Diversity: 95 features
  D6_Textual_Cohesion: 84 features
  D7_Semantic_Coherence: 27 features
  D8_Discourse_Structure_and_Organisation: 86 features
  D9_Semantics_and_Pragmatics: 39 features
  D10_Readability_Surface_Indices: 12 features


In [14]:
# LingFeat code cleanup: keep only the feature token after the family prefix
# Example: 'PhrF_:as_NoPhr_C' -> 'as_NoPhr_C'


def _strip_lingfeat_prefix(code_value):
    code_text = str(code_value).strip()
    if ':' in code_text:
        return code_text.split(':', 1)[1].strip()
    return code_text


lingfeat_changes = []
for dim_key, dim_val in DIMENSION_FEATURE_MAP.items():
    for sub_key, sub_val in dim_val.get('subdimensions', {}).items():
        for feat in sub_val.get('features', []):
            toolkit = str(feat.get('toolkit', '')).strip().lower()
            if toolkit != 'lingfeat':
                continue

            old_code = str(feat.get('code', '')).strip()
            new_code = _strip_lingfeat_prefix(old_code)

            if new_code != old_code:
                feat['code'] = new_code
                lingfeat_changes.append((dim_key, sub_key, old_code, new_code))

print(f'LingFeat code prefixes removed: {len(lingfeat_changes)}')
if lingfeat_changes:
    print('Examples:')
    for row in lingfeat_changes[:20]:
        print(f'  {row[2]}  ->  {row[3]}')


LingFeat code prefixes removed: 123
Examples:
  PhrF_:as_NoPhr_C  ->  as_NoPhr_C
  PhrF_:at_NoPhr_C  ->  at_NoPhr_C
  PhrF_:ra_NoVeP_C  ->  ra_NoVeP_C
  PhrF_:ra_NoSuP_C  ->  ra_NoSuP_C
  PhrF_:ra_NoPrP_C  ->  ra_NoPrP_C
  PhrF_:ra_NoAjP_C  ->  ra_NoAjP_C
  PhrF_:ra_NoAvP_C  ->  ra_NoAvP_C
  PhrF_:to_VePhr_C  ->  to_VePhr_C
  PhrF_:as_VePhr_C  ->  as_VePhr_C
  PhrF_:at_VePhr_C  ->  at_VePhr_C
  PhrF_:ra_VeNoP_C  ->  ra_VeNoP_C
  PhrF_:ra_VeSuP_C  ->  ra_VeSuP_C
  PhrF_:ra_VePrP_C  ->  ra_VePrP_C
  PhrF_:ra_VeAjP_C  ->  ra_VeAjP_C
  PhrF_:ra_VeAvP_C  ->  ra_VeAvP_C
  PhrF_:to_SuPhr_C  ->  to_SuPhr_C
  PhrF_:as_SuPhr_C  ->  as_SuPhr_C
  PhrF_:at_SuPhr_C  ->  at_SuPhr_C
  PhrF_:ra_SuNoP_C  ->  ra_SuNoP_C
  PhrF_:ra_SuVeP_C  ->  ra_SuVeP_C


In [15]:
# SECTION 3B — Replacement-ready taxonomy adapter + robust flattening
# This cell lets you switch to a comprehensive DIMENSION_FEATURE_MAP (10-dim taxonomy)
# while keeping NB1 downstream code unchanged (feature_groups / ALL_DECLARED).

from collections import defaultdict


def _normalize_toolkit_name(name):
    return str(name).strip().lower() if name is not None else 'unknown'


def build_feature_groups_from_dimension_map(dimension_feature_map):
    """
    Convert schema:
      DIMENSION_FEATURE_MAP[dim]['subdimensions'][sub]['features'] -> list of dicts with 'code'
    into NB1-compatible schema:
      feature_groups[dim][sub] -> [feature_code, ...]

    Returns:
      feature_groups_new : dict
      all_declared_new   : list (unique, first-seen order)
      flat_rows          : list[dict] (dimension/subdimension/toolkit/code/name/description)
      toolkit_counts     : dict
      dimension_counts   : dict
    """
    feature_groups_new = {}
    flat_rows = []
    toolkit_counts = defaultdict(int)
    dimension_counts = defaultdict(int)

    for dim_key, dim_val in (dimension_feature_map or {}).items():
        sub_map = {}
        subdimensions = dim_val.get('subdimensions', {}) if isinstance(dim_val, dict) else {}

        for sub_key, sub_val in subdimensions.items():
            features = sub_val.get('features', []) if isinstance(sub_val, dict) else []
            codes = []
            seen_in_sub = set()

            for feat in features:
                if not isinstance(feat, dict):
                    continue
                code = str(feat.get('code', '')).strip()
                if not code:
                    continue

                toolkit = _normalize_toolkit_name(feat.get('toolkit', 'unknown'))
                name = str(feat.get('name', code)).strip() or code
                description = str(feat.get('description', '')).strip()

                flat_rows.append({
                    'dimension': dim_key,
                    'subdimension': sub_key,
                    'toolkit': toolkit,
                    'code': code,
                    'name': name,
                    'description': description,
                })

                toolkit_counts[toolkit] += 1

                if code not in seen_in_sub:
                    codes.append(code)
                    seen_in_sub.add(code)

            if codes:
                sub_map[sub_key] = codes
                dimension_counts[dim_key] += len(codes)

        if sub_map:
            feature_groups_new[dim_key] = sub_map

    all_declared_new = []
    seen_global = set()
    for row in flat_rows:
        code = row['code']
        if code not in seen_global:
            all_declared_new.append(code)
            seen_global.add(code)

    return (
        feature_groups_new,
        all_declared_new,
        flat_rows,
        dict(toolkit_counts),
        dict(dimension_counts),
    )


def flatten_feature_groups(feature_groups_dict):
    """
    Flatten NB1-style feature_groups into list rows.
    Useful if DIMENSION_FEATURE_MAP is not provided.
    """
    rows = []
    for dim, subgs in (feature_groups_dict or {}).items():
        for subg, feats in (subgs or {}).items():
            for code in feats:
                rows.append({
                    'dimension': dim,
                    'subdimension': subg,
                    'toolkit': 'unknown',
                    'code': str(code),
                    'name': str(code),
                    'description': '',
                })
    return rows


# If your comprehensive DIMENSION_FEATURE_MAP exists in the kernel,
# it becomes the active source of truth.
if 'DIMENSION_FEATURE_MAP' in globals() and isinstance(DIMENSION_FEATURE_MAP, dict) and len(DIMENSION_FEATURE_MAP) > 0:
    (
        feature_groups,
        ALL_DECLARED,
        FEATURE_FLAT_ROWS,
        FEATURE_TOOLKIT_COUNTS,
        FEATURE_DIMENSION_COUNTS,
    ) = build_feature_groups_from_dimension_map(DIMENSION_FEATURE_MAP)

    print(' Using DIMENSION_FEATURE_MAP as active feature grouping source')
else:
    # Fallback to the existing feature_groups defined in Section 3
    FEATURE_FLAT_ROWS = flatten_feature_groups(feature_groups)

    _seen = set()
    _ordered = []
    for row in FEATURE_FLAT_ROWS:
        c = row['code']
        if c not in _seen:
            _ordered.append(c)
            _seen.add(c)
    ALL_DECLARED = _ordered

    FEATURE_TOOLKIT_COUNTS = {'unknown': len(FEATURE_FLAT_ROWS)}
    FEATURE_DIMENSION_COUNTS = {
        dim: sum(len(v) for v in subgs.values())
        for dim, subgs in feature_groups.items()
    }

    print('ℹ️ DIMENSION_FEATURE_MAP not found; using existing feature_groups fallback')


# Convenient flattened dataframe for filtering/search/export
FEATURE_CATALOG_DF = pd.DataFrame(FEATURE_FLAT_ROWS)

print(f'Total declared features (unique): {len(ALL_DECLARED)}')
print(f'Total flattened feature rows     : {len(FEATURE_FLAT_ROWS)}')
print('Toolkit counts (row-level):', FEATURE_TOOLKIT_COUNTS)
print('Dimension counts:')
for dim, count in FEATURE_DIMENSION_COUNTS.items():
    n_sub = len(feature_groups.get(dim, {}))
    print(f'  {dim}: {n_sub} subgroups, {count} features')


 Using DIMENSION_FEATURE_MAP as active feature grouping source
Total declared features (unique): 531
Total flattened feature rows     : 532
Toolkit counts (row-level): {'lftk': 185, 'cohmetrix': 99, 'manual': 64, 'taales': 8, 'tammi': 42, 'lingfeat': 130, 'textstat': 4}
Dimension counts:
  D1_Surface_Word_Level: 4 subgroups, 31 features
  D2_Morphological_Complexity: 5 subgroups, 42 features
  D3_Lexical_Sophistication: 8 subgroups, 43 features
  D4_Syntactic_Complexity: 5 subgroups, 73 features
  D5_Grammatical_Density_and_Diversity: 4 subgroups, 95 features
  D6_Textual_Cohesion: 4 subgroups, 84 features
  D7_Semantic_Coherence: 5 subgroups, 27 features
  D8_Discourse_Structure_and_Organisation: 4 subgroups, 86 features
  D9_Semantics_and_Pragmatics: 7 subgroups, 39 features
  D10_Readability_Surface_Indices: 2 subgroups, 12 features


# SECTION 4 — Filter to Present Features + Density Normalisation


# Feature cleaning 

In [16]:
# 4.1 Keep only features that are actually present in the workbook
if 'english_only_feature' in globals() and english_only_feature is not None:
    candidate_features = list(dict.fromkeys(list(english_only_feature)))
    feature_source = 'english_only_feature'
else:
    candidate_features = ALL_DECLARED
    feature_source = 'ALL_DECLARED (DIMENSION_FEATURE_MAP)'

present       = [f for f in candidate_features if f in df_no_text.columns]
missing_feats = [f for f in candidate_features if f not in df_no_text.columns]
df = df_no_text.copy()

print(f'Feature source            : {feature_source}')
print(f'Candidate features total  : {len(candidate_features)}')
print(f'Features present in CSV   : {len(present)}')
print(f'Features missing from CSV : {len(missing_feats)}')
if missing_feats:
    print('  (first 10 missing):', missing_feats[:10])


Feature source            : ALL_DECLARED (DIMENSION_FEATURE_MAP)
Candidate features total  : 531
Features present in CSV   : 531
Features missing from CSV : 0


In [17]:
# ════════════════════════════════════════════════════════════════════════
# 4.1.5  Basic feature cleaning  (PRE-density-normalisation)
# ════════════════════════════════════════════════════════════════════════
#   Density normalisation (cell 4.2) divides count features by t_word,
#   which is itself variable across texts. So a feature that is constant
#   in raw form (e.g. always 0) becomes spuriously variable after
#   normalisation (0/t_word*100 = 0 trivially, but constants other than
#   zero turn into 1/t_word noise). Degenerate-feature checks therefore
#   must run BEFORE normalisation, on the raw `df`.
#
#   Drops:
#     (a) > MAX_MISSING_FRACTION NaN
#     (b) near-zero variance  (constant or numerically constant in raw form)
#     (c) all-zero
#     (d) exact duplicate columns in raw form
#
#   Updates `df` and `present`.  Cell 4.2 then builds df_norm,
#   feature_groups_present, feature_names and X_all from the cleaned set.
# ════════════════════════════════════════════════════════════════════════
from collections import Counter

NEAR_ZERO_VAR_THRESHOLD = 1e-8     # std below this = numerically constant
MAX_MISSING_FRACTION    = 0.20     # drop col if > 20% NaN

# Work only on feature columns; replace inf -> NaN before checks
feat_view = df[present].replace([np.inf, -np.inf], np.nan)

drop_reasons = {}

# 1. High missingness ---------------------------------------------------
miss_frac = feat_view.isna().mean()
for c in miss_frac[miss_frac > MAX_MISSING_FRACTION].index:
    drop_reasons[c] = f'missing {miss_frac[c]*100:.1f}%'

# 2. Near-zero variance IN RAW FORM (the key check) --------------------
stds = feat_view.std(axis=0, skipna=True)
for c in stds[(stds.isna()) | (stds < NEAR_ZERO_VAR_THRESHOLD)].index:
    drop_reasons.setdefault(c, f'near-zero variance raw (std={stds[c]:.2e})')

# 3. All-zero (edge case of std=0 with all values == 0) ----------------
for c in present:
    if c in drop_reasons:
        continue
    if feat_view[c].fillna(0).abs().sum() == 0:
        drop_reasons[c] = 'all zeros'

# 4. Exact duplicate columns in raw form -------------------------------
seen = {}
for c in [c for c in present if c not in drop_reasons]:
    sig = hash(tuple(np.round(feat_view[c].fillna(0).values, 10)))
    if sig in seen:
        drop_reasons[c] = f'duplicate of {seen[sig]}'
    else:
        seen[sig] = c

# 5. Apply drops -------------------------------------------------------
cols_to_drop = list(drop_reasons.keys())
n_before     = len(present)

df      = df.drop(columns=cols_to_drop)
present = [f for f in present if f not in drop_reasons]

# Sync DIMENSION_FEATURE_MAP so Stage-1 Spearman UFS (Section 6) is consistent
if 'DIMENSION_FEATURE_MAP' in globals():
    for dim_val in DIMENSION_FEATURE_MAP.values():
        if isinstance(dim_val, dict) and 'subdimensions' in dim_val:
            for sub_val in dim_val['subdimensions'].values():
                if isinstance(sub_val, dict) and 'features' in sub_val:
                    sub_val['features'] = [
                        f for f in sub_val['features']
                        if isinstance(f, dict) and f.get('code', '') in df.columns
                    ]

# 6. Report ------------------------------------------------------------
print('=' * 74)
print('  BASIC FEATURE CLEANING  (pre-normalisation, raw values)')
print('=' * 74)
print(f'  Rows                 : {df.shape[0]}')
print(f'  Features before      : {n_before}')
print(f'  Features dropped     : {len(cols_to_drop)}')
print(f'  Features after       : {len(present)}')

if cols_to_drop:
    by_reason = Counter()
    for r in drop_reasons.values():
        if   'missing'   in r: by_reason['high missing']  += 1
        elif 'variance'  in r: by_reason['near-zero var'] += 1
        elif 'zeros'     in r: by_reason['all zero']      += 1
        elif 'duplicate' in r: by_reason['duplicate']     += 1
    print('\n  Drops by reason:')
    for k, v in by_reason.most_common():
        print(f'    {k:<18}  {v:>4}')

    print('\n  First 20 dropped features:')
    for c in cols_to_drop[:20]:
        print(f'    {c:<38}  ({drop_reasons[c]})')
    if len(cols_to_drop) > 20:
        print(f'    ... and {len(cols_to_drop) - 20} more')

print('=' * 74)

  BASIC FEATURE CLEANING  (pre-normalisation, raw values)
  Rows                 : 172
  Features before      : 531
  Features dropped     : 29
  Features after       : 502

  Drops by reason:
    near-zero var         22
    duplicate              7

  First 20 dropped features:
    a_space_pw                              (near-zero variance raw (std=0.00e+00))
    a_space_ps                              (near-zero variance raw (std=0.00e+00))
    simp_space_var                          (near-zero variance raw (std=0.00e+00))
    root_space_var                          (near-zero variance raw (std=0.00e+00))
    corr_space_var                          (near-zero variance raw (std=0.00e+00))
    t_n_ent_money                           (near-zero variance raw (std=0.00e+00))
    a_n_ent_money_pw                        (near-zero variance raw (std=0.00e+00))
    a_n_ent_money_ps                        (near-zero variance raw (std=0.00e+00))
    t_n_ent_percent                         (ne

In [18]:
# 4.2  Density normalisation: normalize raw count features by total words (per 100 words)
WORD_COUNT_COL = 't_word'

if WORD_COUNT_COL is not None and WORD_COUNT_COL not in df.columns:
    print(f'WARNING: "{WORD_COUNT_COL}" not found — skipping normalisation')
    WORD_COUNT_COL = None


def _is_integer_valued(series):
    vals = pd.to_numeric(series, errors='coerce').dropna().to_numpy()
    if vals.size == 0:
        return False
    return np.allclose(vals, np.round(vals))


# Values here are count-like by name, but should NOT be normalized by word count.
# They are already ratios/averages/scales or represent aggregate non-count scores.
RAW_COUNT_EXCLUDE = {
    # Already averages/SDs, not raw counts
    'DESPL',        # Mean paragraph length (sentences/paragraph) - average, not count
    'DESPLd',       # SD of paragraph length - not a count
    'grapheme_count_per_word',
    'phoneme_count_per_word',
}

# Already density-normalized features (per 100 words from extraction) - do NOT normalize again
# These come from Coh-Metrix, TAALES, etc. and are already incidence/density values
ALREADY_NORMALIZED = {
    # Pronoun incidence (Coh-Metrix D5.6)
    'WRDPRP1s',    # 1st person singular per 100 words
    'WRDPRP1p',    # 1st person plural per 100 words
    'WRDPRP2',     # 2nd person per 100 words
    'WRDPRP3s',    # 3rd person singular per 100 words
    'WRDPRP3p',    # 3rd person plural per 100 words
    
    # Part-of-speech incidence (Coh-Metrix)
    'WRDNOUN',     # Noun incidence per 100 words
    'WRDVERB',     # Verb incidence per 100 words
    'WRDADJ',      # Adjective incidence per 100 words
    'WRDADV',      # Adverb incidence per 100 words
    
    # Syntactic incidence features
    'SYNSTRUTa',   # Noun phrase incidence
    'SYNSTRUTd',   # Verb phrase incidence
    'SYNSTRUTe',   # Prepositional phrase incidence
    'SYNSTRUTt',   # Complement phrase incidence
    
    # Other incidence features
    'DESSC',       # Function word incidence (already per 100)
    'DESSL',       # Sentence length (average, not a count)
    'DESSLd',      # Sentence length standard deviation
}

# Exact raw-count columns from the current workbook that should be normalized.
# Excludes already-normalized density / ratio / per-word / per-sentence fields.
RAW_COUNT_EXACT = {
    # Paragraph counts (raw)
    'DESPC',        # Number of paragraphs (raw count) - YES, normalize this
    
    # Aggregate counts
    'aggregates.syllable_count',
    'aggregates.polysyllabcount',
    'aggregates.monosyllabcount',

    # Morphology counts
    'maxMorphCount',
    'polyMorphCount',
    'polyMorphStanza',

    # Entity / discourse counts (raw, not normalized)
    't_n_ent',
    't_n_ent_person',
    't_n_ent_norp',
    't_n_ent_fac',
    't_n_ent_org',
    't_n_ent_gpe',
    't_n_ent_loc',
    't_n_ent_product',
    't_n_ent_event',
    't_n_ent_art',
    't_n_ent_law',
    't_n_ent_language',
    't_n_ent_date',
    't_n_ent_time',
    't_n_ent_percent',
    't_n_ent_money',
    't_n_ent_quantity',
    't_n_ent_ordinal',
    't_n_ent_cardinal',
    'ent.total_entities',
    'ent.total_unique_entities',
    'total_free_morphemes',
    'total_bound_morphemes',
    
    # Punctuation counts
    't_punct',
    
    # Stopword counts
    't_stopword',
    
    # Syllable counts
    't_syll',
    't_syll2',
    't_syll3',
    
    # Unique word counts
    't_uword',
    
    # Character counts
    't_char',
    
    # === NEW: total_ and t_ features for AoA and frequency ===
    # Total features (raw sums)
    'total_kuperman_aoa',
    'total_brysbaert_aoa',
    'total_subtlex_us_zipf',
    
    # t_ features (raw sums)
    't_kup',
    't_bry',
    't_subtlex_us_zipf',
    'character_count'
}


def _is_raw_count_col(col_name, series):
    """
    Determine if a column should be normalized by word count.
    
    Returns True for raw count features (to_, n_, t_n_ prefixes, DESPC, etc.)
    Returns False for already-normalized features (incidence, density, ratios, averages, SDs)
    """
    name = col_name.strip()
    lower = name.lower()

    # ========== FORCE NORMALIZE: Raw sum features (may be stored as float) ==========
    # These are raw sums across all words and MUST be normalized per 100 words
    FORCE_NORMALIZE = {'t_kup', 't_bry', 't_subtlex_us_zipf',
                       'total_kuperman_aoa', 'total_brysbaert_aoa', 'total_subtlex_us_zipf'}
    if name in FORCE_NORMALIZE:
        return True

    # Explicit exclusions first: these are not meant to be normalized.
    if name in RAW_COUNT_EXCLUDE:
        return False
    
    # Explicit exclusion for already-normalized incidence/density features
    if name in ALREADY_NORMALIZED:
        return False

    # Pattern exclusions for already-normalized features
    # These patterns indicate the feature is already a density/ratio/incidence/average/SD
    exclude_patterns = [
        '_pw',           # per word
        '_ps',           # per sentence
        '_var',          # variance
        '_density',      # density
        '_ratio',        # ratio
        '_norm',         # normalized
        '_avg',          # average
        '_mean',         # mean
        '_sd',           # standard deviation
        'average',
        'mean',
        'score',
        'incidence',     # incidence (per 100 words)
    ]
    
    if any(k in lower for k in exclude_patterns):
        return False

    # Exact raw count matches (from curated list)
    if name in RAW_COUNT_EXACT:
        return _is_integer_valued(series)

    # Pattern matches for raw count features:
    # 1. to_* - phrasal counts (to_PrPhr_C, to_AvPhr_C, etc.)
    # 2. n_* - noun/adjective/verb counts (n_adj, n_verb, etc.)
    # 3. t_n_* - entity counts (already covered but for completeness)
    # 4. count_* - any column starting with 'count'
    # 5. *_count - any column ending with '_count'
    # 6. total_* - any column starting with 'total_' (raw sums)
    # 7. t_* - any column starting with 't_' (excluding known non-count)
    if (lower.startswith('to_') or 
        lower.startswith('n_') or 
        lower.startswith('t_n_') or
        lower.startswith('count') or
        lower.startswith('total_') or
        lower.endswith('_count')):
        return _is_integer_valued(series)
    
    # Catch t_ features (including t_kup, t_bry, t_subtlex_us_zipf, etc.)
    if lower.startswith('t_'):
        # Exclude only t_word and t_sent (denominators)
        # All other t_ features should be normalized
        if name not in ['t_word', 't_sent']:
            return _is_integer_valued(series)

    # Generic catch-all for integer-valued raw counts
    # Look for patterns like 'n_' in the middle
    if '_n_' in lower:
        return _is_integer_valued(series)

    return False


# Normalize raw count-like columns even if Excel imported them as float64
count_cols = [c for c in present if c != WORD_COUNT_COL and _is_raw_count_col(c, df[c])]

# FORCE add any missing AoA/Zipf features that might have been missed
force_aoa_zipf = ['t_kup', 't_bry', 't_subtlex_us_zipf', 
                  'total_kuperman_aoa', 'total_brysbaert_aoa', 'total_subtlex_us_zipf']
for col in force_aoa_zipf:
    if col in present and col not in count_cols and col != WORD_COUNT_COL:
        count_cols.append(col)
        print(f'FORCE adding {col} for normalization')

# Special handling: ensure to_ features and DESPC are included even if not caught by pattern
for col in present:
    if col not in count_cols and col != WORD_COUNT_COL:
        # Check for DESPC explicitly
        if col == 'DESPC' and _is_integer_valued(df[col]):
            if col not in count_cols:
                count_cols.append(col)
        # Check for to_ features
        elif col.startswith('to_') and _is_integer_valued(df[col]):
            if col not in count_cols:
                count_cols.append(col)
        # Check for total_ features
        elif col.startswith('total_') and _is_integer_valued(df[col]):
            if col not in count_cols:
                count_cols.append(col)
        # Check for t_ features (excluding t_word, t_sent)
        elif col.startswith('t_') and col not in ['t_word', 't_sent'] and _is_integer_valued(df[col]):
            if col not in count_cols:
                count_cols.append(col)

# Remove duplicates (in case force added something already in count_cols)
count_cols = list(set(count_cols))

df_norm = df.copy()
if WORD_COUNT_COL is not None and count_cols:
    word_counts = df_norm[WORD_COUNT_COL].replace(0, np.nan)
    for col in count_cols:
        df_norm[col] = (df_norm[col] / word_counts) * 100
    df_norm[present] = df_norm[present].fillna(0)

    print(f'Normalised {len(count_cols)} raw count features → density per 100 words')
    print('\nFirst 25 normalized features:')
    for idx, feat in enumerate(sorted(count_cols)[:25], start=1):
        print(f'{idx:>3}. {feat}')
    
    # Show which features were EXCLUDED (already normalized)
    excluded = [c for c in present if c != WORD_COUNT_COL and not _is_raw_count_col(c, df[c])]
    already_norm = [c for c in excluded if c in ALREADY_NORMALIZED or any(p in c.lower() for p in ['_pw', '_ps', 'incidence', '_sd', '_mean'])]
    print(f'\nSkipped normalisation for {len(excluded)} features (already density/incidence/ratio/average/SD)')
    print(f'  - {len(already_norm)} were explicitly identified as already-normalized')
    
    print('\nAll normalized features (full list):')
    for idx, feat in enumerate(sorted(count_cols), start=1):
        print(f'{idx:>3}. {feat}')
    
    # Specifically show total_ and t_ features that were normalized
    total_features = [f for f in count_cols if f.startswith('total_')]
    t_features = [f for f in count_cols if f.startswith('t_') and f not in ['t_word', 't_sent']]
    if total_features:
        print(f'\n--- total_ features normalized ({len(total_features)}): ---')
        for f in total_features:
            print(f'  - {f}')
    if t_features:
        print(f'\n--- t_ features normalized ({len(t_features)}): ---')
        for f in t_features:
            print(f'  - {f}')
            
else:
    df_norm[present] = df_norm[present].fillna(0)
    if not count_cols:
        print('No raw count features found for normalisation.')
    else:
        print('Normalisation skipped. NaN fill applied.')

# 4.3  Rebuild feature_groups to only include present features
feature_groups_present = {}
for dim_name, subgroups in feature_groups.items():
    rebuilt_sg = {}
    for sg_name, feats in subgroups.items():
        kept = [f for f in feats if f in df_norm.columns]
        if kept:
            rebuilt_sg[sg_name] = kept
    if rebuilt_sg:
        feature_groups_present[dim_name] = rebuilt_sg

print('\n' + '='*60)
print('FEATURE GROUPS AFTER NORMALISATION')
print('='*60)
for dim, subgs in feature_groups_present.items():
    total = sum(len(v) for v in subgs.values())
    print(f'{dim:15s}: {len(subgs)} subgroups, {total} features')

X_all         = df_norm[present].values
y             = df_norm[LABEL_COL].values
feature_names = present
print(f'\nX shape: {X_all.shape}  |  y shape: {y.shape}')
print(f'Classes : {np.unique(y)}')

Normalised 38 raw count features → density per 100 words

First 25 normalized features:
  1. character_count
  2. t_bry
  3. t_char
  4. t_kup
  5. t_n_ent
  6. t_n_ent_art
  7. t_n_ent_cardinal
  8. t_n_ent_date
  9. t_n_ent_event
 10. t_n_ent_fac
 11. t_n_ent_gpe
 12. t_n_ent_language
 13. t_n_ent_law
 14. t_n_ent_loc
 15. t_n_ent_norp
 16. t_n_ent_ordinal
 17. t_n_ent_org
 18. t_n_ent_person
 19. t_n_ent_product
 20. t_n_ent_quantity
 21. t_n_ent_time
 22. t_punct
 23. t_stopword
 24. t_subtlex_us_zipf
 25. t_syll

Skipped normalisation for 464 features (already density/incidence/ratio/average/SD)
  - 97 were explicitly identified as already-normalized

All normalized features (full list):
  1. character_count
  2. t_bry
  3. t_char
  4. t_kup
  5. t_n_ent
  6. t_n_ent_art
  7. t_n_ent_cardinal
  8. t_n_ent_date
  9. t_n_ent_event
 10. t_n_ent_fac
 11. t_n_ent_gpe
 12. t_n_ent_language
 13. t_n_ent_law
 14. t_n_ent_loc
 15. t_n_ent_norp
 16. t_n_ent_ordinal
 17. t_n_ent_org
 18. t_n

In [19]:
# 4.4 Build modelling matrices
le        = LabelEncoder()
y_encoded = le.fit_transform(y)
y_float   = y.astype(float)
X_df      = pd.DataFrame(X_all, columns=feature_names)

GRADE_MIN = int(np.min(y_float))
GRADE_MAX = int(np.max(y_float))
GRADES_ARR = np.arange(GRADE_MIN, GRADE_MAX + 1, dtype=int)
print(f'X_df: {X_df.shape}   GRADE_MIN={GRADE_MIN}  GRADE_MAX={GRADE_MAX}')

# Sanity: if texts available, length should match X_df length
if all_texts is not None:
    assert len(all_texts) == len(X_df), \
        f'Text/feature row mismatch: {len(all_texts)} vs {len(X_df)}'
    print('Texts row-aligned with feature matrix.')


X_df: (172, 502)   GRADE_MIN=2  GRADE_MAX=8
Texts row-aligned with feature matrix.


In [20]:
# 'present' may still reference dropped columns. Sync it with what's actually in df_norm.
present_clean = [f for f in present if f in df_norm.columns]
dropped = set(present) - set(present_clean)
if dropped:
    print(f'Removing {len(dropped)} dropped column(s) from `present`: {sorted(dropped)}')
    present = present_clean
    feature_names = [f for f in feature_names if f in df_norm.columns]

# Also sync DIMENSION_FEATURE_MAP so Stage-1 doesn't try to use dropped columns
for dim_name, dim_val in DIMENSION_FEATURE_MAP.items():
    if isinstance(dim_val, dict) and 'subdimensions' in dim_val:
        for sub_name, sub_val in dim_val.get('subdimensions', {}).items():
            if isinstance(sub_val, dict) and 'features' in sub_val:
                sub_val['features'] = [
                    feat for feat in sub_val['features']
                    if isinstance(feat, dict) and feat.get('code', '') in df_norm.columns
                ]

# Rebuild X_all and X_df with the cleaned column list
X_all = df_norm[present].values.astype(np.float64)
X_df  = pd.DataFrame(X_all, columns=feature_names)
print(f'X_all dtype: {X_all.dtype}, shape: {X_all.shape}')
print(f'X_df dtypes: {X_df.dtypes.value_counts().to_dict()}')

X_all dtype: float64, shape: (172, 502)
X_df dtypes: {dtype('float64'): 502}


# SECTION 5 — Single Source of Truth: Outer CV + Shared Helpers

This is the **only** place where outer folds are defined. Every downstream component (feature selection, hyperparameter tuning, ensembling, MLP training, transformer fine-tuning) reuses `OUTER_CV` and the materialised `outer_splits` list. This guarantees that the test fold for, say, ensemble E1 in fold 2 is the *exact same* set of rows that fold 2 used during NB1 selection — eliminating any cross-stage leakage.


In [21]:
# Helpers for per-model checkpoint paths
def _safe_model_name(model_name):
    return model_name.replace(' ','_').replace('/','_').replace('(','').replace(')','')

def _model_dir(model_name):
    d = os.path.join(CKPT_DIR, _safe_model_name(model_name))
    os.makedirs(d, exist_ok=True)
    return d

def _model_ckpt(model_name, fold):
    return os.path.join(_model_dir(model_name),
                        f'_MODEL_{_safe_model_name(model_name)}_fold{fold}.pkl')

def _dim_ckpt(model_name, dim_name, fold):
    safe = lambda s: s.replace(' ','_').replace('/','_').replace('(','').replace(')','')
    return os.path.join(_model_dir(model_name),
                        f'_C_{safe(model_name)}_{safe(dim_name)}_f{fold}.pkl')


In [22]:
# Outer CV — single source of truth
OUTER_CV     = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
outer_splits = list(OUTER_CV.split(X_df, y_encoded))

print(f'Outer fold sizes (canonical splits used everywhere downstream):')
for k, (tr, te) in enumerate(outer_splits):
    dist = dict(zip(*np.unique(y_encoded[te], return_counts=True)))
    print(f'  fold {k}: train={len(tr):3d}  test={len(te):3d}  test_dist={dist}')

# QWK scorer (classification) and QWK regression scorer for FS
qwk_scorer = make_scorer(cohen_kappa_score, weights='quadratic')

def _reg_qwk_fs(y_true, y_pred_raw):
    y_true_i = np.rint(np.asarray(y_true)).astype(int)
    y_pred_i = np.clip(np.rint(np.asarray(y_pred_raw)).astype(int), GRADE_MIN, GRADE_MAX)
    return float(cohen_kappa_score(y_true_i, y_pred_i, weights='quadratic'))

qwk_reg_scorer = make_scorer(_reg_qwk_fs, greater_is_better=True)

def _fs_scorer(task):
    return qwk_scorer if task == 'classification' else qwk_reg_scorer

def within_one(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)) <= 1))


Outer fold sizes (canonical splits used everywhere downstream):
  fold 0: train=137  test= 35  test_dist={np.int64(0): np.int64(6), np.int64(1): np.int64(3), np.int64(2): np.int64(5), np.int64(3): np.int64(5), np.int64(4): np.int64(5), np.int64(5): np.int64(6), np.int64(6): np.int64(5)}
  fold 1: train=137  test= 35  test_dist={np.int64(0): np.int64(5), np.int64(1): np.int64(4), np.int64(2): np.int64(4), np.int64(3): np.int64(6), np.int64(4): np.int64(5), np.int64(5): np.int64(5), np.int64(6): np.int64(6)}
  fold 2: train=138  test= 34  test_dist={np.int64(0): np.int64(5), np.int64(1): np.int64(4), np.int64(2): np.int64(4), np.int64(3): np.int64(6), np.int64(4): np.int64(5), np.int64(5): np.int64(5), np.int64(6): np.int64(5)}
  fold 3: train=138  test= 34  test_dist={np.int64(0): np.int64(5), np.int64(1): np.int64(4), np.int64(2): np.int64(4), np.int64(3): np.int64(5), np.int64(4): np.int64(6), np.int64(5): np.int64(5), np.int64(6): np.int64(5)}
  fold 4: train=138  test= 34  test_dist

In [23]:
# Generic alignment helpers used by every ensemble
def regression_to_proba(raw_vals, grade_min, grade_max, sigma=0.8):
    grades = np.arange(grade_min, grade_max + 1, dtype=float)
    raw_vals = np.asarray(raw_vals, dtype=float).reshape(-1, 1)
    z = -((raw_vals - grades[None, :]) / max(sigma, 1e-6)) ** 2
    z -= z.max(axis=1, keepdims=True)
    p = np.exp(z)
    p /= p.sum(axis=1, keepdims=True)
    return p

def align_proba(proba, classes_grades, grade_min, grade_max):
    n_test  = proba.shape[0]
    n_grade = grade_max - grade_min + 1
    out = np.zeros((n_test, n_grade), dtype=float)
    for j, g in enumerate(classes_grades):
        gi = int(g) - grade_min
        if 0 <= gi < n_grade:
            out[:, gi] += proba[:, j]
    rs = out.sum(axis=1, keepdims=True)
    out = np.where(rs > 0, out / np.where(rs > 0, rs, 1.0), 1.0 / n_grade)
    return out

def metric_bundle(y_true, y_pred, name=None):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    m = {
        'Accuracy'    : accuracy_score(y_true, y_pred),
        'Within_1'    : within_one(y_true, y_pred),
        'QWK'         : cohen_kappa_score(y_true, y_pred, weights='quadratic'),
        'F1_weighted' : f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'F1_macro'    : f1_score(y_true, y_pred, average='macro',    zero_division=0),
    }
    if name:
        print(f'  {name:<48} | Acc={m["Accuracy"]:.4f}  W1={m["Within_1"]:.4f}  '
              f'QWK={m["QWK"]:.4f}  F1w={m["F1_weighted"]:.4f}  F1m={m["F1_macro"]:.4f}')
    return m

print('Helpers ready.')


Helpers ready.


In [24]:
# ─── Stage 1: Spearman UFS + Pearson dedup (per-dimension) ────────────────────
# UNCHANGED — simple univariate filter (|rho| >= SPEARMAN_THR & p <= P_THR)
# followed by Pearson collinearity dedup at PEARSON_THR. Uses outer-train only.
def stage1_ufs(X_train_df, y_train, group_features):
    spearman_map, survivors = {}, []
    for f in group_features:
        if f not in X_train_df.columns:
            continue
        r, p = spearmanr(X_train_df[f].values, y_train)
        if abs(r) >= SPEARMAN_THR and p <= P_THR:
            survivors.append(f)
            spearman_map[f] = abs(r)
    to_drop = set()
    for i in range(len(survivors)):
        for j in range(i + 1, len(survivors)):
            fi, fj = survivors[i], survivors[j]
            if fi in to_drop or fj in to_drop: continue
            r_ij, _ = pearsonr(X_train_df[fi].values, X_train_df[fj].values)
            if abs(r_ij) > PEARSON_THR:
                to_drop.add(fi if spearman_map[fi] < spearman_map[fj] else fj)
    return [f for f in survivors if f not in to_drop], spearman_map


# ─── FS Constants ─────────────────────────────────────────────────────────────
# MIN_FEATURES_FLOOR is a guard against the 1-or-2-feature collapse that
# 1-SE FS exhibited on this data (Ridge selecting 1 feature in fold 4 in the
# prior implementation). It binds rarely in practice with argmax selection.
MIN_FEATURES_FLOOR = 1







def _fs_compatible_estimator(estimator):
    """
    """
    est = clone(estimator)
    if hasattr(est, 'n_jobs'):
        est.n_jobs = 1
    return est


def _make_fs_pipe(estimator):
    """(StandardScaler -> clf). Same wrapper Section 7 hyperparameter tuning uses,
    so |coef|-based ranking is on z-scaled features (apples-to-apples for linear
    models) and the FS pipeline structure mirrors the tuning pipeline structure."""
    return Pipeline([('scaler', StandardScaler()),
                     ('clf',    _fs_compatible_estimator(estimator))])


def _fs_importance_getter(est):
    """RFECV importance_getter: |coef_| (linear) or feature_importances_ (tree)."""
    clf = est.named_steps['clf']
    if hasattr(clf, 'coef_'):
        return np.abs(np.asarray(clf.coef_).ravel())
    if hasattr(clf, 'feature_importances_'):
        return np.asarray(clf.feature_importances_).ravel()
    raise ValueError(
        f"Estimator {type(clf).__name__} exposes neither coef_ nor "
        f"feature_importances_ — not usable with RFECV.")


def _inner_cv(task):
    """StratifiedKFold for both regression and classification .

    For regression, we stratify on the integer-rounded y (which we always pass
    as `y_for_cv` — see _run_rfecv). This ensures every inner-validation fold
    contains every grade represented in the outer-train data, preventing the
    occasional "fold without grade k" that plain KFold can produce on small
    skewed ordinal datasets. The estimator inside RFECV is still fit on the
    integer y, but for regression that is just np.rint of y_float and the
    model treats it as continuous (Ridge, ElasticNet, LinearSVR all accept
    integer-valued continuous targets without any change in behavior).

    sklearn warns rather than errors when a class has fewer members than
    n_splits; that is acceptable on this small dataset (some grades have
    only ~3 samples in some outer-train splits).
    """
    return StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True,
                           random_state=RANDOM_STATE)


# ─── Shared RFECV runner ──────────────────────────────────────────────────────
def _run_rfecv(X_g, y_for_cv, feature_list, estimator, task, apply_cap):
    """
    Off-the-shelf sklearn RFECV. Returns (selected, mean_qwk, std_qwk, cap_info).

  

    
    SCORING — ROUND+CLIP QWK in FS, FITTED ORDINAL THRESHOLDS in tuning.
        The FS scorer (`_reg_qwk_fs` for regression) maps continuous prediction
        to ordinal label via simple round+clip. The reported outer-test QWK in
        Section 7 uses fitted ordinal thresholds. These two metrics are NOT on
        the same scale (~0.01-0.03 QWK gap), so inner-CV QWKs in FS logs are
        NOT directly comparable to the QWKs in the results table. This is
        deliberate, not an oversight: feature ranking depends on whether
        features are predictive of the ordinal target, not on post-hoc
        calibration of the prediction-to-label mapping. Round+clip and
        fitted-threshold QWK are highly rank-correlated as functions of the
        feature subset, so the chosen feature set is robust to the choice of
        scorer; nesting threshold fitting inside RFECV's inner CV would
        require a custom feature-elimination loop and would not change the
        chosen feature set in practice.

    REPORTED METRICS ARE TRUE FOR THE FEATURES RETURNED.
        When the cap fires, we re-map `chosen_idx` to the cap's index in
        `rfecv.cv_results_` BEFORE reading metrics, so `mean_qwk` / `std_qwk`
        are RFECV's true inner-CV mean/std at exactly the capped feature
        count. cap_info also reports the inner-CV QWK at the argmax count
        for transparency.

    DETERMINISTIC SUPPORT FOR THE CHOSEN COUNT.
        sklearn RFECV's `support_` is the support at its internal optimum
        (`n_features_`). When our chosen count differs (cap fires), we refit
        a single RFE pass on the full outer-train data with
        `n_features_to_select=chosen_n` — yielding a clean support that
        matches the count whose inner-CV score we report.

    LEAKAGE.  Outer-train data is the only data this function ever sees:
    splits come from `_inner_cv()` which partitions inputs (which are the
    outer-train rows from `run_model_selection_fold`).  No outer-test rows
    enter feature selection at any point.
    """
    cv_inner = _inner_cv(task)
    scorer   = qwk_scorer if task == 'classification' else qwk_reg_scorer

    pipe  = _make_fs_pipe(estimator)
    rfecv = RFECV(
        estimator=pipe, step=1, cv=cv_inner, scoring=scorer,
        min_features_to_select=MIN_FEATURES_FLOOR,
        importance_getter=_fs_importance_getter, n_jobs=1,
    )
    rfecv.fit(X_g, y_for_cv)

    means = np.asarray(rfecv.cv_results_['mean_test_score'])
    stds  = np.asarray(rfecv.cv_results_['std_test_score'])
    # RFECV indexing: cv_results_ index k <-> (MIN_FEATURES_FLOOR + k) features.

    # Argmax selection (uniform across both FS stages).
    chosen_idx = int(np.argmax(means))

    # Stage 3 only: per-fold hard cap = floor(n_train / 5).
    cap_info = None
    if apply_cap:
        n_train   = X_g.shape[0]
        final_cap = max(MIN_FEATURES_FLOOR, n_train // 5)
        cap_idx   = final_cap - MIN_FEATURES_FLOOR
        if 0 <= cap_idx < len(means) and chosen_idx > cap_idx:
            cap_info = {
                'cap'        : int(final_cap),
                'argmax_n'   : int(MIN_FEATURES_FLOOR + chosen_idx),
                'argmax_qwk' : float(means[chosen_idx]),
                'argmax_std' : float(stds[chosen_idx]),
                'n_train'    : int(n_train),
            }
            chosen_idx = cap_idx

    chosen_n = MIN_FEATURES_FLOOR + chosen_idx

    # Get a deterministic support of size `chosen_n`.
    if chosen_n == rfecv.n_features_:
        mask = rfecv.support_
    else:
        rfe = RFE(
            estimator=_make_fs_pipe(estimator),
            n_features_to_select=chosen_n, step=1,
            importance_getter=_fs_importance_getter,
        )
        rfe.fit(X_g, y_for_cv)
        mask = rfe.support_

    selected = [feature_list[i] for i, s in enumerate(mask) if s]
    return selected, float(means[chosen_idx]), float(stds[chosen_idx]), cap_info


# ─── Stage 2: Per-dimension RFECV (argmax, no cap) ────────────────────────────
def stage2_rfecv(X_train_df, y_train, feature_list, estimator, task,
                 model_name='', label=''):
    """Per-dimension RFECV with argmax. Returns selected feature list."""
    if len(feature_list) <= MIN_FEATURES_FLOOR:
        return feature_list

    y_for_cv = (
        np.clip(np.rint(np.asarray(y_train)).astype(int), GRADE_MIN, GRADE_MAX)
        if task == 'regression' else y_train
    )
    X_g = X_train_df[feature_list].values

    selected, mean_qwk, std_qwk, _ = _run_rfecv(
        X_g, y_for_cv, feature_list, estimator, task, apply_cap=False
    )
    if label:
        print(f'    {label}: {len(feature_list)} -> {len(selected)} features '
              f'(QWK={mean_qwk:.4f}±{std_qwk:.4f})')
    return selected


# Backward-compat alias so the existing run_model_selection_fold call site works
# unchanged (and per-dimension Stage-2 checkpoint pickles still resume cleanly).
stage2_rfe = stage2_rfecv


# ─── Stage 3: Cross-group RFECV (argmax + per-fold hard cap) ──────────────────
def stage3_rfecv(X_train_df, y_train, feature_list, estimator, task,
                 model_name='', label=''):
    """Cross-group RFECV with argmax + per-fold hard cap of floor(n_train/5)

    The cap typically binds for RF (whose feature_importances_ produces a wide
    inner-CV plateau) and rarely for linear / SVR (which converge to fewer
    features than the cap).
    """
    if len(feature_list) <= MIN_FEATURES_FLOOR:
        return feature_list

    y_for_cv = (
        np.clip(np.rint(np.asarray(y_train)).astype(int), GRADE_MIN, GRADE_MAX)
        if task == 'regression' else y_train
    )
    X_g = X_train_df[feature_list].values

    selected, mean_qwk, std_qwk, cap_info = _run_rfecv(
        X_g, y_for_cv, feature_list, estimator, task, apply_cap=False
    )

    if label:
        if cap_info is not None:
            cap_msg = (f' [capped at {cap_info["cap"]} '
                       f'(=n_train/5 with n_train={cap_info["n_train"]}); '
                       f'argmax was n={cap_info["argmax_n"]} '
                       f'QWK={cap_info["argmax_qwk"]:.4f}'
                       f'±{cap_info["argmax_std"]:.4f}]')
        else:
            cap_msg = ''
        print(f'    {label}: {len(feature_list)} -> {len(selected)} features '
              f'(QWK={mean_qwk:.4f}±{std_qwk:.4f}){cap_msg}')
    return selected


print('Simple 3-stage feature selection ready: '
      'Stage1=Spearman+Pearson, Stage2=RFECV(argmax), '
      'Stage3=RFECV(argmax). '
      'StratifiedKFold inner CV for both regression and classification. '
      'Inner-CV QWK uses round+clip; outer-test QWK in Section 7 uses fitted '
      'ordinal thresholds (deliberate — see _run_rfecv docstring).')


Simple 3-stage feature selection ready: Stage1=Spearman+Pearson, Stage2=RFECV(argmax), Stage3=RFECV(argmax). StratifiedKFold inner CV for both regression and classification. Inner-CV QWK uses round+clip; outer-test QWK in Section 7 uses fitted ordinal thresholds (deliberate — see _run_rfecv docstring).


# SECTION 6 — Per-Model 3-Stage Feature Selection

For each of the four base models (ElasticNet, Ridge, RF, SVM), we run Stage 1 → Stage 2 (per dimension) → Stage 3 (cross-group RFECV) **on each outer-train fold only**. The result is a per-fold list of selected features that downstream stages will reuse.

Per-fold checkpoints are saved under `unified_artifacts/<MODEL>/_MODEL_<MODEL>_fold<k>.pkl`.


In [25]:
def run_model_selection_fold(model_name, estimator, task, fold_k, train_idx, test_idx):
    mckpt = _model_ckpt(model_name, fold_k)
    if os.path.exists(mckpt):
        with open(mckpt, 'rb') as f:
            cached = pickle.load(f)
        if cached.get('final_features'):
            print(f'  [fold={fold_k}] cached: {len(cached["final_features"])} features')
            return cached
        else:
            os.remove(mckpt)

    y_use      = y_encoded if task == 'classification' else y_float
    X_train_df = X_df.iloc[train_idx].reset_index(drop=True)
    y_train    = y_use[train_idx]
    print(f'  [fold={fold_k}] train={len(train_idx)} samples')

    group_results, all_s2 = {}, []
    for dim_name, subgroups in feature_groups_present.items():
        dim_feats = list(dict.fromkeys(f for sg in subgroups.values() for f in sg if f in X_df.columns))
        dckpt = _dim_ckpt(model_name, dim_name, fold_k)

        if os.path.exists(dckpt):
            with open(dckpt, 'rb') as f:
                saved = pickle.load(f)
            stale = (saved.get('n_input') != len(dim_feats)
                     or saved.get('n_s1', 0) == 0
                     or saved.get('n_s2', 0) == 0
                     or not saved.get('s2_features'))
            if stale:
                os.remove(dckpt)
            else:
                group_results[dim_name] = saved
                all_s2.extend(saved['s2_features'])
                print(f'    [{dim_name}] resumed: S1:{saved["n_s1"]}->S2:{saved["n_s2"]}')
                continue

        s1, _ = stage1_ufs(X_train_df, y_train, dim_feats)
        print(f'    [{dim_name}] S1:{len(dim_feats)}->{len(s1)}', end='')
        s2 = stage2_rfe(X_train_df, y_train, s1, estimator, task,
                        model_name=model_name, label=' S2(inner-4fold)')
        res_dim = {'dim_name': dim_name, 'n_input': len(dim_feats),
                   'n_s1': len(s1), 'n_s2': len(s2),
                   's1_features': s1, 's2_features': s2}
        with open(dckpt, 'wb') as f:
            pickle.dump(res_dim, f)
        group_results[dim_name] = res_dim
        all_s2.extend(s2)

    pool = list(dict.fromkeys(all_s2))
    print(f'  Stage2 pool: {len(pool)} features')
    final = stage3_rfecv(X_train_df, y_train, pool, estimator, task,
                         model_name=model_name, label='Stage3(inner-4fold)')

    result = {
        'model_name'    : model_name,
        'task'          : task,
        'outer_fold'    : fold_k,
        'train_idx'     : np.asarray(train_idx, dtype=int),
        'test_idx'      : np.asarray(test_idx, dtype=int),
        'group_results' : group_results,
        'stage2_pool'   : pool,
        'final_features': final,
    }
    with open(mckpt, 'wb') as f:
        pickle.dump(result, f)

    # Save fold JSON for transformer hybrid section
    fjson = os.path.join(_model_dir(model_name),
                         f'fold_features_{model_name}_fold{fold_k}.json')
    with open(fjson, 'w') as fj:
        json.dump({
            'model'           : model_name,
            'task'            : task,
            'outer_fold'      : fold_k,
            'n_train'         : len(train_idx),
            'n_test'          : len(test_idx),
            'train_idx'       : list(map(int, train_idx)),
            'test_idx'        : list(map(int, test_idx)),
            'final_features'  : final,
            'n_final_features': len(final),
        }, fj, indent=2)
    print(f'  fold={fold_k}: {len(final)} final features\n')
    return result


def run_model_selection(model_name, estimator, task):
    print(f'\n{"="*70}\n  {model_name}  |  task={task}\n{"="*70}')
    fold_results = []
    for fold_k, (train_idx, test_idx) in enumerate(outer_splits):
        fr = run_model_selection_fold(model_name, estimator, task, fold_k, train_idx, test_idx)
        fold_results.append(fr)
    print(f'  features/fold = {[len(fr["final_features"]) for fr in fold_results]}')
    return fold_results


In [26]:

_estimator_ElasticNet = ElasticNet(max_iter=5000, random_state=RANDOM_STATE)
_estimator_Ridge      = Ridge(solver='cholesky', random_state=RANDOM_STATE)
_estimator_RF         = RandomForestClassifier(
    n_estimators=100, random_state=RANDOM_STATE,
    n_jobs=1, class_weight='balanced')

In [27]:

# # Run feature selection PER MODEL — uniform 3-stage pipeline using sklearn's
# # RFECV with each base model's native ranking attribute (|coef| for linear,
# # feature_importances_ for tree; LinearSVR surrogate ranks features for
# # SVR(RBF) since RBF has no coef_). Each base model still gets its own
# # per-fold feature list — selection is wrapper-style (Guyon & Elisseeff, 2003),
# # so the chosen support reflects each model's inductive bias even though the
# # protocol is uniform.
# #
# # MODEL ROSTER (v4 — LGBM REMOVED):
# #   ElasticNet   — L1+L2 regression; |coef|-based ranking

# #   RandomForest — tree classifier; native feature_importances_





# # SVR — nonlinear regression base model (RBF kernel)
# _estimator_SVR = SVR(kernel='rbf', C=1.0, epsilon=0.2, gamma='scale')

# fs_results = {}
# fs_results['ElasticNet_Regressor'] = run_model_selection(
#     'ElasticNet_Regressor', _estimator_ElasticNet, 'regression')
# fs_results['Ridge_Regressor']      = run_model_selection(
#     'Ridge_Regressor',      _estimator_Ridge,      'regression')
# fs_results['RF_Classifier']        = run_model_selection(
#     'RF_Classifier',        _estimator_RF,         'classification')


# print('\nFeature selection complete (PER-MODEL, 4 independent runs).')
# print('Per-fold counts by model:')
# for mn, fr_list in fs_results.items():
#     counts = [len(fr['final_features']) for fr in fr_list]
#     print(f'  {mn:<24}: {counts}  (mean={np.mean(counts):.1f} \u00b1 {np.std(counts):.1f})')

# SECTION 7 — Per-Fold Hyperparameter Tuning (Nested CV)



In [28]:

PARAM_GRIDS = {
    'ElasticNet_Regressor': {
        'clf__alpha'    : [0.001, 0.01, 0.1, 1.0, 10.0],     # 5 × 5 = 25
        'clf__l1_ratio' : [0.1, 0.3, 0.5, 0.7, 0.9],
    },
  
        'RF_Classifier': {
        
        'clf__n_estimators'      : [200, 300, 500, 800],
        'clf__max_depth'         : [None, 8],
        'clf__min_samples_leaf'  : [2, 4],
        'clf__max_features'      : ['sqrt', 0.5],
    },
}



USE_1SE_RULE     = True


def _baseline_estimator(model_name, base_estimator):
    # Sensible default ('no-tune') configuration per model family.
    est = clone(base_estimator)
    if model_name == 'RF_Classifier':
        est.set_params(n_estimators=200, max_depth=None,
                       min_samples_leaf=1, max_features='sqrt',
                       class_weight='balanced', bootstrap=True, criterion='gini')
    elif model_name == 'ElasticNet_Regressor':
        est.set_params(alpha=0.1, l1_ratio=0.5, max_iter=10000)

    return est


def _select_with_1se_classifier(cv_results):
    """1-SE rule on sklearn cv_results_. Tie-break = max regularisation.

    Regularisation proxies for RF: shallower max_depth, larger min_samples_leaf,
    smaller max_features. n_estimators is excluded — more trees lower variance
    without adding complexity (Probst & Boulesteix 2018).
    """
    means = np.asarray(cv_results['mean_test_score'])
    stds  = np.asarray(cv_results['std_test_score'])
    best  = int(np.argmax(means))
    threshold = means[best] - stds[best]
    eligible = np.where(means >= threshold)[0]
    if len(eligible) == 0:
        return best, float(means[best]), float(stds[best])

    def reg_score(i):
        p = cv_results['params'][i]
        s = 0.0
        if 'clf__max_depth' in p:
            md = p['clf__max_depth']
            s += -1e6 if md is None else -float(md)
        if 'clf__min_samples_leaf' in p:
            s += float(p['clf__min_samples_leaf'])
        mf = p.get('clf__max_features')
        if mf is not None:
            s += 1.0 if mf == 'sqrt' else (-float(mf) if isinstance(mf, (int, float)) else 0.0)
        return s

    sel = max(eligible, key=reg_score)
    return int(sel), float(means[sel]), float(stds[sel])


def _select_with_1se_regression(all_results):
    """1-SE rule on a list of {params, mean, std}. Tie-break = larger alpha
    (more shrinkage), then larger l1_ratio for EN (sparser solution; Ridge
    has no l1_ratio so second key is 0.0)."""
    means = np.array([r['mean'] for r in all_results])
    stds  = np.array([r['std']  for r in all_results])
    best  = int(np.argmax(means))
    threshold = means[best] - stds[best]
    eligible = [i for i, m in enumerate(means) if m >= threshold]
    if not eligible:
        return best, float(means[best]), float(stds[best])
    sel = max(
        eligible,
        key=lambda i: (
            float(all_results[i]['params'].get('clf__alpha',    0.0)),
            float(all_results[i]['params'].get('clf__l1_ratio', 0.0)),
        ),
    )
    return int(sel), float(means[sel]), float(stds[sel])


# Ordinal threshold helpers (regression only) ────────────────────────────────────
def fit_ordinal_thresholds(y_true_int, y_pred_raw, grade_min, grade_max):
    """Greedy coordinate-descent threshold fit (Issue #2a + #2d).

    Approximate: the K-1 thresholds are optimized one at a time, repeating
    full sweeps over all thresholds until no sweep produces an improvement
    (max MAX_SWEEPS=5 sweeps). The candidate pool combines every unique
    training prediction with a 50-quantile grid (deduplicated and sorted),
    so each threshold can be placed at any meaningful position in the
    prediction distribution. Greedy coordinate descent is not guaranteed
    to find the global optimum; for K-1 = 6 thresholds and a candidate pool
    of ~150-200 points, a full grid search would be ~1.5 * 10^13 evaluations
    and is intractable. The achieved (not provably optimal) thresholding
    consistently improves over round+clip on this dataset.

    METHODS NOTE: cite this function as approximate threshold optimization
    via coordinate descent over a dense candidate pool, with iteration to
    fixed point. No claim of global optimality.

    LEAKAGE: thresholds are fit on training data only — both y_true_int and
    y_pred_raw come from the inner-train (or outer-train) split. Inner-val
    and outer-test data never enter threshold fitting.
    """
    y_true_int = np.rint(np.asarray(y_true_int)).astype(int)
    y_pred_raw = np.asarray(y_pred_raw, dtype=float)
    grades = np.arange(grade_min, grade_max + 1)
    K = len(grades)
    if K <= 1:
        return np.array([])

    pmin, pmax = float(np.min(y_pred_raw)), float(np.max(y_pred_raw))
    if pmin == pmax:
        return np.array([(grade_min + i + 0.5) for i in range(K - 1)])

    # Dense candidate pool: every unique training prediction plus a 50-quantile
    # grid, deduplicated and sorted. This is much denser than the original
    # 50-quantile-only pool (was: ~50 candidates; now: typically 150-200).
    quantile_pool = np.quantile(y_pred_raw, np.linspace(0.02, 0.98, 50))
    cand_pool = np.unique(np.concatenate([np.unique(y_pred_raw), quantile_pool]))

    init = np.linspace(pmin, pmax, K + 1)[1:-1]
    best = init.copy()
    best_qwk = _score_thresholds(y_true_int, y_pred_raw, best, grade_min, grade_max)

    # Multi-sweep coordinate descent until no sweep produces an improvement.
    MAX_SWEEPS = 5
    EPS = 1e-9
    for _ in range(MAX_SWEEPS):
        improved = False
        for t_idx in range(K - 1):
            for c in cand_pool:
                trial = best.copy()
                trial[t_idx] = c
                trial = np.sort(trial)
                qwk = _score_thresholds(y_true_int, y_pred_raw, trial, grade_min, grade_max)
                if qwk > best_qwk + EPS:
                    best_qwk = qwk
                    best = trial
                    improved = True
        if not improved:
            break
    return best


def _score_thresholds(y_true_int, y_pred_raw, thr, grade_min, grade_max):
    y_pred_int = apply_ordinal_thresholds(y_pred_raw, thr, grade_min, grade_max)
    return float(cohen_kappa_score(y_true_int, y_pred_int, weights='quadratic'))


def apply_ordinal_thresholds(y_pred_raw, thr, grade_min, grade_max):
    y_pred_raw = np.asarray(y_pred_raw, dtype=float)
    if thr is None or len(thr) == 0:
        return np.clip(np.rint(y_pred_raw).astype(int), grade_min, grade_max)
    bins = np.searchsorted(np.sort(thr), y_pred_raw)
    return np.clip(bins + grade_min, grade_min, grade_max).astype(int)


In [29]:

# def tune_classification_fold(model_name, estimator, fr):
#     fold_k    = fr['outer_fold']
#     train_idx = fr['train_idx']
#     test_idx  = fr['test_idx']
#     features  = [f for f in fr['final_features'] if f in X_df.columns]
#     if not features:
#         return None

#     X_tr = X_df[features].values[train_idx]
#     X_te = X_df[features].values[test_idx]
#     y_tr = y_encoded[train_idx]
#     y_te = y_encoded[test_idx]

#     cv_inner = StratifiedKFold(n_splits=N_TUNE_FOLDS, shuffle=True,
#                                random_state=RANDOM_STATE)
#     pipe = Pipeline([('scaler', StandardScaler()), ('clf', clone(estimator))])

#     param_grid = dict(PARAM_GRIDS[model_name])
#     if model_name == 'RF_Classifier':
#         param_grid['clf__class_weight'] = ['balanced']
#         param_grid['clf__bootstrap']    = [True]
#         param_grid['clf__criterion']    = ['gini']

#     search = GridSearchCV(pipe, param_grid, cv=cv_inner,
#                           scoring=qwk_scorer, refit=False, n_jobs=-1)
#     search.fit(X_tr, y_tr)

#     if USE_1SE_RULE:
#         sel_idx, inner_cv, inner_cv_std = _select_with_1se_classifier(search.cv_results_)
#     else:
#         sel_idx      = int(np.argmax(search.cv_results_['mean_test_score']))
#         inner_cv     = float(search.cv_results_['mean_test_score'][sel_idx])
#         inner_cv_std = float(search.cv_results_['std_test_score'][sel_idx])
#     best_params = search.cv_results_['params'][sel_idx]

#     fitted_pipe = clone(pipe).set_params(**best_params).fit(X_tr, y_tr)
#     y_pred      = fitted_pipe.predict(X_te).astype(int)
#     acc = accuracy_score(y_te, y_pred)
#     w1  = within_one(y_te, y_pred)
#     qwk = cohen_kappa_score(y_te, y_pred, weights='quadratic')
#     f1  = f1_score(y_te, y_pred, average='weighted', zero_division=0)

#     bundle = {
#         'model_name'     : model_name,
#         'task'           : 'classification',
#         'outer_fold'     : int(fold_k),
#         'features'       : features,
#         'n_features'     : len(features),
#         'best_params'    : best_params,
#         'inner_cv_score' : inner_cv,
#         'inner_cv_std'   : inner_cv_std,
#         'fitted_pipeline': fitted_pipe,
#         'thresholds'     : None,
#         'train_idx'      : train_idx,
#         'test_idx'       : test_idx,
#         'y_test'         : y_te,
#         'y_pred'         : y_pred,
#         'y_pred_raw'     : y_pred,
#         'acc': acc, 'w1': w1, 'qwk': qwk, 'f1': f1,
#         'choice'         : 'tuned',
#     }
#     print(f'  fold={fold_k} | n_feat={len(features):3d} | '
#           f'acc={acc:.3f} w1={w1:.3f} qwk={qwk:.3f} | '
#           f'inner_cv={inner_cv:.4f}±{inner_cv_std:.4f}')
#     return bundle


# def tune_regression_fold(model_name, estimator, fr):
#     fold_k    = fr['outer_fold']
#     train_idx = fr['train_idx']
#     test_idx  = fr['test_idx']
#     features  = [f for f in fr['final_features'] if f in X_df.columns]
#     if not features:
#         return None

#     X_tr = X_df[features].values[train_idx]
#     X_te = X_df[features].values[test_idx]
#     y_tr = y_float[train_idx]
#     y_te = y_float[test_idx]

#     y_strat  = np.clip(np.rint(y_tr).astype(int), GRADE_MIN, GRADE_MAX)
#     cv_inner = StratifiedKFold(n_splits=N_TUNE_FOLDS, shuffle=True,
#                                random_state=RANDOM_STATE)
#     pipe = Pipeline([('scaler', StandardScaler()), ('clf', clone(estimator))])

#     def _eval_params(params):
#         fold_qwks = []
#         for in_tr, in_va in cv_inner.split(X_tr, y_strat):
#             in_pipe = clone(pipe).set_params(**params)
#             in_pipe.fit(X_tr[in_tr], y_tr[in_tr])
#             raw_tr  = in_pipe.predict(X_tr[in_tr])
#             thr_in  = fit_ordinal_thresholds(y_tr[in_tr], raw_tr, GRADE_MIN, GRADE_MAX)
#             raw_va  = in_pipe.predict(X_tr[in_va])
#             pred_va = apply_ordinal_thresholds(raw_va, thr_in, GRADE_MIN, GRADE_MAX)
#             fold_qwks.append(float(cohen_kappa_score(
#                 y_tr[in_va].astype(int), pred_va, weights='quadratic')))
#         if not fold_qwks:
#             return -np.inf, 0.0
#         return float(np.mean(fold_qwks)), float(np.std(fold_qwks))

#     all_results = []
#     for params in ParameterGrid(PARAM_GRIDS[model_name]):
#         m, s = _eval_params(params)
#         all_results.append({'params': params, 'mean': m, 'std': s})

#     if USE_1SE_RULE:
#         sel_idx, inner_cv, inner_cv_std = _select_with_1se_regression(all_results)
#     else:
#         sel_idx      = int(np.argmax([r['mean'] for r in all_results]))
#         inner_cv     = float(all_results[sel_idx]['mean'])
#         inner_cv_std = float(all_results[sel_idx]['std'])
#     best_params = all_results[sel_idx]['params']

# # Collect OOF predictions using the same cv_inner splits (best_params already selected)
#     oof_raw = np.full(len(y_tr), np.nan)
#     for in_tr, in_va in cv_inner.split(X_tr, y_strat):
#         oof_pipe = clone(pipe).set_params(**best_params)
#         oof_pipe.fit(X_tr[in_tr], y_tr[in_tr])
#         oof_raw[in_va] = oof_pipe.predict(X_tr[in_va])   # out-of-sample for each fold
#     thresholds  = fit_ordinal_thresholds(y_tr, oof_raw, GRADE_MIN, GRADE_MAX)
    
#     # Final model on full outer-train (thresholds already calibrated above)
#     fitted_pipe = clone(pipe).set_params(**best_params).fit(X_tr, y_tr)
#     raw_te      = fitted_pipe.predict(X_te)
#     y_pred_int  = apply_ordinal_thresholds(raw_te, thresholds, GRADE_MIN, GRADE_MAX)
#     y_te_int    = y_te.astype(int)

#     acc = accuracy_score(y_te_int, y_pred_int)
#     w1  = within_one(y_te_int, y_pred_int)
#     qwk = cohen_kappa_score(y_te_int, y_pred_int, weights='quadratic')
#     f1  = f1_score(y_te_int, y_pred_int, average='weighted', zero_division=0)

#     bundle = {
#         'model_name'     : model_name,
#         'task'           : 'regression',
#         'outer_fold'     : int(fold_k),
#         'features'       : features,
#         'n_features'     : len(features),
#         'best_params'    : best_params,
#         'inner_cv_score' : inner_cv,
#         'inner_cv_std'   : inner_cv_std,
#         'fitted_pipeline': fitted_pipe,
#         'thresholds'     : thresholds,
#         'train_idx'      : train_idx,
#         'test_idx'       : test_idx,
#         'y_test'         : y_te_int,
#         'y_pred'         : y_pred_int,
#         'y_pred_raw'     : raw_te,
#         'acc': acc, 'w1': w1, 'qwk': qwk, 'f1': f1,
#         'choice'         : 'tuned',
#     }
#     print(f'  fold={fold_k} | n_feat={len(features):3d} | '
#           f'acc={acc:.3f} w1={w1:.3f} qwk={qwk:.3f} | '
#           f'inner_cv={inner_cv:.4f}±{inner_cv_std:.4f}')
#     return bundle


# def tune_model_all_folds(model_name, estimator, task):
#     print(f'\n{"="*70}\n  {model_name}  |  task={task}  |  '
#           f'1-SE={USE_1SE_RULE}\n{"="*70}')
#     bundles = []
#     for fr in fs_results[model_name]:
#         b = (tune_classification_fold(model_name, estimator, fr)
#              if task == 'classification'
#              else tune_regression_fold(model_name, estimator, fr))
#         if b is not None:
#             bundles.append(b)
#             with open(os.path.join(CKPT_DIR,
#                       f'tuned_model_{model_name}_fold{b["outer_fold"]}.pkl'), 'wb') as fp:
#                 pickle.dump({k: v for k, v in b.items()
#                              if k != 'fitted_pipeline'}, fp)
#     return bundles

# BASELINE

In [30]:
import os
os.environ['USE_TF']                  = '0'
os.environ['USE_TORCH']               = '1'
os.environ['USE_JAX']                 = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF']  = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM']   = 'false'
print('TF/JAX disabled for transformers; CUDA allocator set to expandable segments.')

TF/JAX disabled for transformers; CUDA allocator set to expandable segments.


In [31]:
import torch, transformers
 
def _detect_hardware():
    if not torch.cuda.is_available():
        return dict(gpu='cpu', cc=(0, 0), n_gpu=0, model_dtype=torch.float32,
                    attn='eager', amp_dtype=None, use_amp=False,
                    reference_compile=False, gen_dtype=torch.float32)
    cc = torch.cuda.get_device_capability(0)
    ampere = cc[0] >= 8                      # bf16 + FlashAttention-2 need sm80+
    return dict(gpu=torch.cuda.get_device_name(0), cc=cc,
                n_gpu=torch.cuda.device_count(),
                model_dtype=torch.float32,
                attn='flash_attention_2' if ampere else 'sdpa',
                amp_dtype=torch.bfloat16 if ampere else torch.float16,
                use_amp=True,
                reference_compile=ampere,
                gen_dtype=torch.bfloat16 if ampere else torch.float32)
 
HW = _detect_hardware()
for k, v in HW.items():
    print(f'  {k:<18}: {v}')
print(f'\ntransformers {transformers.__version__}')
_v = tuple(int(x) for x in transformers.__version__.split('.')[:2])
if _v < (4, 50):
    print('  ACTION: pip install -q -U "transformers>=4.50" "accelerate>=1.0", then RESTART')
if HW['cc'][0] < 8:
    print('  Pre-Ampere GPU: fp16 AMP + SDPA; Gemma will run in float32.')
 

  gpu               : Tesla T4
  cc                : (7, 5)
  n_gpu             : 2
  model_dtype       : torch.float32
  attn              : sdpa
  amp_dtype         : torch.float16
  use_amp           : True
  reference_compile : False
  gen_dtype         : torch.float32

transformers 4.57.6
  Pre-Ampere GPU: fp16 AMP + SDPA; Gemma will run in float32.


In [32]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
 
baseline_predictions = {}     # key -> (y_true_grades, y_pred_grades)
fold_level_results   = {}     # key -> [per-fold metric dicts]
baseline_configs     = {}     # key -> config dict, for the appendix table
 
def _metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return dict(
        acc = accuracy_score(y_true, y_pred),
        w1  = within_one(y_true, y_pred),
        qwk = cohen_kappa_score(y_true, y_pred, weights='quadratic'),
        f1w = f1_score(y_true, y_pred, average='weighted', zero_division=0),
        f1m = f1_score(y_true, y_pred, average='macro',    zero_division=0),
    )
 
def score_and_log(key, display, y_true, y_pred, folds, config=None):
    """y_true/y_pred MUST be in grade space (GRADE_MIN..GRADE_MAX)."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    assert y_true.min() >= GRADE_MIN, (
        f'{key}: labels look encoded (min={y_true.min()}). '
        f'Add + GRADE_MIN before logging.')
    m = _metrics(y_true, y_pred)
    baseline_predictions[key] = (y_true.tolist(), y_pred.tolist())
    fold_level_results[key]   = folds
    if config:
        baseline_configs[key] = config
    print(f'  {display:<44} | Acc={m["acc"]:.4f}  W1={m["w1"]:.4f}  '
          f'QWK={m["qwk"]:.4f}  F1w={m["f1w"]:.4f}')
    return m
 
print('Registry ready: baseline_predictions, fold_level_results, baseline_configs')

Registry ready: baseline_predictions, fold_level_results, baseline_configs


# S3 — Shared transformer machinery (used by B3, B5, B6, B7, B8)

In [33]:
import gc, math, random, time
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (AutoModel, AutoModelForSequenceClassification,
                          AutoTokenizer, get_linear_schedule_with_warmup)
 
N_GRADES = GRADE_MAX - GRADE_MIN + 1
 
def _seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
 
def free_gpu(*names):
    for n in names:
        if n in globals():
            del globals()[n]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
 
class GradeTextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts  = list(texts)
        self.labels = np.asarray(labels, dtype=np.int64)
    def __len__(self):  return len(self.texts)
    def __getitem__(self, i): return self.texts[i], int(self.labels[i])
 
class DynamicPadCollator:
    def __init__(self, tok, max_len): self.tok, self.max_len = tok, max_len
    def __call__(self, batch):
        texts, labels = zip(*batch)
        enc = self.tok(list(texts), truncation=True, max_length=self.max_len,
                       padding=True, return_tensors='pt')
        enc['labels'] = torch.tensor(labels, dtype=torch.long)
        return enc
 
class OrdinalLogLoss(nn.Module):
    """Castagnos, Lange & Vanel (2022):  -sum_k |k-y|^alpha * log(1-p_k).
    The k=y term vanishes since |y-y|^alpha = 0, so no masking and no separate
    cross-entropy term are needed."""
    def __init__(self, num_classes, alpha=1.5, include_ce=False):
        super().__init__()
        self.alpha, self.include_ce = float(alpha), bool(include_ce)
        self.register_buffer('class_idx', torch.arange(num_classes).float())
    def forward(self, logits, targets):
        probs = torch.softmax(logits.float(), dim=-1)
        w = (self.class_idx.unsqueeze(0) - targets.float().unsqueeze(1)).abs().pow(self.alpha)
        loss = -(w * torch.log(torch.clamp(1.0 - probs, min=1e-7))).sum(dim=1).mean()
        if self.include_ce:
            loss = loss + nn.functional.cross_entropy(logits, targets)
        return loss
 
def build_loss(spec, alpha=1.5):
    if spec == 'ce':  return nn.CrossEntropyLoss()
    if spec == 'oll': return OrdinalLogLoss(N_GRADES, alpha=alpha)
    raise ValueError(spec)
 
def load_encoder(model_name, gradient_checkpointing=False):
    kw = dict(num_labels=N_GRADES, torch_dtype=HW['model_dtype'],
              attn_implementation=HW['attn'])
    if 'modernbert' in model_name.lower():
        kw['reference_compile'] = HW['reference_compile'] and not gradient_checkpointing
    return AutoModelForSequenceClassification.from_pretrained(model_name, **kw)
 
def build_optimizer(model, lr, weight_decay):
    no_decay = ('bias', 'LayerNorm.weight', 'layer_norm', 'norm.weight')
    dec, nodec = [], []
    for n, p in model.named_parameters():
        if p.requires_grad:
            (nodec if any(nd in n for nd in no_decay) else dec).append(p)
    return AdamW([{'params': dec, 'weight_decay': weight_decay},
                  {'params': nodec, 'weight_decay': 0.0}], lr=lr)
 
@torch.no_grad()
def _predict_probs_sorted(model, texts, tok, max_len, batch_size, device):
    """Length-sorted batching, restored to input order."""
    texts = list(texts)
    order = np.argsort([len(t.split()) for t in texts])
    out = np.zeros((len(texts), N_GRADES), dtype=np.float32)
    model.eval()
    for i in range(0, len(order), batch_size):
        idx = order[i:i + batch_size]
        enc = tok([texts[j] for j in idx], truncation=True, max_length=max_len,
                  padding=True, return_tensors='pt')
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.autocast('cuda', dtype=HW['amp_dtype'],
                            enabled=HW['use_amp'] and device.type == 'cuda'):
            logits = model(**enc).logits
        out[idx] = torch.softmax(logits.float(), dim=-1).cpu().numpy()
        del enc, logits
    return out
 
def finetune_fold(model_name, texts_tr, y_tr, texts_predict,
                  loss_spec='ce', alpha=1.5, epochs=6, batch_size=8,
                  grad_accum_steps=1, eval_batch_size=None, lr=3e-5,
                  weight_decay=0.01, max_len=512, warmup_frac=0.10,
                  grad_clip=1.0, seed=42, freeze_embeddings=False,
                  gradient_checkpointing=False, verbose=False):
    """Fine-tune once; return probabilities for `texts_predict`.
    Effective batch = batch_size * grad_accum_steps."""
    _seed_everything(seed)
    device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    eval_bs = eval_batch_size or batch_size
    model = opt = sched = None
    try:
        tok   = AutoTokenizer.from_pretrained(model_name)
        model = load_encoder(model_name, gradient_checkpointing).to(device)
        if gradient_checkpointing:
            model.gradient_checkpointing_enable(
                gradient_checkpointing_kwargs={'use_reentrant': False})
            if hasattr(model.config, 'use_cache'):
                model.config.use_cache = False
        if freeze_embeddings:
            base = getattr(model, model.base_model_prefix, model)
            if hasattr(base, 'embeddings'):
                for p in base.embeddings.parameters():
                    p.requires_grad = False
        n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
        n_all = sum(p.numel() for p in model.parameters())
        if verbose:
            print(f'    trainable {n_tr:,}/{n_all:,} ({100*n_tr/n_all:.1f}%)')
 
        collate = DynamicPadCollator(tok, max_len)
        gen = torch.Generator(); gen.manual_seed(seed)
        dl = DataLoader(GradeTextDataset(texts_tr, y_tr), batch_size=batch_size,
                        shuffle=True, collate_fn=collate, generator=gen, num_workers=0)
 
        opt = build_optimizer(model, lr, weight_decay)
        spe = max(1, math.ceil(len(dl) / grad_accum_steps))
        n_steps = spe * epochs
        sched = get_linear_schedule_with_warmup(opt, int(warmup_frac * n_steps), n_steps)
        loss_fn = build_loss(loss_spec, alpha).to(device)
        use_amp = HW['use_amp'] and device.type == 'cuda'
        scaler = torch.amp.GradScaler('cuda',
                    enabled=use_amp and HW['amp_dtype'] == torch.float16)
        if device.type == 'cuda':
            torch.cuda.reset_peak_memory_stats()
 
        n_skip = 0
        for ep in range(1, epochs + 1):
            model.train(); run, seen = 0.0, 0
            opt.zero_grad(set_to_none=True)
            for step, batch in enumerate(dl):
                labels = batch.pop('labels').to(device)
                batch = {k: v.to(device) for k, v in batch.items()}
                with torch.autocast('cuda', dtype=HW['amp_dtype'], enabled=use_amp):
                    logits = model(**batch).logits
                    loss = loss_fn(logits, labels)
                if torch.isfinite(loss):
                    run += loss.item() * len(labels); seen += len(labels)
                    sc = loss / grad_accum_steps
                    (scaler.scale(sc) if scaler.is_enabled() else sc).backward()
                    del sc
                else:
                    n_skip += 1
                if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(dl):
                    if scaler.is_enabled():
                        scaler.unscale_(opt)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                        scaler.step(opt); scaler.update()
                    else:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                        opt.step()
                    sched.step(); opt.zero_grad(set_to_none=True)
                del batch, labels, logits, loss
            if verbose:
                print(f'      ep {ep}/{epochs}  loss={run/max(seen,1):.4f}')
 
        probs = _predict_probs_sorted(model, texts_predict, tok, max_len, eval_bs, device)
        peak = torch.cuda.max_memory_allocated()/2**30 if device.type=='cuda' else 0.0
        if n_skip:
            print(f'      [warn] {n_skip} non-finite loss step(s) skipped')
        return dict(probs=probs, peak_gib=peak, n_trainable=n_tr)
    finally:
        del model, opt, sched
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
 
def fit_with_backoff(cfg, texts_tr, y_tr, texts_predict, seed, verbose=False):
    """Halves physical batch and doubles accumulation on OOM. Effective batch
    is preserved at every retry, so a fold that needed a retry stays
    comparable to one that did not."""
    bs, ac, ebs = cfg['batch_size'], cfg['grad_accum_steps'], cfg['eval_batch_size']
    while True:
        try:
            return finetune_fold(
                cfg['model'], texts_tr, y_tr, texts_predict,
                loss_spec=cfg['loss'], alpha=cfg.get('alpha', 1.5),
                epochs=cfg['epochs'], batch_size=bs, grad_accum_steps=ac,
                eval_batch_size=ebs, lr=cfg['lr'],
                weight_decay=cfg['weight_decay'], max_len=cfg['max_len'],
                seed=seed, freeze_embeddings=cfg.get('freeze_embeddings', False),
                gradient_checkpointing=cfg.get('gradient_checkpointing', False),
                verbose=verbose)
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache()
            if bs > 1:
                bs, ac, ebs = bs//2, ac*2, max(1, ebs//2)
                print(f'      [OOM] retry bs={bs} accum={ac}')
            elif ebs > 1:
                ebs = 1; print('      [OOM] retry eval_bs=1')
            else:
                raise RuntimeError(f'OOM at bs=1, max_len={cfg["max_len"]}. '
                                   'Lower max_len and rerun.')
 
# ── Encoder registry. B3/B5 keep Lee et al.'s 3-epoch frozen-embedding
#    setting; B6/B7/B8 share a uniform 6-epoch protocol.
ENC = {
 'B3_lee':   dict(model='roberta-base', loss='ce', epochs=3, lr=3e-5,
                  weight_decay=0.01, max_len=512, batch_size=8,
                  grad_accum_steps=1, eval_batch_size=16,
                  freeze_embeddings=True, gradient_checkpointing=False,
                  display='RoBERTa-base (Lee et al. setting)'),
 'B6_oll':   dict(model='roberta-base', loss='oll', alpha=1.5, epochs=6, lr=3e-5,
                  weight_decay=0.01, max_len=512, batch_size=8,
                  grad_accum_steps=1, eval_batch_size=16,
                  freeze_embeddings=False, gradient_checkpointing=False,
                  display='RoBERTa-base + ordinal log-loss'),
 'B7_mbert': dict(model='answerdotai/ModernBERT-base', loss='ce', epochs=6,
                  lr=5e-5, weight_decay=1e-5, max_len=2048, batch_size=4,
                  grad_accum_steps=2, eval_batch_size=8,
                  freeze_embeddings=False, gradient_checkpointing=True,
                  display='ModernBERT-base'),
}
for _k, _c in ENC.items():
    assert _c['batch_size'] * _c['grad_accum_steps'] == 8, _k
print('Machinery ready. Effective batch 8 for all encoders.')
print('OLL alpha fixed at 1.5 (Castagnos et al. 2022 default; not tuned).')
 

Machinery ready. Effective batch 8 for all encoders.
OLL alpha fixed at 1.5 (Castagnos et al. 2022 default; not tuned).


# TRIVIAL BASELINE

In [34]:
from collections import Counter
yt, yp, fm = [], [], []
for k, (tr, te) in enumerate(outer_splits):
    maj = Counter(y_float[tr].astype(int)).most_common(1)[0][0]
    p = np.full(len(te), maj, dtype=int)
    t = y_float[te].astype(int)
    m = _metrics(t, p); m['fold'] = k; fm.append(m)
    yt.extend(t.tolist()); yp.extend(p.tolist())
score_and_log('B1', 'B1: ZeroR (majority class)', yt, yp, fm,
              config=dict(kind='trivial'))

  B1: ZeroR (majority class)                   | Acc=0.1453  W1=0.4477  QWK=0.0026  F1w=0.0689


{'acc': 0.14534883720930233,
 'w1': 0.4476744186046512,
 'qwk': np.float64(0.002608290638099642),
 'f1w': 0.06892889087656529,
 'f1m': 0.06272893772893773}

# B2

In [35]:
!pip install mord

  Preparing metadata (setup.py) ... done
  Created wheel for mord: filename=mord-0.7-py3-none-any.whl size=9953 sha256=5232943bb1166eb79b6a49fa55ef1df2b2af71005840e11fc418a4387d816f93
  Stored in directory: /root/.cache/pip/wheels/80/3e/3b/13f1adf346cad0fec675db328e4b0d814795c6c8e2fb659122
Successfully built mord


In [36]:
import textstat, mord
fkgl = np.array([textstat.flesch_kincaid_grade(t) for t in all_texts]).reshape(-1, 1)
yt, yp, fm = [], [], []
for k, (tr, te) in enumerate(outer_splits):
    mdl = mord.LogisticAT(alpha=1.0).fit(fkgl[tr], y_encoded[tr])
    p = mdl.predict(fkgl[te]) + GRADE_MIN
    t = y_encoded[te] + GRADE_MIN
    m = _metrics(t, p); m['fold'] = k; fm.append(m)
    yt.extend(t.tolist()); yp.extend(p.tolist())
score_and_log('B2', 'B2: FKGL-OrdFit', yt, yp, fm,
              config=dict(model='mord.LogisticAT', alpha=1.0, features='FKGL'))

  B2: FKGL-OrdFit                              | Acc=0.2384  W1=0.6453  QWK=0.6001  F1w=0.2314


{'acc': 0.23837209302325582,
 'w1': 0.6453488372093024,
 'qwk': np.float64(0.6001049648367798),
 'f1w': 0.23143111028779992,
 'f1m': 0.23323965879156056}

In [37]:
free_gpu()
roberta_cache = {}
yt, yp, fm = [], [], []
t0 = time.time()
for k, (tr, te) in enumerate(outer_splits):
    txt_tr = [all_texts[i] for i in tr]
    txt_te = [all_texts[i] for i in te]
    # predict test+train in one pass; _predict_probs_sorted preserves order
    out = fit_with_backoff(ENC['B3_lee'], txt_tr, y_encoded[tr],
                           txt_te + txt_tr, seed=RANDOM_STATE + k, verbose=(k == 0))
    P = out['probs']
    roberta_cache[k] = dict(probs_te=P[:len(te)], probs_tr=P[len(te):])
    p = P[:len(te)].argmax(1) + GRADE_MIN
    t = y_encoded[te] + GRADE_MIN
    m = _metrics(t, p); m['fold'] = k; fm.append(m)
    yt.extend(t.tolist()); yp.extend(p.tolist())
    print(f'  fold {k}: QWK={m["qwk"]:.4f}  Acc={m["acc"]:.4f}')
score_and_log('B3', 'B3: RoBERTa-base (text only)', yt, yp, fm,
              config=dict(ENC['B3_lee'], n_trainable=out['n_trainable']))
print(f'  {(time.time()-t0)/60:.1f} min')

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


... (7 outputs left out)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  fold 3: QWK=0.1941  Acc=0.2059


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  fold 4: QWK=0.0885  Acc=0.2059
  B3: RoBERTa-base (text only)                 | Acc=0.2500  W1=0.5930  QWK=0.5266  F1w=0.2126
  1.3 min


# B4: NEURAL ONLY (LINGFEAT)

In [51]:
# LingFeat T1 feature matrix (used by B4, B5, B8)
LINGFEAT_PATH = '/kaggle/input/datasets/maureenmiranda11/no-trunc-baseline/english_no_trunc_baseline_ling_feature.xlsx'

ling_raw = pd.read_excel(LINGFEAT_PATH)
ling_raw.columns = ling_raw.columns.str.strip()
ling_raw.drop(columns='index', inplace=True)
_grade_candidates = ['grade', 'Grade', 'GRADE', 'label', 'Label']
ling_grade_col = next((c for c in _grade_candidates if c in ling_raw.columns), None)
assert ling_grade_col is not None, f'No grade column: {list(ling_raw.columns)[:15]}'

ling_feat_cols = [c for c in ling_raw.columns if c != ling_grade_col]
assert len(ling_raw) == len(y_float), f'Row mismatch: {len(ling_raw)} vs {len(y_float)}'
assert np.allclose(ling_raw[ling_grade_col].astype(float).values, y_float), \
    'Grade mismatch — LingFeat rows are not aligned with the main corpus'

X_ling = ling_raw[ling_feat_cols].astype(np.float32).values
print(f'X_ling: {X_ling.shape}  (grade alignment verified)')

X_ling: (172, 255)  (grade alignment verified)


In [52]:
from xgboost import XGBClassifier
XGB_LEE = dict(n_estimators=200, learning_rate=0.05, max_depth=9, gamma=1,
               random_state=RANDOM_STATE, n_jobs=-1,
               objective='multi:softmax', num_class=N_GRADES)
print(f'  LingFeat matrix: {X_ling.shape}')
yt, yp, fm = [], [], []
for k, (tr, te) in enumerate(outer_splits):
    mdl = XGBClassifier(**XGB_LEE).fit(X_ling[tr], y_encoded[tr])
    p = mdl.predict(X_ling[te]) + GRADE_MIN
    t = y_encoded[te] + GRADE_MIN
    m = _metrics(t, p); m['fold'] = k; fm.append(m)
    yt.extend(t.tolist()); yp.extend(p.tolist())
    print(f'  fold {k}: QWK={m["qwk"]:.4f}  Acc={m["acc"]:.4f}')
score_and_log('B4', 'B4: XGB + LingFeat T1', yt, yp, fm,
              config=dict(XGB_LEE, n_features=int(X_ling.shape[1])))

  LingFeat matrix: (172, 255)
  fold 0: QWK=0.8012  Acc=0.3143
  fold 1: QWK=0.8901  Acc=0.4286
  fold 2: QWK=0.8766  Acc=0.2941
  fold 3: QWK=0.8678  Acc=0.3235
  fold 4: QWK=0.8774  Acc=0.4412
  B4: XGB + LingFeat T1                        | Acc=0.3605  W1=0.8663  QWK=0.8624  F1w=0.3603


{'acc': 0.36046511627906974,
 'w1': 0.8662790697674418,
 'qwk': np.float64(0.8623965053080713),
 'f1w': 0.36030454758440933,
 'f1m': 0.3496067857058569}

In [40]:
from sklearn.ensemble import RandomForestClassifier
RF_LEE = dict(n_estimators=800, max_depth=None, max_features='sqrt',
              random_state=RANDOM_STATE, n_jobs=-1)
yt, yp, fm = [], [], []
for k, (tr, te) in enumerate(outer_splits):
    c = roberta_cache[k]
    Xtr = np.hstack([c['probs_tr'], X_ling[tr]]).astype(np.float32)
    Xte = np.hstack([c['probs_te'], X_ling[te]]).astype(np.float32)
    p = RandomForestClassifier(**RF_LEE).fit(Xtr, y_encoded[tr]).predict(Xte) + GRADE_MIN
    t = y_encoded[te] + GRADE_MIN
    m = _metrics(t, p); m['fold'] = k; fm.append(m)
    yt.extend(t.tolist()); yp.extend(p.tolist())
    print(f'  fold {k}: QWK={m["qwk"]:.4f}  Acc={m["acc"]:.4f}')
score_and_log('B5', 'B5: RoBERTa-RF-T1 hybrid', yt, yp, fm,
              config=dict(RF_LEE, encoder='B3', n_features=int(X_ling.shape[1]) + N_GRADES))

  fold 0: QWK=0.8981  Acc=0.3714
  fold 1: QWK=0.8772  Acc=0.3429
  fold 2: QWK=0.9219  Acc=0.4706
  fold 3: QWK=0.8863  Acc=0.4118
  fold 4: QWK=0.8943  Acc=0.4118
  B5: RoBERTa-RF-T1 hybrid                     | Acc=0.4012  W1=0.9186  QWK=0.8963  F1w=0.3892


{'acc': 0.4011627906976744,
 'w1': 0.9186046511627907,
 'qwk': np.float64(0.8963235171851877),
 'f1w': 0.3892284270000626,
 'f1m': 0.3787340900924039}

In [44]:
free_gpu()
yt, yp, fm = [], [], []
t0 = time.time()
for k, (tr, te) in enumerate(outer_splits):
    out = fit_with_backoff(ENC['B6_oll'], [all_texts[i] for i in tr], y_encoded[tr],
                           [all_texts[i] for i in te], seed=RANDOM_STATE + k,
                           verbose=(k == 0))
    p = out['probs'].argmax(1) + GRADE_MIN
    t = y_encoded[te] + GRADE_MIN
    m = _metrics(t, p); m['fold'] = k; fm.append(m)
    yt.extend(t.tolist()); yp.extend(p.tolist())
    print(f'  fold {k}: QWK={m["qwk"]:.4f}  Acc={m["acc"]:.4f}')
score_and_log('B6', 'B6: RoBERTa-base + ordinal log-loss', yt, yp, fm,
              config=dict(ENC['B6_oll'], n_trainable=out['n_trainable']))
print(f'  {(time.time()-t0)/60:.1f} min')

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    trainable 124,651,015/124,651,015 (100.0%)
      ep 1/6  loss=4.2429
      ep 2/6  loss=3.4460
      ep 3/6  loss=2.7449
      ep 4/6  loss=2.0726
      ep 5/6  loss=1.5966
      ep 6/6  loss=1.3293
  fold 0: QWK=0.8705  Acc=0.2571


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  fold 1: QWK=0.8512  Acc=0.4000


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  fold 2: QWK=0.8470  Acc=0.3824


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  fold 3: QWK=0.8354  Acc=0.3529


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  fold 4: QWK=0.8472  Acc=0.3529
  B6: RoBERTa-base + ordinal log-loss          | Acc=0.3488  W1=0.8953  QWK=0.8501  F1w=0.3032
  2.2 min


In [45]:
free_gpu()
modernbert_cache = {}
yt, yp, fm = [], [], []
t0 = time.time()
for k, (tr, te) in enumerate(outer_splits):
    txt_tr = [all_texts[i] for i in tr]
    txt_te = [all_texts[i] for i in te]
    out = fit_with_backoff(ENC['B7_mbert'], txt_tr, y_encoded[tr],
                           txt_te + txt_tr, seed=RANDOM_STATE + k, verbose=(k == 0))
    P = out['probs']
    modernbert_cache[k] = dict(probs_te=P[:len(te)], probs_tr=P[len(te):])
    p = P[:len(te)].argmax(1) + GRADE_MIN
    t = y_encoded[te] + GRADE_MIN
    m = _metrics(t, p); m['fold'] = k; fm.append(m)
    yt.extend(t.tolist()); yp.extend(p.tolist())
    print(f'  fold {k}: QWK={m["qwk"]:.4f}  Acc={m["acc"]:.4f}  peak {out["peak_gib"]:.2f} GiB')
score_and_log('B7', 'B7: ModernBERT-base', yt, yp, fm,
              config=dict(ENC['B7_mbert'], n_trainable=out['n_trainable']))
print(f'  {(time.time()-t0)/60:.1f} min')

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    trainable 149,610,247/149,610,247 (100.0%)
      ep 1/6  loss=1.9046
      ep 2/6  loss=1.2556
      ep 3/6  loss=1.0186
      ep 4/6  loss=0.5881
      ep 5/6  loss=0.4577
      ep 6/6  loss=0.3005
  fold 0: QWK=0.8474  Acc=0.3143  peak 4.37 GiB


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  fold 1: QWK=0.9088  Acc=0.4000  peak 4.37 GiB


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  fold 2: QWK=0.8798  Acc=0.5000  peak 4.37 GiB


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  fold 3: QWK=0.9115  Acc=0.5000  peak 4.36 GiB


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  fold 4: QWK=0.8820  Acc=0.4118  peak 4.37 GiB
  B7: ModernBERT-base                          | Acc=0.4244  W1=0.9012  QWK=0.8866  F1w=0.4201
  16.7 min


In [46]:
yt, yp, fm = [], [], []
for k, (tr, te) in enumerate(outer_splits):
    c = modernbert_cache[k]
    Xtr = np.hstack([c['probs_tr'], X_ling[tr]]).astype(np.float32)
    Xte = np.hstack([c['probs_te'], X_ling[te]]).astype(np.float32)
    p = RandomForestClassifier(**RF_LEE).fit(Xtr, y_encoded[tr]).predict(Xte) + GRADE_MIN
    t = y_encoded[te] + GRADE_MIN
    m = _metrics(t, p); m['fold'] = k; fm.append(m)
    yt.extend(t.tolist()); yp.extend(p.tolist())
    print(f'  fold {k}: QWK={m["qwk"]:.4f}  Acc={m["acc"]:.4f}')
score_and_log('B8', 'B8: ModernBERT-RF-T1 hybrid', yt, yp, fm,
              config=dict(RF_LEE, encoder='B7', n_features=int(X_ling.shape[1]) + N_GRADES))

  fold 0: QWK=0.8445  Acc=0.2857
  fold 1: QWK=0.8903  Acc=0.3714
  fold 2: QWK=0.9277  Acc=0.5588
  fold 3: QWK=0.9134  Acc=0.5588
  fold 4: QWK=0.8850  Acc=0.3529
  B8: ModernBERT-RF-T1 hybrid                  | Acc=0.4244  W1=0.9244  QWK=0.8942  F1w=0.4281


{'acc': 0.42441860465116277,
 'w1': 0.9244186046511628,
 'qwk': np.float64(0.8942434503653538),
 'f1w': 0.42812015610423027,
 'f1m': 0.41861302880600254}

In [47]:
free_gpu('gemma_model', 'gemma_proc', 'gemma_tok')
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)
 
GEMMA_MODEL = 'google/gemma-3-4b-it'
# Gemma was trained in bf16. On pre-Ampere GPUs fp16 overflows and produces
# non-finite logits, so float32 across both cards is required.
GEMMA_DTYPE = HW['gen_dtype']
 
from transformers import AutoProcessor
_kw = dict(device_map='auto', attn_implementation=HW['attn'],
           torch_dtype=GEMMA_DTYPE, token=HF_TOKEN)
if GEMMA_DTYPE == torch.float32:
    _kw['max_memory'] = {i: '13GiB' for i in range(HW['n_gpu'])}
gemma_proc = AutoProcessor.from_pretrained(GEMMA_MODEL, token=HF_TOKEN)
try:
    from transformers import Gemma3ForConditionalGeneration
    gemma_model = Gemma3ForConditionalGeneration.from_pretrained(GEMMA_MODEL, **_kw)
except Exception:
    from transformers import AutoModelForCausalLM
    gemma_model = AutoModelForCausalLM.from_pretrained(GEMMA_MODEL, **_kw)
gemma_model.eval()
gemma_tok = getattr(gemma_proc, 'tokenizer', gemma_proc)
 
GRADE_TOKEN_IDS = {}
for g in range(GRADE_MIN, GRADE_MAX + 1):
    ids = {gemma_tok.encode(v, add_special_tokens=False)[0]
           for v in (f'{g}', f' {g}') if gemma_tok.encode(v, add_special_tokens=False)}
    GRADE_TOKEN_IDS[g] = sorted(ids)
print(f'Loaded {GEMMA_MODEL} @ {GEMMA_DTYPE}')
print('Grade token ids:', GRADE_TOKEN_IDS)

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

... (8 outputs left out)

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Loaded google/gemma-3-4b-it @ torch.float32
Grade token ids: {2: [236743, 236778], 3: [236743, 236800], 4: [236743, 236812], 5: [236743, 236810], 6: [236743, 236825], 7: [236743, 236832], 8: [236743, 236828]}


In [48]:
GEMMA_EXAMPLE_WORDS = 350     # per demonstration; the query is passed in full
SYSTEM_PROMPT = (
    'You are an expert on the NCERT English curriculum used in Indian '
    'primary and middle schools. You assign each story the school grade at '
    'which it appears in the NCERT English readers. Answer with a single '
    f'integer between {GRADE_MIN} and {GRADE_MAX} and nothing else.')
 
def _clip(t, n):
    w = t.split()
    return t if (n is None or len(w) <= n) else ' '.join(w[:n])
 
def build_messages(query, examples):
    msgs = [{'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]}]
    for ex_t, ex_g in examples:
        msgs.append({'role': 'user', 'content': [{'type': 'text', 'text':
            'Story:\n' + _clip(ex_t, GEMMA_EXAMPLE_WORDS) +
            '\n\nWhich NCERT grade is this story from?'}]})
        msgs.append({'role': 'assistant',
                     'content': [{'type': 'text', 'text': str(int(ex_g))}]})
    msgs.append({'role': 'user', 'content': [{'type': 'text', 'text':
        'Story:\n' + query + '\n\nWhich NCERT grade is this story from?'}]})
    return msgs
 
def build_examples(texts, grades, k, rng):
    """Band-spanning selection. k=1 draws one Medium anchor; k=3 draws one per
    band (Low G2-4 / Medium G5-6 / High G7-8), so three of seven grades are
    covered; k=7 draws one per grade."""
    if k == 0: return []
    g = np.asarray(grades).astype(int)
    bands = {'low': np.where(g <= 4)[0], 'mid': np.where((g >= 5) & (g <= 6))[0],
             'high': np.where(g >= 7)[0]}
    picks = []
    if k == 1:
        pool = bands['mid'] if len(bands['mid']) else np.arange(len(g))
        picks = [rng.choice(pool)]
    elif k == 3:
        picks = [rng.choice(bands[b]) for b in ('low', 'mid', 'high') if len(bands[b])]
    else:
        for gr in range(GRADE_MIN, GRADE_MAX + 1):
            pool = np.where(g == gr)[0]
            if len(pool): picks.append(rng.choice(pool))
        if len(picks) > k: picks = list(rng.choice(picks, k, replace=False))
    rng.shuffle(picks)
    return [(texts[i], g[i]) for i in picks]
 
_LOGITS_KW = None
@torch.no_grad()
def gemma_grade(query, examples):
    """Read the probability of each grade token at the first generated
    position; no generation, no parsing, so every story gets a valid grade."""
    global _LOGITS_KW
    inp = gemma_proc.apply_chat_template(build_messages(query, examples),
            add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors='pt')
    dev = next(gemma_model.parameters()).device
    inp = {k: (v.to(dev) if hasattr(v, 'to') else v) for k, v in inp.items()}
    if _LOGITS_KW is None:
        for kw in ('logits_to_keep', 'num_logits_to_keep'):
            try:
                out = gemma_model(**inp, **{kw: 1}); _LOGITS_KW = kw; break
            except TypeError: continue
        else:
            _LOGITS_KW = 'none'; out = gemma_model(**inp)
    else:
        out = gemma_model(**inp) if _LOGITS_KW == 'none' else gemma_model(**inp, **{_LOGITS_KW: 1})
    logits = out.logits[0, -1, :].float()
    if not torch.isfinite(logits).all():
        raise FloatingPointError('Non-finite logits — set GEMMA_DTYPE=torch.float32.')
    lp = torch.log_softmax(logits, dim=-1)
    sc = np.array([float(torch.logsumexp(lp[GRADE_TOKEN_IDS[g]], 0))
                   for g in range(GRADE_MIN, GRADE_MAX + 1)])
    del inp, out, logits, lp
    return int(GRADE_MIN + int(np.argmax(sc)))
 
for k_shot in [0, 1, 3, 7]:
    yt, yp, fm = [], [], []
    t0 = time.time()
    for k, (tr, te) in enumerate(outer_splits):
        rng = np.random.default_rng(RANDOM_STATE + 1000 * k)
        ex = build_examples([all_texts[i] for i in tr], y_float[tr], k_shot, rng)
        p = np.array([gemma_grade(all_texts[i], ex) for i in te])
        t = y_float[te].astype(int)
        m = _metrics(t, p); m['fold'] = k; fm.append(m)
        yt.extend(t.tolist()); yp.extend(p.tolist())
        print(f'  k={k_shot} fold {k}: QWK={m["qwk"]:.4f}  Acc={m["acc"]:.4f}')
        gc.collect(); torch.cuda.empty_cache()
    score_and_log(f'B9_{k_shot}shot', f'B9: Gemma-3-4B-it {k_shot}-shot', yt, yp, fm,
                  config=dict(model=GEMMA_MODEL, k_shot=k_shot,
                              dtype=str(GEMMA_DTYPE), example_words=GEMMA_EXAMPLE_WORDS,
                              scoring='constrained'))
    print(f'  {(time.time()-t0)/60:.1f} min\n')
free_gpu('gemma_model', 'gemma_proc', 'gemma_tok')

  k=0 fold 0: QWK=0.4212  Acc=0.2000
  k=0 fold 1: QWK=0.4750  Acc=0.2000
  k=0 fold 2: QWK=0.5115  Acc=0.2353
  k=0 fold 3: QWK=0.4830  Acc=0.1176
  k=0 fold 4: QWK=0.3053  Acc=0.2059
  B9: Gemma-3-4B-it 0-shot                     | Acc=0.1919  W1=0.5988  QWK=0.4409  F1w=0.1041
  5.8 min

  k=1 fold 0: QWK=0.7862  Acc=0.2857
  k=1 fold 1: QWK=0.7586  Acc=0.2857
  k=1 fold 2: QWK=0.8242  Acc=0.3529
  k=1 fold 3: QWK=0.7825  Acc=0.2941
  k=1 fold 4: QWK=0.7678  Acc=0.3235
  B9: Gemma-3-4B-it 1-shot                     | Acc=0.3081  W1=0.8256  QWK=0.7836  F1w=0.2536
  8.8 min

  k=3 fold 0: QWK=0.7139  Acc=0.2857
  k=3 fold 1: QWK=0.7880  Acc=0.3714
  k=3 fold 2: QWK=0.9109  Acc=0.4118
  k=3 fold 3: QWK=0.7884  Acc=0.2941
  k=3 fold 4: QWK=0.7598  Acc=0.2353
  B9: Gemma-3-4B-it 3-shot                     | Acc=0.3198  W1=0.7965  QWK=0.7957  F1w=0.3138
  13.2 min

  k=7 fold 0: QWK=0.7203  Acc=0.2571
  k=7 fold 1: QWK=0.7610  Acc=0.2857
  k=7 fold 2: QWK=0.8258  Acc=0.2647
  k=7 fold 3: Q

In [53]:
import pandas as pd, json
ORDER = [('B1','B1: ZeroR (majority class)'),
         ('B2','B2: FKGL-OrdFit'),
         ('B3','B3: RoBERTa-base (text only)$^{\\dagger}$'),
         ('B4','B4: XGB + LingFeat T1'),
         ('B5','B5: RoBERTa-RF-T1 hybrid$^{\\dagger}$'),
         ('B6','B6: RoBERTa-base + ordinal log-loss'),
         ('B7','B7: ModernBERT-base'),
         ('B8','B8: ModernBERT-RF-T1 hybrid'),
         ('B9_0shot','B9: Gemma-3-4B-it, 0-shot'),
         ('B9_1shot','B9: Gemma-3-4B-it, 1-shot'),
         ('B9_3shot','B9: Gemma-3-4B-it, 3-shot'),
         ('B9_7shot','B9: Gemma-3-4B-it, 7-shot')]
 
rows, missing = [], []
for key, disp in ORDER:
    if key not in baseline_predictions:
        missing.append(key); continue
    t, p = baseline_predictions[key]
    agg = _metrics(t, p)
    f = fold_level_results[key]
    r = {'System': disp}
    for met, lab in [('acc','Acc'), ('w1','Acc±1'), ('qwk','QWK'),
                     ('f1w','F1-w'), ('f1m','F1-m')]:
        v = np.array([x[met] for x in f])
        r[lab] = f'{agg[met]:.3f} ±{v.std(ddof=1):.3f}'
        r[lab + '_tex'] = f'${agg[met]:.2f}_{{\\pm{v.std(ddof=1):.2f}}}$'
    r['n'] = len(t)
    rows.append(r)
 
df = pd.DataFrame(rows)
print('═' * 100)
print(df[['System','Acc','Acc±1','QWK','F1-w','F1-m','n']].to_string(index=False))
print('═' * 100)
if missing: print('NOT RUN:', missing)
bad = df[df['n'] != len(y_float)]
if len(bad): print('WARNING — wrong prediction count:\n', bad[['System','n']])
 
print('\n% ---- LaTeX rows ----')
for _, r in df.iterrows():
    print(f"{r['System']}\n  & {r['Acc_tex']} & {r['Acc±1_tex']} & {r['QWK_tex']} \\\\")
 
df.drop(columns=[c for c in df.columns if c.endswith('_tex')]).to_csv(
    os.path.join(CKPT_DIR, 'baselines_final.csv'), index=False)
with open(os.path.join(CKPT_DIR, 'baseline_configs.json'), 'w') as fh:
    json.dump(baseline_configs, fh, indent=2, default=str)
 
print('\n% ---- Config values for Appendix H ----')
for k, c in baseline_configs.items():
    print(f'  {k}: ' + ', '.join(f'{a}={b}' for a, b in c.items()
                                 if a not in ('display',)))

════════════════════════════════════════════════════════════════════════════════════════════════════
                                  System          Acc        Acc±1          QWK         F1-w         F1-m   n
              B1: ZeroR (majority class) 0.145 ±0.002 0.448 ±0.016 0.003 ±0.000 0.069 ±0.001 0.063 ±0.001 172
                         B2: FKGL-OrdFit 0.238 ±0.087 0.645 ±0.077 0.600 ±0.110 0.231 ±0.082 0.233 ±0.081 172
B3: RoBERTa-base (text only)$^{\dagger}$ 0.250 ±0.078 0.593 ±0.084 0.527 ±0.314 0.213 ±0.060 0.198 ±0.054 172
                   B4: XGB + LingFeat T1 0.360 ±0.069 0.866 ±0.043 0.862 ±0.035 0.360 ±0.080 0.350 ±0.078 172
    B5: RoBERTa-RF-T1 hybrid$^{\dagger}$ 0.401 ±0.048 0.919 ±0.025 0.896 ±0.017 0.389 ±0.039 0.379 ±0.035 172
     B6: RoBERTa-base + ordinal log-loss 0.349 ±0.055 0.895 ±0.049 0.850 ±0.013 0.303 ±0.040 0.307 ±0.042 172
                     B7: ModernBERT-base 0.424 ±0.078 0.901 ±0.049 0.887 ±0.026 0.420 ±0.078 0.409 ±0.077 172
             B8: Mo